<a href="https://colab.research.google.com/github/khaling/phase-coherence-based-feature-selection-for-asd-td-classification/blob/main/phase_coherence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Embedding
from sklearn.model_selection import KFold, StratifiedKFold


In [ ]:
import nibabel as nib
import pandas as pd

In [ ]:
pip install nilearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
da

In [ ]:
import nilearn
from nilearn import image
from nilearn import datasets

In [ ]:
from keras.layers import Input, Add, Dense, Activation, ZeroPadding2D,ZeroPadding3D, BatchNormalization, MaxPooling1D, Flatten,Conv1D, Conv2D,Conv3D,AveragePooling1D, AveragePooling2D,AveragePooling3D, MaxPooling2D, MaxPooling3D, GlobalMaxPooling2D
from keras.models import Model, load_model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q nilearn

import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    Embedding,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [ ]:
!pip install -q hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 7.5 MB/s eta 0:00:00


In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# All-Sites Comparison: Plain Static FC vs. Percentile-Masked FC
# ACROSS MULTIPLE FREQUENCY BANDS + PER-SITE TIME-SERIES LENGTH DIAGNOSTICS
#
# Loads every available ABIDE site, then runs a WITHIN-SITE 5-fold
# stratified CV comparison between Full Static FC and Phase-Coherence
# Masked FC, separately for each site AND separately for each of
# several candidate frequency bands (both feature types/bands share
# the exact same folds within a site, so any difference is
# attributable to the feature/band, not to fold composition).
#
# This answers: "how does percentile-masked FC performance vary by
# site AND by the frequency band used for the phase-coherence step?"
# It also reports each site's native (pre-resampling) scan length, in
# both timepoints and seconds, since sites with short scans get
# zero-padded rather than subsampled -- which can degrade both the FC
# estimate and (especially) the phase-coherence estimate for the
# lower frequency bands, which need more cycles to stabilize.
#
# BANDS TESTED (Hz):
#   0.01-0.03  slow-4 / very-low-frequency range
#   0.03-0.05  slow-3 range
#   0.04-0.07  the paper's default band (Glerean et al., 2012)
#   0.07-0.10  slow-2 / upper conventional resting-state range
#
# CRITICAL METHODOLOGICAL FIX FOR MULTI-SITE PHASE-COHERENCE:
# The main pipeline assumed a single fixed TR (2.0s) for the phase-
# coherence band-pass filter. Different ABIDE sites use different TRs
# (documented below), so naively reusing one fixed TR across sites
# would silently make each nominal band correspond to a DIFFERENT
# physical frequency range at each site -- invalidating the
# comparison. This script fixes that by computing phase-coherence with
# each subject's OWN site-specific TR (SITE_TR_MAP below), so each
# frequency band means the same physical thing for every subject
# regardless of which site they came from.
#
# It ALSO handles the more subtle version of this problem: subjects
# whose native scan length exceeds TARGET_TIME are subsampled (not
# just padded), which stretches/compresses the EFFECTIVE sampling
# interval away from the nominal TR. This script tracks each subject's
# native (pre-resampling) length and computes an EFFECTIVE per-subject
# TR that accounts for this, rather than only correcting for
# site-level TR differences.
#
# SITE TR REFERENCE TABLE (all 15 ABIDE-I sites; SITE_TR_MAP below
# covers every one of them). Verify every value against your own ABIDE
# phenotypic file before trusting results -- sourced from a published
# ABIDE-I acquisition-parameters table (TR converted from ms to
# seconds):
#   CALTECH : TR = 2.000s      OLIN    : TR = 1.500s
#   CMU     : TR = 2.000s      SBL     : TR = 2.200s
#   KKI     : TR = 2.500s      SDSU    : TR = 2.000s
#   LEUVEN  : TR = 1.667s      TRINITY : TR = 2.000s
#   MAXMUN  : TR = 3.000s      UCLA    : TR = 3.000s
#   NYU     : TR = 2.000s      UM      : TR = 2.000s
#   PITT    : TR = 1.500s      USM     : TR = 2.000s
#   YALE    : TR = 2.000s
# SITES_TO_LOAD = None loads every site found; sites with too few
# subjects per class for meaningful stratified CV are skipped
# automatically (see MIN_PER_CLASS below), with a printed warning.
# =====================================================
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "font.size": 11, "font.family": "serif",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER -- multi-site, tracks native length AND post-resample
#    real (non-padded) length, since both are needed for the effective-
#    TR correction below.
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"
N_ROIS = 200
TARGET_TIME = 200
SITES_TO_LOAD = None   # None loads every site found; set to a list (e.g. ["NYU","UM","PITT"]) to restrict

SITE_TR_MAP = {
    "CALTECH": 2.0,
    "CMU": 2.0,
    "KKI": 2.5,
    "LEUVEN": 1.667,
    "MAXMUN": 3.0,     # "Ludwig" in the source table
    "NYU": 2.0,
    "OLIN": 1.5,
    "SBL": 2.2,
    "SDSU": 2.0,
    "TRINITY": 2.0,
    "UCLA": 3.0,
    "UM": 2.0,         # "UMich" in the source table
    "PITT": 1.5,       # "UPitt" in the source table
    "USM": 2.0,
    "YALE": 2.0,
    "STANFORD": 2.0,
    "OHSU": 2.0,
}
NETWORKS = {"WHOLE_BRAIN": list(range(200))}
SELECTED_ROIS = NETWORKS["WHOLE_BRAIN"]

def parse_site_from_filename(name):
    return name.split("_")[0]

def load_subjects(folder, label, sites_to_load=None):
    """
    Same as the main pipeline's loader, but additionally records:
      - site           : parsed site code
      - native_length  : timepoints BEFORE any subsample/pad (needed to
                          detect and correct for subsampling-induced
                          effective-TR distortion, and for the length
                          diagnostics in section 2b)
      - real_length    : timepoints actually carrying real (non-padded)
                          signal after resampling, as before
    """
    results = {"X": [], "y": [], "subject_ids": [], "sites": [],
               "real_lengths": [], "native_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results
    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if sites_to_load is not None and site.upper() not in [s.upper() for s in sites_to_load]:
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape); continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape); continue
            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            native_length = ts.shape[0]   # BEFORE any resampling
            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]
            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site.upper())
            results["real_lengths"].append(real_length)
            results["native_lengths"].append(native_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")
    return results

asd_data = load_subjects(ASD_DATA_PATH, 1, sites_to_load=SITES_TO_LOAD)
td_data  = load_subjects(TD_DATA_PATH, 0, sites_to_load=SITES_TO_LOAD)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
sites = np.array(asd_data["sites"] + td_data["sites"])
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)
native_lengths = np.asarray(asd_data["native_lengths"] + td_data["native_lengths"], dtype=np.int32)

print("\n================ MULTI-SITE DATASET SUMMARY ================")
for site_name in sorted(set(sites)):
    site_mask = sites == site_name
    print(f"  {site_name}: {site_mask.sum()} subjects "
          f"({(y[site_mask] == 1).sum()} ASD, {(y[site_mask] == 0).sum()} TD)")
print("Total subjects:", len(X))

unmapped = set(sites) - set(SITE_TR_MAP.keys())
assert not unmapped, f"No TR mapping for site(s) {unmapped} -- add them to SITE_TR_MAP before proceeding."

# =====================================================
# 2b. PER-SITE TIME-SERIES LENGTH DIAGNOSTICS
#     Some sites may have shorter native scans than TARGET_TIME (178),
#     meaning many/most of their subjects get zero-padded rather than
#     subsampled. Padding can hurt phase-coherence estimation (edge
#     artifacts in filtfilt/hilbert operate on padded zeros) and means
#     less real signal is driving the FC estimate for that subject.
#     This block reports native scan length (in timepoints, and in
#     seconds using each site's nominal TR) so you can see which sites
#     are short before trusting their classification numbers.
# =====================================================
print("\n================ PER-SITE TIME-SERIES LENGTH (native, pre-resampling) ================")
length_header = f"{'Site':<10}{'N':<6}{'Min':<8}{'Median':<8}{'Mean':<8}{'Max':<8}{'%Padded':<10}{'%Subsampled':<12}"
print(length_header)
print("-" * len(length_header))

site_length_summary = {}
for site_name in sorted(set(sites)):
    site_mask = sites == site_name
    lens = native_lengths[site_mask]
    n = len(lens)
    pct_padded = 100.0 * np.mean(lens < TARGET_TIME)
    pct_subsampled = 100.0 * np.mean(lens > TARGET_TIME)
    nominal_tr = SITE_TR_MAP.get(site_name, np.nan)
    site_length_summary[site_name] = {
        "n": n, "min": int(lens.min()), "median": float(np.median(lens)),
        "mean": float(lens.mean()), "max": int(lens.max()),
        "pct_padded": pct_padded, "pct_subsampled": pct_subsampled,
        "nominal_tr": nominal_tr,
    }
    row = (f"{site_name:<10}{n:<6}{lens.min():<8}{np.median(lens):<8.1f}"
           f"{lens.mean():<8.1f}{lens.max():<8}{pct_padded:<10.1f}{pct_subsampled:<12.1f}")
    print(row)

print(f"\nSites where >50% of subjects are shorter than TARGET_TIME "
      f"({TARGET_TIME} timepoints) and therefore zero-padded:")
flagged_any = False
for site_name, s in site_length_summary.items():
    if s["pct_padded"] > 50:
        flagged_any = True
        real_secs = s["median"] * s["nominal_tr"] if not np.isnan(s["nominal_tr"]) else float("nan")
        print(f"  {site_name}: median native length = {s['median']:.0f} tp "
              f"({s['pct_padded']:.0f}% of subjects padded), nominal TR = {s['nominal_tr']}s "
              f"-> ~{real_secs:.0f}s of real scan time")
if not flagged_any:
    print("  (none)")

print("\n================ PER-SITE SCAN DURATION (native_length x nominal TR, seconds) ================")
dur_header = f"{'Site':<10}{'TR(s)':<8}{'MedianTP':<10}{'MedianDur(s)':<14}{'MinDur(s)':<12}{'MaxDur(s)':<12}"
print(dur_header)
print("-" * len(dur_header))
for site_name, s in site_length_summary.items():
    tr = s["nominal_tr"]
    med_dur = s["median"] * tr
    min_dur = s["min"] * tr
    max_dur = s["max"] * tr
    print(f"{site_name:<10}{tr:<8}{s['median']:<10.1f}{med_dur:<14.1f}{min_dur:<12.1f}{max_dur:<12.1f}")

def plot_native_length_by_site(native_lengths, sites, target_time, out_name="native_length_by_site"):
    included = sorted(set(sites))
    data = [native_lengths[sites == s] for s in included]
    fig, ax = plt.subplots(figsize=(max(10, 0.6 * len(included)), 5.5))
    ax.boxplot(data, labels=included, showfliers=True)
    ax.axhline(target_time, color="red", linestyle="--", linewidth=1,
               label=f"TARGET_TIME={target_time}")
    ax.set_xticklabels(included, rotation=45, ha="right")
    ax.set_ylabel("Native timepoints (pre-resampling)")
    ax.set_title("Distribution of Native Scan Length by Site")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{out_name}.png and .pdf")
    plt.close(fig)

plot_native_length_by_site(native_lengths, sites, TARGET_TIME)

# =====================================================
# 3. EFFECTIVE PER-SUBJECT TR (site TR + subsampling correction)
# =====================================================
def compute_effective_tr(site_name, native_length, target_time):
    nominal_tr = SITE_TR_MAP[site_name]
    if native_length <= target_time:
        return nominal_tr
    return nominal_tr * (native_length - 1) / (target_time - 1)

effective_trs = np.array([
    compute_effective_tr(sites[i], native_lengths[i], TARGET_TIME) for i in range(len(X))
])
print("\nEffective TR range across loaded subjects:", effective_trs.min(), "-", effective_trs.max())
n_subsampled = (native_lengths > TARGET_TIME).sum()
print(f"Subjects requiring subsampling (native_length > TARGET_TIME={TARGET_TIME}): "
      f"{n_subsampled} / {len(X)}  -- these get an effective-TR correction; others use nominal site TR.")

# =====================================================
# 4. FEATURE CONSTRUCTION
#    Static FC: site/TR/band-independent, computed once.
#    Phase-coherence masked FC: uses PER-SUBJECT effective TR, computed
#    SEPARATELY for each candidate frequency band below.
# =====================================================
PERCENTILE = 25.0   # held fixed; this experiment varies the band, not the threshold

# The four candidate bands (Hz) -- see header comment for rationale.
BANDS = {
    "0.01-0.03 (slow-4/VLF)":            (0.01, 0.03),
    "0.03-0.05 (slow-3)":                (0.03, 0.05),
    "0.04-0.07 (default, Glerean 2012)": (0.04, 0.07),
    "0.07-0.10 (slow-2)":                (0.07, 0.10),
}

def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    padlen = min(3 * (2 * order + 1), ts.shape[0] - 1)
    return filtfilt(b, a, ts, axis=0, padlen=padlen)

def compute_static_fc_matrix(ts_subject, L, R):
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    return (fc_mat + fc_mat.T) / 2.0

def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    ts = ts_subject[:L]
    fs = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    return ((cos_p.T @ cos_p + sin_p.T @ sin_p) / L).astype(np.float32)

def precompute_static_fc(X_raw, real_lengths):
    """Static FC does not depend on band -- computed once for the whole dataset."""
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fc_triu_all = np.zeros((N, n_pairs), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        fc_triu_all[i] = fc_mat[triu_idx]
    return fc_triu_all, triu_idx

def precompute_masked_fc_for_band(X_raw, real_lengths, effective_trs, fc_triu_all, triu_idx,
                                   band, percentile=PERCENTILE):
    """
    Phase-coherence masked FC for ONE band, across the whole dataset.
    Each subject uses its own effective TR so the band means the same
    physical thing regardless of site.
    """
    N = X_raw.shape[0]
    n_pairs = fc_triu_all.shape[1]
    masked_triu_all = np.zeros((N, n_pairs), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        R = X_raw.shape[2]
        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, effective_trs[i], band)
        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = abs_phase_triu >= threshold
        masked_triu_all[i] = np.where(mask, fc_triu_all[i], 0.0).astype(np.float32)
    return masked_triu_all

print("\nPrecomputing static FC (band-independent, once for all subjects)...")
fc_triu_all, triu_idx = precompute_static_fc(X, real_lengths)
print("Static FC shape:", fc_triu_all.shape)

masked_triu_by_band = {}
for band_name, band in BANDS.items():
    print(f"Precomputing phase-coherence-masked FC for band {band_name} "
          f"(each subject uses its own effective TR)...")
    masked_triu_by_band[band_name] = precompute_masked_fc_for_band(
        X, real_lengths, effective_trs, fc_triu_all, triu_idx, band
    )
print("Done. Bands computed:", list(masked_triu_by_band.keys()))

# =====================================================
# 5. SHARED SVM TRAIN/EVAL
# =====================================================
def train_and_eval_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_preds = model.predict(Xva)
    val_auc = roc_auc_score(y_val, val_scores) if len(np.unique(y_val)) > 1 else np.nan
    return val_auc, val_preds

def metrics_from_preds(y_val, val_preds):
    tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = f1_score(y_val, val_preds, zero_division=0)
    acc = accuracy_score(y_val, val_preds)
    return sensitivity, specificity, f1, acc

def evaluate_cv_single_site(feat_all, y, site_mask, skf, label=""):
    """Standard stratified CV within one site (fold object passed in, so
    it can be shared across all feature/band variants for a fair comparison)."""
    X_site = feat_all[site_mask]
    y_site = y[site_mask]
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    for tr_idx, va_idx in skf.split(X_site, y_site):
        auc_val, val_preds = train_and_eval_svm(X_site[tr_idx], y_site[tr_idx], X_site[va_idx], y_site[va_idx])
        sens, spec, f1, acc = metrics_from_preds(y_site[va_idx], val_preds)
        aucs.append(auc_val); senss.append(sens); specs.append(spec); f1s.append(f1); accs.append(acc)
    result = {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}
    print(f"[{label}] N={site_mask.sum()}  "
          f"AUC={np.nanmean(aucs):.3f}±{np.nanstd(aucs):.3f}  "
          f"Sens={np.mean(senss):.3f}±{np.std(senss):.3f}  "
          f"Spec={np.mean(specs):.3f}±{np.std(specs):.3f}  "
          f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}  "
          f"Acc={np.mean(accs):.3f}±{np.std(accs):.3f}")
    return result

# =====================================================
# 6. RUN FULL-STATIC + EVERY BAND'S MASKED FC, PER SITE
# =====================================================
N_SPLITS_DEFAULT = 5
MIN_PER_CLASS = 5
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]

FEATURE_SETS = {"Full Static FC": fc_triu_all}
for band_name, masked_all in masked_triu_by_band.items():
    FEATURE_SETS[f"Masked FC [{band_name}]"] = masked_all

site_results = {}   # site_name -> {variant_name -> result dict}
skipped_sites = []

for site_name in sorted(set(sites)):
    site_mask = sites == site_name
    n_asd = int((y[site_mask] == 1).sum())
    n_td = int((y[site_mask] == 0).sum())
    min_class = min(n_asd, n_td)
    if min_class < MIN_PER_CLASS:
        skipped_sites.append((site_name, n_asd, n_td))
        print(f"\nSkipping {site_name}: only {n_asd} ASD / {n_td} TD "
              f"(need >= {MIN_PER_CLASS} per class for stratified CV).")
        continue
    n_splits_site = min(N_SPLITS_DEFAULT, min_class)
    skf_site = StratifiedKFold(n_splits=n_splits_site, shuffle=True, random_state=42)

    print("\n" + "=" * 70)
    print(f"SITE: {site_name}  (N={site_mask.sum()}, {n_asd} ASD / {n_td} TD, "
          f"{n_splits_site}-fold CV)")
    print("=" * 70)
    site_results[site_name] = {}
    for variant_name, feat_all in FEATURE_SETS.items():
        site_results[site_name][variant_name] = evaluate_cv_single_site(
            feat_all, y, site_mask, skf_site, label=f"{site_name} | {variant_name}"
        )

if skipped_sites:
    print("\n" + "-" * 70)
    print(f"Skipped {len(skipped_sites)} site(s) with insufficient subjects per class:")
    for s, n_asd, n_td in skipped_sites:
        print(f"  {s}: {n_asd} ASD / {n_td} TD")

# =====================================================
# 7. ALL-SITES x ALL-BANDS SUMMARY TABLE + PAIRED TESTS
# =====================================================
included_sites = list(site_results.keys())
band_names = list(BANDS.keys())

print("\n" + "=" * 120)
print("ALL-SITES x ALL-BANDS SUMMARY (mean across each site's folds)")
print("=" * 120)
header = f"{'Site':<10}{'Variant':<38}" + "".join(f"{m.upper():<12}" for m in metrics)
print(header)
print("-" * len(header))
for site_name in included_sites:
    for variant_name in FEATURE_SETS:
        vals_row = site_results[site_name][variant_name]
        row = f"{site_name:<10}{variant_name:<38}"
        for m in metrics:
            vals = np.array(vals_row[m], dtype=np.float64)
            row += f"{np.nanmean(vals):<12.3f}"
        print(row)
    print("-" * len(header))

print("\n" + "=" * 120)
print("PER-SITE PAIRED TEST: each band's Masked FC vs. Full Static FC (paired t-test across that site's own folds)")
print("=" * 120)
for band_name in band_names:
    variant_name = f"Masked FC [{band_name}]"
    print(f"\n-- Band: {band_name} --")
    pt_header = f"{'Site':<10}" + "".join(f"{m.upper()+' delta(p)':<20}" for m in metrics)
    print(pt_header)
    print("-" * len(pt_header))
    for site_name in included_sites:
        row = f"{site_name:<10}"
        for m in metrics:
            masked_vals = np.array(site_results[site_name][variant_name][m], dtype=np.float64)
            full_vals = np.array(site_results[site_name]["Full Static FC"][m], dtype=np.float64)
            delta = np.nanmean(masked_vals) - np.nanmean(full_vals)
            valid = ~(np.isnan(masked_vals) | np.isnan(full_vals))
            if valid.sum() >= 2 and np.std(masked_vals[valid] - full_vals[valid]) > 0:
                _, p_val = stats.ttest_rel(masked_vals[valid], full_vals[valid])
                cell = f"{delta:+.3f} (p={p_val:.3f})"
            else:
                cell = f"{delta:+.3f} (p=n/a)"
            row += f"{cell:<20}"
        print(row)

print("\n" + "-" * 70)
print("POOLED PAIRED TEST across all sites' folds, per band (Masked - Full):")
print("-" * 70)
for band_name in band_names:
    variant_name = f"Masked FC [{band_name}]"
    print(f"\n-- Band: {band_name} --")
    for m in metrics:
        pooled_masked, pooled_full = [], []
        for site_name in included_sites:
            pooled_masked.extend(site_results[site_name][variant_name][m])
            pooled_full.extend(site_results[site_name]["Full Static FC"][m])
        pooled_masked = np.array(pooled_masked, dtype=np.float64)
        pooled_full = np.array(pooled_full, dtype=np.float64)
        valid = ~(np.isnan(pooled_masked) | np.isnan(pooled_full))
        delta = np.nanmean(pooled_masked) - np.nanmean(pooled_full)
        if valid.sum() >= 2 and np.std(pooled_masked[valid] - pooled_full[valid]) > 0:
            _, p_val = stats.ttest_rel(pooled_masked[valid], pooled_full[valid])
            print(f"  {m:<12}: delta={delta:+.3f}  p={p_val:.3f}  (n={valid.sum()} pooled fold-observations)")
        else:
            print(f"  {m:<12}: delta={delta:+.3f}  p=n/a")

# =====================================================
# 8. FIGURES
#    (a) All-sites bar comparison for the default band (kept from the
#        original script, useful as a familiar reference view).
#    (b) Per-site line plot showing AUC/Accuracy across the four
#        frequency bands, with the Full Static FC baseline for context.
#    (c) Pooled across-site view of band performance.
# =====================================================
def plot_all_sites_comparison(site_results, band_name, out_name="all_sites_comparison"):
    variant_name = f"Masked FC [{band_name}]"
    included = list(site_results.keys())
    n_sites = len(included)
    fig, axes = plt.subplots(1, 2, figsize=(max(10, 1.1 * n_sites), 5.5))
    colors = {"Full Static FC": "#1f77b4", variant_name: "#ff7f0e"}
    for ax, metric in zip(axes, ["auc", "accuracy"]):
        x = np.arange(n_sites)
        width = 0.35
        for i, feat_name in enumerate(["Full Static FC", variant_name]):
            vals = [np.nanmean(site_results[s][feat_name][metric]) for s in included]
            offset = (i - 0.5) * width
            ax.bar(x + offset, vals, width, label=feat_name, color=colors[feat_name], alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(included, rotation=45, ha="right")
        ax.set_ylabel(metric.upper()); ax.set_ylim(0, 1.05)
        ax.set_title(f"{metric.upper()} by Site: Full Static FC vs. Masked FC\n(band: {band_name})")
        ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"\nSaved figures/{out_name}.png and .pdf")
    plt.close(fig)

def plot_bands_by_site(site_results, band_names, metric="auc", out_name="bands_by_site"):
    """One line per site, x-axis = frequency band, y-axis = metric.
    A dashed horizontal reference per site marks its Full Static FC score."""
    included = list(site_results.keys())
    fig, ax = plt.subplots(figsize=(max(9, 1.3 * len(band_names)), 6))
    x = np.arange(len(band_names))
    cmap = plt.cm.tab10
    for i, site_name in enumerate(included):
        y_vals = [np.nanmean(site_results[site_name][f"Masked FC [{b}]"][metric]) for b in band_names]
        color = cmap(i % 10)
        ax.plot(x, y_vals, marker="o", label=site_name, color=color)
        static_val = np.nanmean(site_results[site_name]["Full Static FC"][metric])
        ax.axhline(static_val, color=color, linestyle="--", alpha=0.25, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(band_names, rotation=20, ha="right")
    ax.set_ylabel(metric.upper())
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{metric.upper()} vs. Frequency Band, by Site\n(dashed line = that site's Full Static FC baseline)")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}_{metric}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}_{metric}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{out_name}_{metric}.png and .pdf")
    plt.close(fig)

def plot_bands_pooled(site_results, band_names, out_name="bands_pooled"):
    """Mean +/- SEM across sites, per band, for AUC and Accuracy -- the
    highest-level view of 'which frequency band works best overall'."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    included = list(site_results.keys())
    for ax, metric in zip(axes, ["auc", "accuracy"]):
        means, sems = [], []
        for band_name in band_names:
            vals = [np.nanmean(site_results[s][f"Masked FC [{band_name}]"][metric]) for s in included]
            vals = np.array(vals, dtype=np.float64)
            means.append(np.nanmean(vals))
            sems.append(np.nanstd(vals) / max(1, np.sqrt(np.sum(~np.isnan(vals)))))
        static_vals = [np.nanmean(site_results[s]["Full Static FC"][metric]) for s in included]
        static_mean = np.nanmean(static_vals)
        x = np.arange(len(band_names))
        ax.bar(x, means, yerr=sems, capsize=4, color="#ff7f0e", alpha=0.9, label="Masked FC")
        ax.axhline(static_mean, color="#1f77b4", linestyle="--", label="Full Static FC (mean)")
        ax.set_xticks(x)
        ax.set_xticklabels(band_names, rotation=20, ha="right")
        ax.set_ylabel(metric.upper()); ax.set_ylim(0, 1.05)
        ax.set_title(f"Across-Site Mean {metric.upper()} by Band")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{out_name}.png and .pdf")
    plt.close(fig)

plot_all_sites_comparison(site_results, band_name="0.04-0.07 (default, Glerean 2012)")
plot_bands_by_site(site_results, band_names, metric="auc")
plot_bands_by_site(site_results, band_names, metric="accuracy")
plot_bands_pooled(site_results, band_names)


================ MULTI-SITE DATASET SUMMARY ================
  CALTECH: 15 subjects (5 ASD, 10 TD)
  CMU: 11 subjects (6 ASD, 5 TD)
  KKI: 33 subjects (12 ASD, 21 TD)
  LEUVEN: 56 subjects (26 ASD, 30 TD)
  MAXMUN: 46 subjects (19 ASD, 27 TD)
  NYU: 172 subjects (74 ASD, 98 TD)
  OHSU: 25 subjects (12 ASD, 13 TD)
  OLIN: 28 subjects (14 ASD, 14 TD)
  PITT: 50 subjects (24 ASD, 26 TD)
  SBL: 26 subjects (12 ASD, 14 TD)
  SDSU: 27 subjects (8 ASD, 19 TD)
  STANFORD: 25 subjects (12 ASD, 13 TD)
  TRINITY: 44 subjects (19 ASD, 25 TD)
  UCLA: 85 subjects (48 ASD, 37 TD)
  UM: 120 subjects (47 ASD, 73 TD)
  USM: 67 subjects (43 ASD, 24 TD)
  YALE: 41 subjects (22 ASD, 19 TD)
Total subjects: 871

================ PER-SITE TIME-SERIES LENGTH (native, pre-resampling) ================
Site      N     Min     Median  Mean    Max     %Padded   %Subsampled 
----------------------------------------------------------------------
CALTECH   15    146     146.0   146.0   146     100.0     0.0         


/tmp/ipykernel_7177/3196090585.py:250: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=included, showfliers=True)


Saved figures/native_length_by_site.png and .pdf

Effective TR range across loaded subjects: 1.5 - 3.1658291457286434
Subjects requiring subsampling (native_length > TARGET_TIME=200): 295 / 871  -- these get an effective-TR correction; others use nominal site TR.

Precomputing static FC (band-independent, once for all subjects)...


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Static FC shape: (871, 19900)
Precomputing phase-coherence-masked FC for band 0.01-0.03 (slow-4/VLF) (each subject uses its own effective TR)...
Precomputing phase-coherence-masked FC for band 0.03-0.05 (slow-3) (each subject uses its own effective TR)...
Precomputing phase-coherence-masked FC for band 0.04-0.07 (default, Glerean 2012) (each subject uses its own effective TR)...
Precomputing phase-coherence-masked FC for band 0.07-0.10 (slow-2) (each subject uses its own effective TR)...
Done. Bands computed: ['0.01-0.03 (slow-4/VLF)', '0.03-0.05 (slow-3)', '0.04-0.07 (default, Glerean 2012)', '0.07-0.10 (slow-2)']

SITE: CALTECH  (N=15, 5 ASD / 10 TD, 5-fold CV)
[CALTECH | Full Static FC] N=15  AUC=0.600±0.374  Sens=0.000±0.000  Spec=1.000±0.000  F1=0.000±0.000  Acc=0.667±0.000
[CALTECH | Masked FC [0.01-0.03 (slow-4/VLF)]] N=15  AUC=0.600±0.374  Sens=0.000±0.000  Spec=1.000±0.000  F1=0.000±0.000  Acc=0.667±0.000
[CALTECH | Masked FC [0.03-0.05 (slow-3)]] N=15  AUC=0.800±0.400  Sens=0

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Sensitivity Analysis: Phase-Coherence Frequency Band Sweep
# (Phase-Coherence Masked FC ONLY, at the fixed default percentile)
#
# PURPOSE:
# Tests whether the benefit of phase-coherence-guided edge selection
# is specific to a particular oscillatory band, or holds similarly
# across bands (in which case "phase-coherence" may just be acting as
# a generic noise-reduction signal rather than something tied to a
# specific, functionally meaningful frequency range).
#
# BANDS TESTED (Hz):
#   0.01-0.03  slow-4 / very-low-frequency range
#   0.03-0.05  slow-3 range
#   0.04-0.07  the paper's default band (Glerean et al., 2012)
#   0.07-0.10  slow-2 / upper conventional resting-state range
#
# NYQUIST CHECK: with TR = 2.0 s, fs = 1/TR = 0.5 Hz, Nyquist = 0.25 Hz.
# All four bands sit well below Nyquist, so all are filterable. The
# lowest band (0.01-0.03 Hz) has the least temporal resolution --
# a 0.01 Hz oscillation has a ~100 s period, so a 356 s scan (178
# volumes at TR=2s) contains only ~3.5 full cycles at that frequency,
# versus far more cycles for the higher bands. This is flagged in the
# printed output and should be flagged in the paper as a caveat on the
# lowest band specifically, not treated as equally reliable as the
# others.
#
# WHAT IS RECOMPUTED PER BAND vs. REUSED:
# The static FC matrix does not depend on frequency at all and is
# precomputed ONCE, exactly as in the percentile sweep. The
# phase-coherence matrix DOES depend on the band (it's the output of a
# band-pass filter + Hilbert transform), so unlike the percentile
# sweep, it must be recomputed separately for each of the 4 bands here
# -- there is no shortcut available for this dimension.
# =====================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME + 176
SITE_FILTER = "NYU"

NETWORKS = {"WHOLE_BRAIN": list(range(200))}
SELECTED_NETWORK = "WHOLE_BRAIN"
SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {"X": [], "y": [], "subject_ids": [], "sites": [], "real_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (fixed across the entire sweep)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4. NYQUIST / FEASIBILITY CHECK FOR EACH CANDIDATE BAND
# =====================================================
PHASE_TR = 2.0
DEFAULT_PERCENTILE = 25.0   # held fixed for this analysis; only the band varies

BANDS = {
    "0.01-0.03 Hz": (0.01, 0.03),
    "0.03-0.05 Hz": (0.03, 0.05),
    "0.04-0.07 Hz": (0.04, 0.07),   # paper's default band (Glerean et al., 2012)
    "0.07-0.10 Hz": (0.07, 0.10),
}

fs = 1.0 / PHASE_TR
nyquist = fs / 2.0
print(f"\nSampling rate fs = 1/TR = {fs:.3f} Hz  |  Nyquist frequency = {nyquist:.3f} Hz")
print("Feasibility check for each candidate band:")
for name, (low, high) in BANDS.items():
    n_cycles_full_scan = (TARGET_TIME * PHASE_TR) * low  # approx cycles of the LOW edge across the scan
    ok = high < nyquist
    flag = "OK" if ok else "INVALID (>= Nyquist)"
    caveat = "  <-- fewer full cycles resolved; treat with extra caution" if n_cycles_full_scan < 5 else ""
    print(f"  {name:<14} low={low}, high={high}  ~{n_cycles_full_scan:.1f} cycles of low-edge freq "
          f"across full scan  [{flag}]{caveat}")
    assert ok, f"Band {name} is at or above Nyquist ({nyquist} Hz) and cannot be used with TR={PHASE_TR}s."


# =====================================================
# 5. PRECOMPUTE STATIC FC ONCE (band-independent, reused across all bands)
# =====================================================
def compute_static_fc_matrix(ts_subject, L, R):
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    fc_mat = (fc_mat + fc_mat.T) / 2.0
    return fc_mat


def precompute_static_fc(X_raw, real_lengths):
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    fc_triu_all = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        fc_mat = compute_static_fc_matrix(X_raw[i], int(real_lengths[i]), R)
        fc_triu_all[i] = fc_mat[triu_idx]
    return fc_triu_all, triu_idx


print("\nPrecomputing static FC (band-independent, computed once for all bands)...")
fc_triu_all, triu_idx = precompute_static_fc(X, real_lengths)
print("Done. fc_triu_all shape:", fc_triu_all.shape)


# =====================================================
# 6. PHASE-COHERENCE COMPUTATION (recomputed once PER BAND)
# =====================================================
def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    # Explicit, safe padlen: butter(order=4, band) internally doubles
    # to an 8th-order-equivalent filter (2*order coefficients per
    # section), so the default filtfilt padlen requirement is
    # 3*(2*order+1). Passing it explicitly (capped at L-1) avoids the
    # ValueError this pipeline can otherwise hit on short segments
    # (see code-review note on filtfilt padlen).
    padlen = min(3 * (2 * order + 1), ts.shape[0] - 1)
    return filtfilt(b, a, ts, axis=0, padlen=padlen)


def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    ts = ts_subject[:L]
    fs_local = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs_local, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    phase_mat = (cos_p.T @ cos_p + sin_p.T @ sin_p) / L
    return phase_mat.astype(np.float32)


def precompute_phase_coherence_for_band(X_raw, real_lengths, triu_idx, TR, band):
    N, T, R = X_raw.shape
    phase_triu_all = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, TR, band)
        phase_triu_all[i] = phase_mat[triu_idx]
    return phase_triu_all


def masked_features_from_precomputed(fc_triu, phase_triu, percentile):
    abs_phase = np.abs(phase_triu)
    thresholds = np.percentile(abs_phase, percentile, axis=1, keepdims=True)
    mask = abs_phase >= thresholds
    return np.where(mask, fc_triu, 0.0).astype(np.float32)


# =====================================================
# 7. SHARED SVM TRAIN/EVAL
# =====================================================
def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)
    return val_auc, val_preds


def evaluate_band(fc_triu_all, phase_triu_all, y, skf, percentile, C=1.0, kernel="linear"):
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    for tr_idx, va_idx in skf.split(fc_triu_all, y):
        X_train_feat = masked_features_from_precomputed(fc_triu_all[tr_idx], phase_triu_all[tr_idx], percentile)
        X_val_feat = masked_features_from_precomputed(fc_triu_all[va_idx], phase_triu_all[va_idx], percentile)
        y_train, y_val = y[tr_idx], y[va_idx]

        auc_val, val_preds = train_one_fold_svm(X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel)

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

    return {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}


# =====================================================
# 8. RUN THE BAND SWEEP (percentile held fixed at DEFAULT_PERCENTILE)
# =====================================================
print("\n" + "=" * 70)
print(f"FREQUENCY-BAND SWEEP (percentile fixed at P={DEFAULT_PERCENTILE:.0f})")
print("Phase-Coherence Masked FC only -- percentile and all other arms held constant")
print("=" * 70)

band_results = {}
for band_name, band_range in BANDS.items():
    print(f"\n--- Band {band_name} ---")
    phase_triu_band = precompute_phase_coherence_for_band(X, real_lengths, triu_idx, PHASE_TR, band_range)
    res = evaluate_band(fc_triu_all, phase_triu_band, y, skf, DEFAULT_PERCENTILE)
    band_results[band_name] = res
    print(f"  AUC={np.mean(res['auc']):.3f}±{np.std(res['auc']):.3f}  "
          f"Sens={np.mean(res['sensitivity']):.3f}±{np.std(res['sensitivity']):.3f}  "
          f"Spec={np.mean(res['specificity']):.3f}±{np.std(res['specificity']):.3f}  "
          f"F1={np.mean(res['f1']):.3f}±{np.std(res['f1']):.3f}  "
          f"Acc={np.mean(res['accuracy']):.3f}±{np.std(res['accuracy']):.3f}")


# =====================================================
# 9. SUMMARY TABLE + PAIRWISE TESTS vs. THE DEFAULT BAND
# =====================================================
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
default_band_name = "0.04-0.07 Hz"

print("\n" + "=" * 100)
print("BAND-SWEEP SUMMARY TABLE (mean ± SD across folds)")
print("=" * 100)
header = f"{'Band':<16}" + "".join(f"{m.upper():<16}" for m in metrics)
print(header)
print("-" * len(header))
for band_name in BANDS:
    row = f"{band_name:<16}"
    for m in metrics:
        vals = np.array(band_results[band_name][m], dtype=np.float64)
        row += f"{np.mean(vals):.3f}±{np.std(vals):.3f}  "
    print(row)

print("\n" + "-" * 100)
print(f"Paired t-test per metric, each band vs. the default band ({default_band_name}), same folds:")
print("-" * 100)
sig_header = f"{'Band':<16}" + "".join(f"{m.upper()+' p':<16}" for m in metrics)
print(sig_header)
for band_name in BANDS:
    if band_name == default_band_name:
        continue
    row = f"{band_name:<16}"
    for m in metrics:
        vals_band = np.array(band_results[band_name][m], dtype=np.float64)
        vals_default = np.array(band_results[default_band_name][m], dtype=np.float64)
        valid = ~(np.isnan(vals_band) | np.isnan(vals_default))
        if valid.sum() >= 2 and np.std(vals_band[valid] - vals_default[valid]) > 0:
            _, p_val = stats.ttest_rel(vals_band[valid], vals_default[valid])
            row += f"{p_val:.3f}          "
        else:
            row += "n/a            "
    print(row)

print("\n" + "-" * 70)
print("Best-performing band per metric:")
print("-" * 70)
for m in metrics:
    means = {band_name: np.mean(band_results[band_name][m]) for band_name in BANDS}
    best_band = max(means, key=means.get)
    print(f"  {m:<12}: {best_band} (mean={means[best_band]:.3f})  "
          f"[default band {default_band_name}: {means[default_band_name]:.3f}]")


# =====================================================
# 10. BAND-SWEEP FIGURE
# =====================================================
def plot_band_sensitivity(band_results, bands, default_band_name, out_name="band_sensitivity"):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    band_names = list(bands.keys())
    x = np.arange(len(band_names))
    colors = {"auc": "#1f77b4", "sensitivity": "#ff7f0e", "specificity": "#2ca02c",
              "f1": "#d62728", "accuracy": "#9467bd"}

    for m in metrics:
        means = [np.mean(band_results[b][m]) for b in band_names]
        stds = [np.std(band_results[b][m]) for b in band_names]
        ax.errorbar(x, means, yerr=stds, marker="o", markersize=6, capsize=4,
                    lw=1.8, label=m.upper(), color=colors[m])

    default_idx = band_names.index(default_band_name)
    ax.axvline(default_idx, linestyle="--", color="gray", lw=1, label="Default band")

    ax.set_xticks(x)
    ax.set_xticklabels(band_names, rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Sensitivity of Phase-Coherence Masked FC to Frequency Band")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"\nSaved figures/{out_name}.png and .pdf")
    plt.close(fig)


plot_band_sensitivity(band_results, BANDS, default_band_name)

Selected Network : WHOLE_BRAIN
Number of ROIs   : 200

================ DATASET SUMMARY ================
ASD subjects   : 74
TD subjects    : 98
Total subjects : 172
Shape          : (172, 200, 200)

Sampling rate fs = 1/TR = 0.500 Hz  |  Nyquist frequency = 0.250 Hz
Feasibility check for each candidate band:
  0.01-0.03 Hz   low=0.01, high=0.03  ~4.0 cycles of low-edge freq across full scan  [OK]  <-- fewer full cycles resolved; treat with extra caution
  0.03-0.05 Hz   low=0.03, high=0.05  ~12.0 cycles of low-edge freq across full scan  [OK]
  0.04-0.07 Hz   low=0.04, high=0.07  ~16.0 cycles of low-edge freq across full scan  [OK]
  0.07-0.10 Hz   low=0.07, high=0.1  ~28.0 cycles of low-edge freq across full scan  [OK]

Precomputing static FC (band-independent, computed once for all bands)...


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Done. fc_triu_all shape: (172, 19900)

FREQUENCY-BAND SWEEP (percentile fixed at P=25)
Phase-Coherence Masked FC only -- percentile and all other arms held constant

--- Band 0.01-0.03 Hz ---
  AUC=0.698±0.079  Sens=0.539±0.087  Spec=0.734±0.041  F1=0.569±0.082  Acc=0.651±0.058

--- Band 0.03-0.05 Hz ---
  AUC=0.705±0.090  Sens=0.485±0.081  Spec=0.796±0.043  F1=0.550±0.074  Acc=0.662±0.043

--- Band 0.04-0.07 Hz ---
  AUC=0.713±0.100  Sens=0.552±0.079  Spec=0.826±0.026  F1=0.617±0.063  Acc=0.709±0.035

--- Band 0.07-0.10 Hz ---
  AUC=0.702±0.076  Sens=0.553±0.072  Spec=0.767±0.064  F1=0.593±0.069  Acc=0.675±0.052

BAND-SWEEP SUMMARY TABLE (mean ± SD across folds)
Band            AUC             SENSITIVITY     SPECIFICITY     F1              ACCURACY        
------------------------------------------------------------------------------------------------
0.01-0.03 Hz    0.698±0.079  0.539±0.087  0.734±0.041  0.569±0.082  0.651±0.058  
0.03-0.05 Hz    0.705±0.090  0.485±0.081  0.796±0.04

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Sensitivity Analysis: Phase-Coherence Frequency Band Sweep
# + STATIC FC BASELINE FOR COMPARISON
#
# PURPOSE:
# Tests whether the benefit of phase-coherence-guided edge selection
# is specific to a particular oscillatory band, or holds similarly
# across bands (in which case "phase-coherence" may just be acting as
# a generic noise-reduction signal rather than something tied to a
# specific, functionally meaningful frequency range). The plain STATIC
# FC classifier (no phase-coherence masking at all) is now included as
# a band-independent baseline, evaluated on the SAME folds as every
# band, so you can see directly whether masking at any band actually
# beats just using the full static FC matrix.
#
# BANDS TESTED (Hz):
#   0.01-0.03  slow-4 / very-low-frequency range
#   0.03-0.05  slow-3 range
#   0.04-0.07  the paper's default band (Glerean et al., 2012)
#   0.07-0.10  slow-2 / upper conventional resting-state range
#
# NYQUIST CHECK: with TR = 2.0 s, fs = 1/TR = 0.5 Hz, Nyquist = 0.25 Hz.
# All four bands sit well below Nyquist, so all are filterable. The
# lowest band (0.01-0.03 Hz) has the least temporal resolution --
# a 0.01 Hz oscillation has a ~100 s period, so a 356 s scan (178
# volumes at TR=2s) contains only ~3.5 full cycles at that frequency,
# versus far more cycles for the higher bands. This is flagged in the
# printed output and should be flagged in the paper as a caveat on the
# lowest band specifically, not treated as equally reliable as the
# others.
#
# WHAT IS RECOMPUTED PER BAND vs. REUSED:
# The static FC matrix does not depend on frequency at all and is
# precomputed ONCE -- it is used both (a) as the edge-weight source
# that gets phase-coherence-masked for every band, and (b) on its own,
# unmasked, as the STATIC FC BASELINE described above. The
# phase-coherence matrix DOES depend on the band (it's the output of a
# band-pass filter + Hilbert transform), so it must be recomputed
# separately for each of the 4 bands -- there is no shortcut available
# for this dimension.
# =====================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME =176
SITE_FILTER = "SDSU"

NETWORKS = {"WHOLE_BRAIN": list(range(200))}
SELECTED_NETWORK = "WHOLE_BRAIN"
SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {"X": [], "y": [], "subject_ids": [], "sites": [], "real_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (fixed across the entire sweep, INCLUDING
#    the static FC baseline, so every comparison is fold-matched)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4. NYQUIST / FEASIBILITY CHECK FOR EACH CANDIDATE BAND
# =====================================================
PHASE_TR = 3
DEFAULT_PERCENTILE = 25.0   # held fixed for this analysis; only the band varies

BANDS = {
    "0.01-0.03 Hz": (0.01, 0.03),
    "0.03-0.05 Hz": (0.03, 0.05),
    "0.04-0.07 Hz": (0.04, 0.07),   # paper's default band (Glerean et al., 2012)
    "0.07-0.10 Hz": (0.07, 0.10),
}

fs = 1.0 / PHASE_TR
nyquist = fs / 2.0
print(f"\nSampling rate fs = 1/TR = {fs:.3f} Hz  |  Nyquist frequency = {nyquist:.3f} Hz")
print("Feasibility check for each candidate band:")
for name, (low, high) in BANDS.items():
    n_cycles_full_scan = (TARGET_TIME * PHASE_TR) * low  # approx cycles of the LOW edge across the scan
    ok = high < nyquist
    flag = "OK" if ok else "INVALID (>= Nyquist)"
    caveat = "  <-- fewer full cycles resolved; treat with extra caution" if n_cycles_full_scan < 5 else ""
    print(f"  {name:<14} low={low}, high={high}  ~{n_cycles_full_scan:.1f} cycles of low-edge freq "
          f"across full scan  [{flag}]{caveat}")
    assert ok, f"Band {name} is at or above Nyquist ({nyquist} Hz) and cannot be used with TR={PHASE_TR}s."


# =====================================================
# 5. PRECOMPUTE STATIC FC ONCE (band-independent, reused across all
#    bands AND used directly, unmasked, as the static-FC baseline)
# =====================================================
def compute_static_fc_matrix(ts_subject, L, R):
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    fc_mat = (fc_mat + fc_mat.T) / 2.0
    return fc_mat


def precompute_static_fc(X_raw, real_lengths):
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    fc_triu_all = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        fc_mat = compute_static_fc_matrix(X_raw[i], int(real_lengths[i]), R)
        fc_triu_all[i] = fc_mat[triu_idx]
    return fc_triu_all, triu_idx


print("\nPrecomputing static FC (band-independent, computed once for all bands)...")
fc_triu_all, triu_idx = precompute_static_fc(X, real_lengths)
print("Done. fc_triu_all shape:", fc_triu_all.shape)


# =====================================================
# 6. PHASE-COHERENCE COMPUTATION (recomputed once PER BAND)
# =====================================================
def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    # Explicit, safe padlen: butter(order=4, band) internally doubles
    # to an 8th-order-equivalent filter (2*order coefficients per
    # section), so the default filtfilt padlen requirement is
    # 3*(2*order+1). Passing it explicitly (capped at L-1) avoids the
    # ValueError this pipeline can otherwise hit on short segments
    # (see code-review note on filtfilt padlen).
    padlen = min(3 * (2 * order + 1), ts.shape[0] - 1)
    return filtfilt(b, a, ts, axis=0, padlen=padlen)


def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    ts = ts_subject[:L]
    fs_local = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs_local, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    phase_mat = (cos_p.T @ cos_p + sin_p.T @ sin_p) / L
    return phase_mat.astype(np.float32)


def precompute_phase_coherence_for_band(X_raw, real_lengths, triu_idx, TR, band):
    N, T, R = X_raw.shape
    phase_triu_all = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, TR, band)
        phase_triu_all[i] = phase_mat[triu_idx]
    return phase_triu_all


def masked_features_from_precomputed(fc_triu, phase_triu, percentile):
    abs_phase = np.abs(phase_triu)
    thresholds = np.percentile(abs_phase, percentile, axis=1, keepdims=True)
    mask = abs_phase >= thresholds
    return np.where(mask, fc_triu, 0.0).astype(np.float32)


# =====================================================
# 7. SHARED SVM TRAIN/EVAL
# =====================================================
def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)
    return val_auc, val_preds


def evaluate_band(fc_triu_all, phase_triu_all, y, skf, percentile, C=1.0, kernel="linear"):
    """Phase-coherence MASKED FC for one band."""
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    for tr_idx, va_idx in skf.split(fc_triu_all, y):
        X_train_feat = masked_features_from_precomputed(fc_triu_all[tr_idx], phase_triu_all[tr_idx], percentile)
        X_val_feat = masked_features_from_precomputed(fc_triu_all[va_idx], phase_triu_all[va_idx], percentile)
        y_train, y_val = y[tr_idx], y[va_idx]

        auc_val, val_preds = train_one_fold_svm(X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel)

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

    return {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}


def evaluate_static_fc(fc_triu_all, y, skf, C=1.0, kernel="linear"):
    """
    Plain STATIC FC baseline -- no phase-coherence masking at all, just
    the full static FC edge vector fed straight to the SVM. Uses the
    SAME skf fold object as evaluate_band, so it is directly comparable
    (fold-matched) to every band's masked-FC result.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    for tr_idx, va_idx in skf.split(fc_triu_all, y):
        X_train_feat = fc_triu_all[tr_idx]
        X_val_feat = fc_triu_all[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        auc_val, val_preds = train_one_fold_svm(X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel)

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

    return {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}


# =====================================================
# 8. RUN THE STATIC FC BASELINE + THE BAND SWEEP
#    (percentile held fixed at DEFAULT_PERCENTILE for the masked arms;
#    static FC has no percentile/band -- it's the same folds, same
#    features, evaluated once)
# =====================================================
print("\n" + "=" * 70)
print("STATIC FC BASELINE (no phase-coherence masking)")
print("=" * 70)
static_fc_result = evaluate_static_fc(fc_triu_all, y, skf)
print(f"  AUC={np.mean(static_fc_result['auc']):.3f}±{np.std(static_fc_result['auc']):.3f}  "
      f"Sens={np.mean(static_fc_result['sensitivity']):.3f}±{np.std(static_fc_result['sensitivity']):.3f}  "
      f"Spec={np.mean(static_fc_result['specificity']):.3f}±{np.std(static_fc_result['specificity']):.3f}  "
      f"F1={np.mean(static_fc_result['f1']):.3f}±{np.std(static_fc_result['f1']):.3f}  "
      f"Acc={np.mean(static_fc_result['accuracy']):.3f}±{np.std(static_fc_result['accuracy']):.3f}")

print("\n" + "=" * 70)
print(f"FREQUENCY-BAND SWEEP (percentile fixed at P={DEFAULT_PERCENTILE:.0f})")
print("Phase-Coherence Masked FC, per band -- percentile and all other arms held constant")
print("=" * 70)

band_results = {}
for band_name, band_range in BANDS.items():
    print(f"\n--- Band {band_name} ---")
    phase_triu_band = precompute_phase_coherence_for_band(X, real_lengths, triu_idx, PHASE_TR, band_range)
    res = evaluate_band(fc_triu_all, phase_triu_band, y, skf, DEFAULT_PERCENTILE)
    band_results[band_name] = res
    print(f"  AUC={np.mean(res['auc']):.3f}±{np.std(res['auc']):.3f}  "
          f"Sens={np.mean(res['sensitivity']):.3f}±{np.std(res['sensitivity']):.3f}  "
          f"Spec={np.mean(res['specificity']):.3f}±{np.std(res['specificity']):.3f}  "
          f"F1={np.mean(res['f1']):.3f}±{np.std(res['f1']):.3f}  "
          f"Acc={np.mean(res['accuracy']):.3f}±{np.std(res['accuracy']):.3f}")

# Combined dict, with the static baseline listed first, used throughout
# sections 9-10 so every table/plot includes it alongside the bands.
all_results = {"Static FC (no masking)": static_fc_result}
all_results.update(band_results)


# =====================================================
# 9. SUMMARY TABLE + PAIRWISE TESTS
#    (each band vs. the default band, AND each band vs. the static FC
#    baseline, so you can see both "does band choice matter" and
#    "does masking help at all vs. just using static FC")
# =====================================================
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
default_band_name = "0.04-0.07 Hz"
static_name = "Static FC (no masking)"

print("\n" + "=" * 100)
print("SUMMARY TABLE: Static FC baseline + Band Sweep (mean ± SD across folds)")
print("=" * 100)
header = f"{'Variant':<24}" + "".join(f"{m.upper():<16}" for m in metrics)
print(header)
print("-" * len(header))
for variant_name in all_results:
    row = f"{variant_name:<24}"
    for m in metrics:
        vals = np.array(all_results[variant_name][m], dtype=np.float64)
        row += f"{np.mean(vals):.3f}±{np.std(vals):.3f}  "
    print(row)

def paired_test_row(name_a, name_b, all_results, metrics):
    row = f"{name_a:<24}"
    for m in metrics:
        vals_a = np.array(all_results[name_a][m], dtype=np.float64)
        vals_b = np.array(all_results[name_b][m], dtype=np.float64)
        valid = ~(np.isnan(vals_a) | np.isnan(vals_b))
        if valid.sum() >= 2 and np.std(vals_a[valid] - vals_b[valid]) > 0:
            _, p_val = stats.ttest_rel(vals_a[valid], vals_b[valid])
            row += f"{p_val:<16.3f}"
        else:
            row += f"{'n/a':<16}"
    return row

print("\n" + "-" * 100)
print(f"Paired t-test per metric, each BAND vs. the default band ({default_band_name}), same folds:")
print("-" * 100)
sig_header = f"{'Band':<24}" + "".join(f"{m.upper()+' p':<16}" for m in metrics)
print(sig_header)
for band_name in BANDS:
    if band_name == default_band_name:
        continue
    print(paired_test_row(band_name, default_band_name, all_results, metrics))

print("\n" + "-" * 100)
print(f"Paired t-test per metric, each VARIANT vs. the Static FC baseline, same folds "
      f"(does masking beat plain static FC?):")
print("-" * 100)
print(sig_header)
for variant_name in all_results:
    if variant_name == static_name:
        continue
    print(paired_test_row(variant_name, static_name, all_results, metrics))

print("\n" + "-" * 70)
print("Best-performing variant per metric (Static FC baseline included):")
print("-" * 70)
for m in metrics:
    means = {variant_name: np.mean(all_results[variant_name][m]) for variant_name in all_results}
    best_variant = max(means, key=means.get)
    print(f"  {m:<12}: {best_variant} (mean={means[best_variant]:.3f})  "
          f"[static FC baseline: {means[static_name]:.3f}, "
          f"default band {default_band_name}: {means[default_band_name]:.3f}]")


# =====================================================
# 10. BAND-SWEEP FIGURE (Static FC shown as a dashed reference line
#     per metric, since it doesn't sit on the frequency axis)
# =====================================================
def plot_band_sensitivity(band_results, static_fc_result, bands, default_band_name,
                           out_name="band_sensitivity"):
    fig, ax = plt.subplots(figsize=(8, 5.5))
    band_names = list(bands.keys())
    x = np.arange(len(band_names))
    colors = {"auc": "#1f77b4", "sensitivity": "#ff7f0e", "specificity": "#2ca02c",
              "f1": "#d62728", "accuracy": "#9467bd"}

    for m in metrics:
        means = [np.mean(band_results[b][m]) for b in band_names]
        stds = [np.std(band_results[b][m]) for b in band_names]
        ax.errorbar(x, means, yerr=stds, marker="o", markersize=6, capsize=4,
                    lw=1.8, label=m.upper(), color=colors[m])
        # Static FC baseline for this metric: a dashed horizontal line
        # in the same color, since it has no frequency-band position.
        static_mean = np.mean(static_fc_result[m])
        ax.axhline(static_mean, color=colors[m], linestyle=":", linewidth=1.2, alpha=0.6)

    default_idx = band_names.index(default_band_name)
    ax.axvline(default_idx, linestyle="--", color="gray", lw=1, label="Default band")
    # One legend entry explaining the dotted lines generically (avoids
    # 5 near-duplicate "Static FC (metric)" entries cluttering the legend).
    ax.plot([], [], color="black", linestyle=":", linewidth=1.2, alpha=0.6,
            label="Static FC baseline (per metric)")

    ax.set_xticks(x)
    ax.set_xticklabels(band_names, rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Sensitivity of Phase-Coherence Masked FC to Frequency Band\n"
                 "(dotted lines = plain Static FC baseline, no masking)")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"\nSaved figures/{out_name}.png and .pdf")
    plt.close(fig)


plot_band_sensitivity(band_results, static_fc_result, BANDS, default_band_name)

Selected Network : WHOLE_BRAIN
Number of ROIs   : 200

================ DATASET SUMMARY ================
ASD subjects   : 8
TD subjects    : 19
Total subjects : 27
Shape          : (27, 176, 200)

Sampling rate fs = 1/TR = 0.333 Hz  |  Nyquist frequency = 0.167 Hz
Feasibility check for each candidate band:
  0.01-0.03 Hz   low=0.01, high=0.03  ~5.3 cycles of low-edge freq across full scan  [OK]
  0.03-0.05 Hz   low=0.03, high=0.05  ~15.8 cycles of low-edge freq across full scan  [OK]
  0.04-0.07 Hz   low=0.04, high=0.07  ~21.1 cycles of low-edge freq across full scan  [OK]
  0.07-0.10 Hz   low=0.07, high=0.1  ~37.0 cycles of low-edge freq across full scan  [OK]

Precomputing static FC (band-independent, computed once for all bands)...
Done. fc_triu_all shape: (27, 19900)

STATIC FC BASELINE (no phase-coherence masking)


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  AUC=0.483±0.244  Sens=0.300±0.400  Spec=0.900±0.122  F1=0.267±0.327  Acc=0.707±0.132

FREQUENCY-BAND SWEEP (percentile fixed at P=25)
Phase-Coherence Masked FC, per band -- percentile and all other arms held constant

--- Band 0.01-0.03 Hz ---
  AUC=0.575±0.218  Sens=0.100±0.200  Spec=0.950±0.100  F1=0.133±0.267  Acc=0.700±0.099

--- Band 0.03-0.05 Hz ---
  AUC=0.600±0.122  Sens=0.100±0.200  Spec=0.900±0.122  F1=0.133±0.267  Acc=0.667±0.128

--- Band 0.04-0.07 Hz ---
  AUC=0.533±0.282  Sens=0.100±0.200  Spec=0.900±0.122  F1=0.133±0.267  Acc=0.667±0.128

--- Band 0.07-0.10 Hz ---
  AUC=0.542±0.239  Sens=0.100±0.200  Spec=0.900±0.122  F1=0.133±0.267  Acc=0.667±0.128

SUMMARY TABLE: Static FC baseline + Band Sweep (mean ± SD across folds)
Variant                 AUC             SENSITIVITY     SPECIFICITY     F1              ACCURACY        
--------------------------------------------------------------------------------------------------------
Static FC (no masking)  0.483±0.244  0.300

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Sensitivity Analysis: Percentile Threshold Sweep
# (Phase-Coherence Masked FC ONLY -- other arms/controls dropped)
#
# PURPOSE:
# Section 5.3 of the paper notes that the percentile threshold
# (P = 25%) was fixed a priori rather than tuned or swept. This script
# addresses that gap directly: it sweeps P across a grid of values and
# evaluates the Phase-Coherence Masked FC classifier at each one, on
# the SAME cross-validation folds throughout, so any change in
# performance across the grid is attributable to the threshold itself.
#
# EFFICIENCY NOTE:
# The static FC matrix and the phase-coherence matrix for a given
# subject do NOT depend on the percentile threshold -- only the
# masking step does. Recomputing the band-pass filter + Hilbert
# transform for every subject at every grid point would be wasteful
# (13 grid points x 5 folds x ~172 subjects x redundant identical
# filtering). Instead, both matrices are computed ONCE per subject up
# front, and the percentile sweep only re-applies a cheap vectorized
# threshold + mask to those fixed, precomputed matrices. This turns an
# O(grid_size) repetition of the expensive step into a single pass.
#
# BONUS: percentile = 0 keeps every edge (threshold = min value, so
# every edge clears it), which is mathematically identical to the
# unmasked Full Static FC baseline. Including 0 in the grid therefore
# gives that baseline "for free" as the natural left-endpoint of the
# sweep, without needing a separate feature function or arm.
# =====================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"

NETWORKS = {"WHOLE_BRAIN": list(range(200))}
SELECTED_NETWORK = "WHOLE_BRAIN"
SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {"X": [], "y": [], "subject_ids": [], "sites": [], "real_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (fixed across the entire sweep)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4. PRECOMPUTE STATIC FC + PHASE-COHERENCE (ONCE, for every subject)
# =====================================================
PHASE_TR = 2.0
PHASE_BAND = (0.04, 0.07)
DEFAULT_PERCENTILE = 25.0   # the value used in the main paper, marked on the sensitivity plot


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_matrix(ts_subject, L, R):
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    fc_mat = (fc_mat + fc_mat.T) / 2.0
    return fc_mat


def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    ts = ts_subject[:L]
    fs = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    phase_mat = (cos_p.T @ cos_p + sin_p.T @ sin_p) / L
    return phase_mat.astype(np.float32)


def precompute_fc_and_phase(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND):
    """
    Runs ONCE for the whole dataset. Returns two (N, n_pairs) arrays:
    the static FC upper-triangle and the phase-coherence upper-triangle
    per subject. Every percentile value in the sweep re-uses these same
    two arrays -- only the threshold/mask step is repeated per grid
    point, which is cheap (vectorized, no filtering or Hilbert
    transform involved).
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fc_triu_all = np.zeros((N, n_pairs), dtype=np.float32)
    phase_triu_all = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, TR, band)
        fc_triu_all[i] = fc_mat[triu_idx]
        phase_triu_all[i] = phase_mat[triu_idx]

    return fc_triu_all, phase_triu_all


print("\nPrecomputing static FC and phase-coherence matrices for all subjects "
      "(this runs once; the percentile sweep below re-uses these matrices)...")
fc_triu_all, phase_triu_all = precompute_fc_and_phase(X, real_lengths)
print("Done. fc_triu_all shape:", fc_triu_all.shape, " phase_triu_all shape:", phase_triu_all.shape)


def masked_features_from_precomputed(fc_triu, phase_triu, percentile):
    """
    Vectorized percentile masking applied to already-computed FC/phase
    arrays for a SUBSET of subjects (e.g. one fold's train or val
    rows). percentile=0 retains every edge (threshold = each row's own
    minimum |phase-coherence|, which every value in that row clears by
    definition), reproducing the unmasked Full Static FC feature as the
    natural left-endpoint of the sweep.
    """
    abs_phase = np.abs(phase_triu)
    # Per-subject (per-row) percentile threshold.
    thresholds = np.percentile(abs_phase, percentile, axis=1, keepdims=True)
    mask = abs_phase >= thresholds
    return np.where(mask, fc_triu, 0.0).astype(np.float32)


# =====================================================
# 5. SHARED SVM TRAIN/EVAL
# =====================================================
def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)
    return val_auc, val_preds


def evaluate_percentile(percentile, fc_triu_all, phase_triu_all, y, skf, C=1.0, kernel="linear"):
    """
    Evaluates the Phase-Coherence Masked FC classifier at a single
    percentile threshold, across all folds of the SAME skf object used
    for every other percentile value in the sweep.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for tr_idx, va_idx in skf.split(fc_triu_all, y):
        X_train_feat = masked_features_from_precomputed(
            fc_triu_all[tr_idx], phase_triu_all[tr_idx], percentile
        )
        X_val_feat = masked_features_from_precomputed(
            fc_triu_all[va_idx], phase_triu_all[va_idx], percentile
        )
        y_train, y_val = y[tr_idx], y[va_idx]

        auc_val, val_preds = train_one_fold_svm(X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel)

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

    return {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}


# =====================================================
# 6. RUN THE PERCENTILE SWEEP
# =====================================================
# PERCENTILE_GRID values follow the same convention as the main
# pipeline: value P means "discard the bottom P% of |phase-coherence|
# edges, keep the top (100-P)%". P=0 keeps 100% of edges (= Full
# Static FC, included here as the sweep's natural left-endpoint).
PERCENTILE_GRID = [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90]

print("\n" + "=" * 70)
print(f"PERCENTILE SWEEP: {PERCENTILE_GRID}")
print("(Phase-Coherence Masked FC only -- all other arms dropped for this analysis)")
print("=" * 70)

sweep_results = {}
for p in PERCENTILE_GRID:
    keep_pct = 100 - p
    print(f"\n--- Percentile P={p} (retaining top {keep_pct}% of edges) ---")
    res = evaluate_percentile(p, fc_triu_all, phase_triu_all, y, skf)
    sweep_results[p] = res
    print(f"  AUC={np.mean(res['auc']):.3f}±{np.std(res['auc']):.3f}  "
          f"Sens={np.mean(res['sensitivity']):.3f}±{np.std(res['sensitivity']):.3f}  "
          f"Spec={np.mean(res['specificity']):.3f}±{np.std(res['specificity']):.3f}  "
          f"F1={np.mean(res['f1']):.3f}±{np.std(res['f1']):.3f}  "
          f"Acc={np.mean(res['accuracy']):.3f}±{np.std(res['accuracy']):.3f}")


# =====================================================
# 7. SUMMARY TABLE + SIGNIFICANCE vs. P=0 (NO MASKING) BASELINE
# =====================================================
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]

print("\n" + "=" * 100)
print("SENSITIVITY-ANALYSIS SUMMARY TABLE (mean ± SD across folds, paired p vs. P=0 baseline)")
print("=" * 100)
header = f"{'P (discard%)':<14}{'Keep %':<9}" + "".join(f"{m.upper():<16}" for m in metrics)
print(header)
print("-" * len(header))

baseline_p = 0
for p in PERCENTILE_GRID:
    row = f"{p:<14}{100-p:<9}"
    for m in metrics:
        vals = np.array(sweep_results[p][m], dtype=np.float64)
        row += f"{np.mean(vals):.3f}±{np.std(vals):.3f}  "
    print(row)

print("\n" + "-" * 100)
print(f"Paired t-test per metric, each P vs. P={baseline_p} (no masking) baseline, same folds:")
print("-" * 100)
sig_header = f"{'P (discard%)':<14}" + "".join(f"{m.upper()+' p':<16}" for m in metrics)
print(sig_header)
for p in PERCENTILE_GRID:
    if p == baseline_p:
        continue
    row = f"{p:<14}"
    for m in metrics:
        vals_p = np.array(sweep_results[p][m], dtype=np.float64)
        vals_base = np.array(sweep_results[baseline_p][m], dtype=np.float64)
        valid = ~(np.isnan(vals_p) | np.isnan(vals_base))
        if valid.sum() >= 2 and np.std(vals_p[valid] - vals_base[valid]) > 0:
            _, p_val = stats.ttest_rel(vals_p[valid], vals_base[valid])
            row += f"{p_val:.3f}          "
        else:
            row += "n/a            "
    print(row)

# Identify the best-performing percentile per metric (for discussion in the paper).
print("\n" + "-" * 70)
print("Best percentile per metric (highest cross-validated mean):")
print("-" * 70)
for m in metrics:
    means = {p: np.mean(sweep_results[p][m]) for p in PERCENTILE_GRID}
    best_p = max(means, key=means.get)
    print(f"  {m:<12}: P={best_p} (keep {100-best_p}%), mean={means[best_p]:.3f}  "
          f"[value at default P={DEFAULT_PERCENTILE:.0f}: {means.get(int(DEFAULT_PERCENTILE), float('nan')):.3f}]")


# =====================================================
# 8. SENSITIVITY-ANALYSIS FIGURE
# =====================================================
def plot_percentile_sensitivity(sweep_results, percentile_grid, default_percentile,
                                 out_name="percentile_sensitivity"):
    """
    One line per metric, x-axis = percent of edges KEPT (100-P), so the
    plot reads left-to-right as "more aggressive masking -> less
    masking", with the default operating point marked.
    """
    fig, ax = plt.subplots(figsize=(8, 5.5))
    keep_pcts = [100 - p for p in percentile_grid]
    colors = {"auc": "#1f77b4", "sensitivity": "#ff7f0e", "specificity": "#2ca02c",
              "f1": "#d62728", "accuracy": "#9467bd"}

    for m in metrics:
        means = [np.mean(sweep_results[p][m]) for p in percentile_grid]
        stds = [np.std(sweep_results[p][m]) for p in percentile_grid]
        ax.errorbar(keep_pcts, means, yerr=stds, marker="o", markersize=4,
                    capsize=3, lw=1.8, label=m.upper(), color=colors[m])

    default_keep_pct = 100 - default_percentile
    ax.axvline(default_keep_pct, linestyle="--", color="gray", lw=1,
               label=f"Default (keep {default_keep_pct:.0f}%)")

    ax.set_xlabel("Percent of Edges Retained (100 - P)")
    ax.set_ylabel("Score")
    ax.set_title("Sensitivity of Phase-Coherence Masked FC to Percentile Threshold")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"\nSaved figures/{out_name}.png and .pdf")
    plt.close(fig)


plot_percentile_sensitivity(sweep_results, PERCENTILE_GRID, DEFAULT_PERCENTILE)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# All-Sites Comparison: Plain Static FC vs. Percentile-Masked FC
#
# Loads every available ABIDE site, then runs a WITHIN-SITE 5-fold
# stratified CV comparison between Full Static FC and Phase-Coherence
# Masked FC, separately for each site (both feature types share the
# exact same folds within a site, so any difference between them is
# attributable to the feature, not to fold composition). This answers
# a narrower and more direct question than leave-one-site-out
# generalization: "does percentile masking help or hurt, site by
# site, when each site is evaluated on its own subjects?" -- i.e. does
# the earlier NYU-only result replicate across the other ABIDE sites,
# rather than testing cross-site transfer.
#
# CRITICAL METHODOLOGICAL FIX FOR MULTI-SITE PHASE-COHERENCE:
# The main pipeline assumed a single fixed TR (2.0s) for the phase-
# coherence band-pass filter. Different ABIDE sites use different TRs
# (documented below), so naively reusing one fixed TR across sites
# would silently make the nominal 0.04-0.07 Hz band correspond to a
# DIFFERENT physical frequency range at each site -- invalidating the
# comparison. This script fixes that by computing phase-coherence with
# each subject's OWN site-specific TR (SITE_TR_MAP below), so the
# frequency band means the same physical thing for every subject
# regardless of which site they came from.
#
# It ALSO handles the more subtle version of this problem: subjects
# whose native scan length exceeds TARGET_TIME are subsampled (not
# just padded), which stretches/compresses the EFFECTIVE sampling
# interval away from the nominal TR (see code-review note on this from
# the main pipeline). This script tracks each subject's native
# (pre-resampling) length and computes an EFFECTIVE per-subject TR
# that accounts for this, rather than only correcting for site-level
# TR differences and leaving the subsampling-level distortion
# unaddressed.
#
# SITE TR REFERENCE TABLE (all 15 ABIDE-I sites; SITE_TR_MAP below
# covers every one of them). Verify every value against your own ABIDE
# phenotypic file before trusting results -- sourced from a published
# ABIDE-I acquisition-parameters table (TR converted from ms to
# seconds):
#   CALTECH : TR = 2.000s      OLIN    : TR = 1.500s
#   CMU     : TR = 2.000s      SBL     : TR = 2.200s
#   KKI     : TR = 2.500s      SDSU    : TR = 2.000s
#   LEUVEN  : TR = 1.667s      TRINITY : TR = 2.000s
#   MAXMUN  : TR = 3.000s      UCLA    : TR = 3.000s
#   NYU     : TR = 2.000s      UM      : TR = 2.000s
#   PITT    : TR = 1.500s      USM     : TR = 2.000s
#   YALE    : TR = 2.000s
# SITES_TO_LOAD = None loads every site found; sites with too few
# subjects per class for meaningful stratified CV are skipped
# automatically (see MIN_PER_CLASS below), with a printed warning.
# =====================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "font.size": 11, "font.family": "serif",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER -- multi-site, tracks native length AND post-resample
#    real (non-padded) length, since both are needed for the effective-
#    TR correction below.
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITES_TO_LOAD = None   # None loads every site found; set to a list (e.g. ["NYU","UM","PITT"]) to restrict

# Verify each of these against your ABIDE phenotypic CSV before trusting
# results (Phenotypic_V1_0b_preprocessed1.csv, column "TR" or the site's
# published acquisition parameters). Any loaded site NOT in this map
# raises an error below rather than silently guessing a TR.
#
# Source: published ABIDE-I acquisition-parameters table (TR converted
# from ms to seconds). Two site-naming notes:
#   - "Ludwig" in the source table is the ABIDE site more commonly
#     folder-named "MaxMun" (Ludwig-Maximilians-University Munich);
#     mapped here as MAXMUN -- verify this matches your actual folder
#     prefix before running.
#   - LEUVEN and UCLA each have two ABIDE sub-site variants in the raw
#     data (LEUVEN_1/LEUVEN_2, UCLA_1/UCLA_2) that may differ slightly
#     in acquisition details; parse_site_from_filename() collapses both
#     to a single "LEUVEN" / "UCLA" key (splits on the first "_"), so
#     both sub-sites share the one TR value below. If your two Leuven
#     or UCLA sub-sites actually used different TRs, split them out
#     with a more specific site-parsing rule before trusting this table.
SITE_TR_MAP = {
    "CALTECH": 2.0,
    "CMU": 2.0,
    "KKI": 2.5,
    "LEUVEN": 1.667,
    "MAXMUN": 3.0,     # "Ludwig" in the source table
    "NYU": 2.0,
    "OLIN": 1.5,
    "SBL": 2.2,
    "SDSU": 2.0,
    "TRINITY": 2.0,
    "UCLA": 3.0,
    "UM": 2.0,         # "UMich" in the source table
    "PITT": 1.5,       # "UPitt" in the source table
    "USM": 2.0,
    "YALE": 2.0,
    "STANFORD": 2.0,
    "OHSU": 2.0,
}

NETWORKS = {"WHOLE_BRAIN": list(range(200))}
SELECTED_ROIS = NETWORKS["WHOLE_BRAIN"]


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label, sites_to_load=None):
    """
    Same as the main pipeline's loader, but additionally records:
      - site           : parsed site code
      - native_length  : timepoints BEFORE any subsample/pad (needed to
                          detect and correct for subsampling-induced
                          effective-TR distortion)
      - real_length    : timepoints actually carrying real (non-padded)
                          signal after resampling, as before
    """
    results = {"X": [], "y": [], "subject_ids": [], "sites": [],
               "real_lengths": [], "native_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if sites_to_load is not None and site.upper() not in [s.upper() for s in sites_to_load]:
            continue

        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape); continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape); continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            native_length = ts.shape[0]   # BEFORE any resampling

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site.upper())   # normalize case so real ABIDE
            # filenames like "Pitt_..." or "Caltech_..." (mixed case) match the
            # all-uppercase keys in SITE_TR_MAP and the "NYU"/"UM"/"PITT"-style
            # literals used throughout this script for site comparisons.
            results["real_lengths"].append(real_length)
            results["native_lengths"].append(native_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1, sites_to_load=SITES_TO_LOAD)
td_data  = load_subjects(TD_DATA_PATH, 0, sites_to_load=SITES_TO_LOAD)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
sites = np.array(asd_data["sites"] + td_data["sites"])
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)
native_lengths = np.asarray(asd_data["native_lengths"] + td_data["native_lengths"], dtype=np.int32)

print("\n================ MULTI-SITE DATASET SUMMARY ================")
for site_name in sorted(set(sites)):
    site_mask = sites == site_name
    print(f"  {site_name}: {site_mask.sum()} subjects "
          f"({(y[site_mask] == 1).sum()} ASD, {(y[site_mask] == 0).sum()} TD)")
print("Total subjects:", len(X))

# Fail loudly on any unmapped site rather than silently assuming a TR.
unmapped = set(sites) - set(SITE_TR_MAP.keys())
assert not unmapped, f"No TR mapping for site(s) {unmapped} -- add them to SITE_TR_MAP before proceeding."


# =====================================================
# 3. EFFECTIVE PER-SUBJECT TR (site TR + subsampling correction)
# =====================================================
def compute_effective_tr(site_name, native_length, target_time):
    """
    Returns the TR that should be used for THIS subject's phase-
    coherence filtering. If native_length <= target_time, the subject
    was zero-padded (not resampled) -- the retained real timepoints are
    exactly at the site's nominal TR, so no correction is needed. If
    native_length > target_time, the subject was subsampled via
    np.linspace, which stretches the effective time between consecutive
    RETAINED samples to native_TR * (native_length - 1) / (target_time - 1);
    using the nominal TR in that case would misstate the physical
    frequency band for that subject specifically.
    """
    nominal_tr = SITE_TR_MAP[site_name]
    if native_length <= target_time:
        return nominal_tr
    return nominal_tr * (native_length - 1) / (target_time - 1)


effective_trs = np.array([
    compute_effective_tr(sites[i], native_lengths[i], TARGET_TIME) for i in range(len(X))
])
print("\nEffective TR range across loaded subjects:", effective_trs.min(), "-", effective_trs.max())
n_subsampled = (native_lengths > TARGET_TIME).sum()
print(f"Subjects requiring subsampling (native_length > TARGET_TIME={TARGET_TIME}): "
      f"{n_subsampled} / {len(X)}  -- these get an effective-TR correction; others use nominal site TR.")


# =====================================================
# 4. FEATURE CONSTRUCTION (static FC: site/TR-independent, computed once;
#    phase-coherence: uses PER-SUBJECT effective TR)
# =====================================================
PHASE_BAND = (0.04, 0.07)
PERCENTILE = 25.0   # held fixed at the paper's default; this experiment tests site generalization, not threshold


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    padlen = min(3 * (2 * order + 1), ts.shape[0] - 1)
    return filtfilt(b, a, ts, axis=0, padlen=padlen)


def compute_static_fc_matrix(ts_subject, L, R):
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    return (fc_mat + fc_mat.T) / 2.0


def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    ts = ts_subject[:L]
    fs = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    return ((cos_p.T @ cos_p + sin_p.T @ sin_p) / L).astype(np.float32)


def precompute_all_features(X_raw, real_lengths, effective_trs, band=PHASE_BAND, percentile=PERCENTILE):
    """
    Runs once for the whole multi-site dataset. Returns:
      fc_triu_all      : (N, n_pairs) plain static FC (site/TR-independent)
      masked_triu_all  : (N, n_pairs) phase-coherence-masked FC, where
                         EACH SUBJECT's phase-coherence used their own
                         effective TR
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fc_triu_all = np.zeros((N, n_pairs), dtype=np.float32)
    masked_triu_all = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        fc_triu = fc_mat[triu_idx]
        fc_triu_all[i] = fc_triu

        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, effective_trs[i], band)
        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = abs_phase_triu >= threshold
        masked_triu_all[i] = np.where(mask, fc_triu, 0.0).astype(np.float32)

    return fc_triu_all, masked_triu_all


print("\nPrecomputing static FC and phase-coherence-masked FC for all subjects "
      "(each subject uses its own effective TR)...")
fc_triu_all, masked_triu_all = precompute_all_features(X, real_lengths, effective_trs)
print("Done. Shapes:", fc_triu_all.shape, masked_triu_all.shape)


# =====================================================
# 5. SHARED SVM TRAIN/EVAL
# =====================================================
def train_and_eval_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_preds = model.predict(Xva)
    val_auc = roc_auc_score(y_val, val_scores) if len(np.unique(y_val)) > 1 else np.nan
    return val_auc, val_preds


def metrics_from_preds(y_val, val_preds):
    tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = f1_score(y_val, val_preds, zero_division=0)
    acc = accuracy_score(y_val, val_preds)
    return sensitivity, specificity, f1, acc


def evaluate_cv_single_site(feat_all, y, site_mask, skf, label=""):
    """Standard stratified CV within one site (fold object passed in, so
    it can be shared across both feature types for a fair comparison)."""
    X_site = feat_all[site_mask]
    y_site = y[site_mask]
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for tr_idx, va_idx in skf.split(X_site, y_site):
        auc_val, val_preds = train_and_eval_svm(X_site[tr_idx], y_site[tr_idx], X_site[va_idx], y_site[va_idx])
        sens, spec, f1, acc = metrics_from_preds(y_site[va_idx], val_preds)
        aucs.append(auc_val); senss.append(sens); specs.append(spec); f1s.append(f1); accs.append(acc)

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs, "f1": f1s, "accuracy": accs}
    print(f"[{label}] N={site_mask.sum()}  "
          f"AUC={np.nanmean(aucs):.3f}±{np.nanstd(aucs):.3f}  "
          f"Sens={np.mean(senss):.3f}±{np.std(senss):.3f}  "
          f"Spec={np.mean(specs):.3f}±{np.std(specs):.3f}  "
          f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}  "
          f"Acc={np.mean(accs):.3f}±{np.std(accs):.3f}")
    return result


# =====================================================
# 6. RUN BOTH FEATURE TYPES, PER SITE, ACROSS EVERY AVAILABLE SITE
# =====================================================
# Sites need enough subjects per class for stratified CV to be
# meaningful. MIN_PER_CLASS is the minimum ASD (or TD) count required
# for a site to be included at all; N_SPLITS_DEFAULT is capped down to
# the site's own minimum class count when that count is smaller (e.g.
# a site with only 6 ASD subjects gets a 5-fold split at most, since
# StratifiedKFold requires n_splits <= the smallest class count).
N_SPLITS_DEFAULT = 5
MIN_PER_CLASS = 5

metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
FEATURE_SETS = {"Full Static FC": fc_triu_all, "Phase-Coherence Masked FC": masked_triu_all}

site_results = {}   # site_name -> {feat_name -> result dict}
skipped_sites = []

for site_name in sorted(set(sites)):
    site_mask = sites == site_name
    n_asd = int((y[site_mask] == 1).sum())
    n_td = int((y[site_mask] == 0).sum())
    min_class = min(n_asd, n_td)

    if min_class < MIN_PER_CLASS:
        skipped_sites.append((site_name, n_asd, n_td))
        print(f"\nSkipping {site_name}: only {n_asd} ASD / {n_td} TD "
              f"(need >= {MIN_PER_CLASS} per class for stratified CV).")
        continue

    n_splits_site = min(N_SPLITS_DEFAULT, min_class)
    skf_site = StratifiedKFold(n_splits=n_splits_site, shuffle=True, random_state=42)

    print("\n" + "=" * 70)
    print(f"SITE: {site_name}  (N={site_mask.sum()}, {n_asd} ASD / {n_td} TD, "
          f"{n_splits_site}-fold CV)")
    print("=" * 70)

    site_results[site_name] = {}
    for feat_name, feat_all in FEATURE_SETS.items():
        # Same skf_site object (and therefore identical folds) reused
        # for both feature types, so the Full-vs-Masked comparison
        # within this site is fold-matched.
        site_results[site_name][feat_name] = evaluate_cv_single_site(
            feat_all, y, site_mask, skf_site, label=f"{site_name} | {feat_name}"
        )

if skipped_sites:
    print("\n" + "-" * 70)
    print(f"Skipped {len(skipped_sites)} site(s) with insufficient subjects per class:")
    for s, n_asd, n_td in skipped_sites:
        print(f"  {s}: {n_asd} ASD / {n_td} TD")


# =====================================================
# 7. ALL-SITES SUMMARY TABLE + PER-SITE PAIRED TESTS + POOLED TEST
# =====================================================
from scipy import stats

included_sites = list(site_results.keys())

print("\n" + "=" * 110)
print("ALL-SITES COMPARISON: Full Static FC vs. Phase-Coherence Masked FC (mean ± SD across each site's folds)")
print("=" * 110)
header = f"{'Site':<10}{'Feature':<28}" + "".join(f"{m.upper():<12}" for m in metrics)
print(header)
print("-" * len(header))
for site_name in included_sites:
    for feat_name in FEATURE_SETS:
        vals_row = site_results[site_name][feat_name]
        row = f"{site_name:<10}{feat_name:<28}"
        for m in metrics:
            vals = np.array(vals_row[m], dtype=np.float64)
            row += f"{np.nanmean(vals):<12.3f}"
        print(row)
    print("-" * len(header))

print("\n" + "=" * 110)
print("PER-SITE PAIRED TEST (Masked - Full, paired t-test across that site's own folds)")
print("=" * 110)
pt_header = f"{'Site':<10}" + "".join(f"{m.upper()+' delta(p)':<20}" for m in metrics)
print(pt_header)
print("-" * len(pt_header))
for site_name in included_sites:
    row = f"{site_name:<10}"
    for m in metrics:
        masked_vals = np.array(site_results[site_name]["Phase-Coherence Masked FC"][m], dtype=np.float64)
        full_vals = np.array(site_results[site_name]["Full Static FC"][m], dtype=np.float64)
        delta = np.nanmean(masked_vals) - np.nanmean(full_vals)
        valid = ~(np.isnan(masked_vals) | np.isnan(full_vals))
        if valid.sum() >= 2 and np.std(masked_vals[valid] - full_vals[valid]) > 0:
            _, p_val = stats.ttest_rel(masked_vals[valid], full_vals[valid])
            cell = f"{delta:+.3f} (p={p_val:.3f})"
        else:
            cell = f"{delta:+.3f} (p=n/a)"
        row += f"{cell:<20}"
    print(row)

# Pooled test: every (site, fold) pair contributes one paired
# observation, giving a single overall test across ALL sites combined
# -- a much larger n than any individual site's own fold count.
print("\n" + "-" * 70)
print("POOLED PAIRED TEST across all sites' folds combined (Masked - Full):")
print("-" * 70)
for m in metrics:
    pooled_masked, pooled_full = [], []
    for site_name in included_sites:
        pooled_masked.extend(site_results[site_name]["Phase-Coherence Masked FC"][m])
        pooled_full.extend(site_results[site_name]["Full Static FC"][m])
    pooled_masked = np.array(pooled_masked, dtype=np.float64)
    pooled_full = np.array(pooled_full, dtype=np.float64)
    valid = ~(np.isnan(pooled_masked) | np.isnan(pooled_full))
    delta = np.nanmean(pooled_masked) - np.nanmean(pooled_full)
    if valid.sum() >= 2 and np.std(pooled_masked[valid] - pooled_full[valid]) > 0:
        _, p_val = stats.ttest_rel(pooled_masked[valid], pooled_full[valid])
        print(f"  {m:<12}: delta={delta:+.3f}  p={p_val:.3f}  (n={valid.sum()} pooled fold-observations)")
    else:
        print(f"  {m:<12}: delta={delta:+.3f}  p=n/a")


# =====================================================
# 8. ALL-SITES COMPARISON FIGURE
# =====================================================
def plot_all_sites_comparison(site_results, out_name="all_sites_comparison"):
    included = list(site_results.keys())
    n_sites = len(included)
    fig, axes = plt.subplots(1, 2, figsize=(max(10, 1.1 * n_sites), 5.5))
    colors = {"Full Static FC": "#1f77b4", "Phase-Coherence Masked FC": "#ff7f0e"}

    for ax, metric in zip(axes, ["auc", "accuracy"]):
        x = np.arange(n_sites)
        width = 0.35
        for i, feat_name in enumerate(FEATURE_SETS):
            vals = [np.nanmean(site_results[s][feat_name][metric]) for s in included]
            offset = (i - 0.5) * width
            ax.bar(x + offset, vals, width, label=feat_name, color=colors[feat_name], alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(included, rotation=45, ha="right")
        ax.set_ylabel(metric.upper()); ax.set_ylim(0, 1.05)
        ax.set_title(f"{metric.upper()} by Site: Full Static FC vs. Masked FC")
        ax.legend(loc="lower right", fontsize=8)

    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{out_name}.pdf"), bbox_inches="tight")
    print(f"\nSaved figures/{out_name}.png and .pdf")
    plt.close(fig)


plot_all_sites_comparison(site_results)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Control Experiments: Phase-Coherence Mask vs. Random Mask
#                      vs. FC-Magnitude Mask vs. Full Static FC (SVM)
#
# RATIONALE (control experiments requested for review):
# The core hypothesis is "phase-coherence tells us which edges to
# keep." That hypothesis is only supported if a phase-coherence mask
# beats OTHER masks that retain the SAME NUMBER of edges but choose
# them by a different (or no) criterion. This script adds two such
# controls, evaluated on the identical folds/classifier as the main
# comparison, so that all four arms differ ONLY in which edges survive
# the mask (or whether there is a mask at all):
#
#   A. Full Static FC             -- no masking, every edge kept.
#   B. Phase-Coherence Masked FC  -- edges kept iff |phase-coherence|
#                                    is in the subject's own top
#                                    (100-PERCENTILE)%. (main hypothesis)
#   C. Random Masked FC           -- the SAME NUMBER of edges per
#                                    subject as (B), but WHICH edges
#                                    survive is chosen uniformly at
#                                    random (reproducible per subject).
#                                    If (B) doesn't beat (C), any gain
#                                    from masking is just sparsification,
#                                    not phase-coherence being informative.
#   D. FC-Magnitude Masked FC     -- the SAME NUMBER of edges per
#                                    subject as (B), but the edges kept
#                                    are simply each subject's own
#                                    strongest |Pearson-FC| edges. This
#                                    is a much stronger control than
#                                    (C): it asks whether phase-coherence
#                                    is a BETTER edge-selection criterion
#                                    than just trusting FC's own magnitude.
#
# If AUC/accuracy/etc. for (B) > (C) and (B) > (D), that is much
# stronger evidence that phase-coherence itself carries useful,
# non-redundant edge-selection information -- not just that dropping
# 25% of edges to reduce dimensionality happens to help the classifier.
# =====================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score, accuracy_score,
                              roc_curve, auc, precision_recall_curve, average_precision_score,
                              ConfusionMatrixDisplay)
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER (unchanged from the main pipeline)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}
SELECTED_NETWORK = "WHOLE_BRAIN"
SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {"X": [], "y": [], "subject_ids": [], "sites": [], "real_lengths": []}
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]

print("\n================ DATASET SUMMARY ================")
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared by every arm)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4. FEATURE-CONSTRUCTION FUNCTIONS
# =====================================================
PHASE_TR = 2.0
PHASE_BAND = (0.04, 0.07)
PERCENTILE = 25.0   # keep top (100 - PERCENTILE)% of edges, by whichever criterion


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_matrix(ts_subject, L, R):
    """Static Pearson FC matrix for one subject's full time series, symmetrized exactly."""
    ts = ts_subject[:L]
    fc_mat = np.corrcoef(ts, rowvar=False).astype(np.float32)
    fc_mat = np.nan_to_num(fc_mat, nan=0.0)
    fc_mat = (fc_mat + fc_mat.T) / 2.0
    return fc_mat


def compute_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    """Static phase-coherence matrix (cos of instantaneous phase difference, time-averaged)."""
    ts = ts_subject[:L]
    fs = 1.0 / TR
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    # Vectorized via cos(a-b) = cos a cos b + sin a sin b -- avoids an
    # (L, R, R) intermediate tensor (see performance note from review).
    cos_p, sin_p = np.cos(phase), np.sin(phase)
    phase_mat = (cos_p.T @ cos_p + sin_p.T @ sin_p) / L
    return phase_mat.astype(np.float32)


# ---- Arm A: Full Static FC (no masking) ----
def compute_static_fc_features(X_raw, real_lengths, subject_indices=None):
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    feats = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        fc_mat = compute_static_fc_matrix(X_raw[i], int(real_lengths[i]), R)
        feats[i] = fc_mat[triu_idx]
    return feats


# ---- Arm B: Phase-Coherence Masked FC (main hypothesis) ----
def compute_percentile_masked_fc_features(X_raw, real_lengths, subject_indices=None,
                                           TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE):
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    feats = np.zeros((N, len(triu_idx[0])), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        phase_mat = compute_phase_coherence_matrix(X_raw[i], L, R, TR, band)

        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold
        combined = np.where(mask, fc_mat, 0.0).astype(np.float32)
        feats[i] = combined[triu_idx]
    return feats


# ---- Arm C: Random Masked FC (sparsification-only control) ----
def compute_random_masked_fc_features(X_raw, real_lengths, subject_indices,
                                       percentile=PERCENTILE, base_seed=12345):
    """
    Retains the SAME NUMBER of edges per subject as the phase-coherence
    mask, but WHICH edges survive is chosen uniformly at random rather
    than by any connectivity criterion.

    Reproducibility across folds: each subject's random mask is seeded
    from that subject's GLOBAL index (subject_indices[i], i.e. its row
    position in the full X/y arrays), not from call order or local
    position within the current fold. This guarantees a given subject
    gets the IDENTICAL random mask whether it happens to be evaluated
    as part of a training fold or a validation fold -- the mask is a
    fixed property of the subject, not an artifact of how folds were
    drawn. Without this, the same subject could silently receive a
    different random mask depending on which fold it landed in across
    the two evaluate_classifier() calls (train-fold call vs.
    validation-fold call), which would make the control itself
    irreproducible and arguably invalid.
    """
    if subject_indices is None:
        raise ValueError("compute_random_masked_fc_features requires subject_indices "
                          "for reproducible per-subject random masks.")
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    n_keep = int(round(n_pairs * (100 - percentile) / 100.0))
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        fc_triu = fc_mat[triu_idx]

        subj_seed = base_seed + int(subject_indices[i])
        rng = np.random.default_rng(subj_seed)
        keep_positions = rng.choice(n_pairs, size=n_keep, replace=False)

        masked = np.zeros(n_pairs, dtype=np.float32)
        masked[keep_positions] = fc_triu[keep_positions]
        feats[i] = masked

    return feats


# ---- Arm D: FC-Magnitude Masked FC (strong control) ----
def compute_fc_magnitude_masked_fc_features(X_raw, real_lengths, subject_indices=None,
                                             percentile=PERCENTILE):
    """
    Retains the SAME NUMBER of edges per subject as the phase-coherence
    mask, but the edges kept are simply each subject's own strongest
    |Pearson-FC| edges (top (100-percentile)% by FC magnitude itself).

    This is a substantially stronger control than random masking: it
    asks specifically whether phase-coherence is a BETTER edge-
    selection criterion than the trivial approach of just trusting
    which edges already have the largest correlation magnitude. If
    phase-coherence masking does not outperform this arm, phase-
    coherence is not adding anything beyond what the FC values already
    reveal about themselves.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    feats = np.zeros((N, len(triu_idx[0])), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        fc_triu = fc_mat[triu_idx]

        abs_fc = np.abs(fc_triu)
        threshold = np.percentile(abs_fc, percentile)
        mask = abs_fc >= threshold

        masked = np.where(mask, fc_triu, 0.0).astype(np.float32)
        feats[i] = masked

    return feats


# =====================================================
# 5. SHARED SVM TRAIN/EVAL (identical for all four arms)
# =====================================================
def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)
    return model, scaler, val_auc, val_preds


def evaluate_classifier(X, y, real_lengths, skf, feature_fn, feature_kwargs=None,
                         C=1.0, kernel="linear", label="classifier"):
    """
    Generic fold-runner shared by all four arms. `subject_indices` is
    always passed to feature_fn as (tr_idx or va_idx) -- the GLOBAL row
    indices into X/y/real_lengths for whichever subset is being
    featurized -- so any feature function that needs a reproducible
    per-subject random state (e.g. the random-mask control) has access
    to a stable subject identity regardless of fold membership.
    """
    feature_kwargs = feature_kwargs or {}
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    fold_y_true, fold_y_score, fold_y_pred = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        X_train_feat = feature_fn(X_train_raw, rl_train, subject_indices=tr_idx, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, subject_indices=va_idx, **feature_kwargs)

        model, scaler, auc_val, val_preds = train_one_fold_svm(
            X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel
        )
        val_scores = model.decision_function(scaler.transform(X_val_feat))

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)
        fold_y_true.append(y_val); fold_y_score.append(val_scores); fold_y_pred.append(val_preds)

        print(f"[{label}] Fold {fold+1}: AUC={auc_val:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {label}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs,
            "fold_y_true": fold_y_true, "fold_y_score": fold_y_score, "fold_y_pred": fold_y_pred}


# =====================================================
# 6. RUN ALL FOUR ARMS ON THE SAME FOLDS
# =====================================================
ARMS = {
    "Full Static FC": (compute_static_fc_features, {}),
    "Phase-Coherence Masked FC": (compute_percentile_masked_fc_features,
                                  dict(TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE)),
    "Random Masked FC": (compute_random_masked_fc_features, dict(percentile=PERCENTILE)),
    "FC-Magnitude Masked FC": (compute_fc_magnitude_masked_fc_features, dict(percentile=PERCENTILE)),
}

results_by_arm = {}
for label, (feature_fn, feature_kwargs) in ARMS.items():
    print("\n" + "=" * 70)
    print(f"ARM: {label}")
    print("=" * 70)
    results_by_arm[label] = evaluate_classifier(
        X, y, real_lengths, skf, feature_fn=feature_fn, feature_kwargs=feature_kwargs, label=label
    )


# =====================================================
# 7. COMPARISON TABLE (all arms) + KEY PAIRWISE TESTS
# =====================================================
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
arm_labels = list(ARMS.keys())

print("\n" + "=" * 100)
print("ALL-ARMS COMPARISON (mean ± SD across folds)")
print("=" * 100)
header = f"{'Metric':<14}" + "".join(f"{lbl[:22]:>24}" for lbl in arm_labels)
print(header)
print("-" * len(header))
for m in metrics:
    row = f"{m:<14}"
    for lbl in arm_labels:
        vals = np.array(results_by_arm[lbl][m], dtype=np.float64)
        row += f"{np.nanmean(vals):>9.3f}±{np.nanstd(vals):<5.3f}      "
    print(row)

# Key pairwise paired t-tests, since these are the comparisons that
# actually test the hypothesis: does Phase-Coherence Masked FC beat
# each control, given they share identical folds?
print("\n" + "=" * 100)
print("KEY PAIRWISE TESTS: Phase-Coherence Masked FC vs. each control (paired t-test across folds)")
print("=" * 100)
pair_header = f"{'Metric':<14}{'vs. Full FC (delta, p)':>28}{'vs. Random Mask (delta, p)':>30}{'vs. FC-Mag Mask (delta, p)':>28}"
print(pair_header)
print("-" * len(pair_header))

phase_label = "Phase-Coherence Masked FC"
for m in metrics:
    phase_vals = np.array(results_by_arm[phase_label][m], dtype=np.float64)
    row = f"{m:<14}"
    for other_label in ["Full Static FC", "Random Masked FC", "FC-Magnitude Masked FC"]:
        other_vals = np.array(results_by_arm[other_label][m], dtype=np.float64)
        delta = np.nanmean(phase_vals) - np.nanmean(other_vals)
        valid = ~(np.isnan(phase_vals) | np.isnan(other_vals))
        if valid.sum() >= 2 and np.std(phase_vals[valid] - other_vals[valid]) > 0:
            _, p_val = stats.ttest_rel(phase_vals[valid], other_vals[valid])
            cell = f"{delta:+.3f}, p={p_val:.3f}"
        else:
            cell = f"{delta:+.3f}, p=n/a"
        row += f"{cell:>28}" if other_label == "Full Static FC" else (
               f"{cell:>30}" if other_label == "Random Masked FC" else f"{cell:>28}")
    print(row)

print("\nInterpretation guide:")
print("  - 'vs. Random Mask' tests whether phase-coherence beats pure sparsification at the same edge count.")
print("  - 'vs. FC-Mag Mask' tests whether phase-coherence beats simply keeping each subject's own")
print("    strongest FC edges -- the stronger and more informative of the two controls.")
print("  - Positive delta + low p => evidence phase-coherence is a genuinely informative edge-selection")
print("    criterion, not just an incidental effect of retaining fewer, arbitrary, or magnitude-biased edges.")


# =====================================================
# 8. FIGURES (generalized for N arms)
# =====================================================
PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
ARM_COLORS = {lbl: PALETTE[i % len(PALETTE)] for i, lbl in enumerate(arm_labels)}


def _savefig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{name}.png and .pdf")


def plot_confusion_matrices_grid(results_dict, out_name="confusion_matrices_4arm"):
    n = len(results_dict)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.6 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (label, results) in zip(axes, results_dict.items()):
        y_true_pooled = np.concatenate(results["fold_y_true"])
        y_pred_pooled = np.concatenate(results["fold_y_pred"])
        cm = confusion_matrix(y_true_pooled, y_pred_pooled, labels=[0, 1])
        disp = ConfusionMatrixDisplay(cm, display_labels=["TD", "ASD"])
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(label, fontsize=11)
        ax.grid(False)
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle("Pooled Out-of-Fold Confusion Matrices: All Four Arms", y=1.02, fontsize=13)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


def plot_roc_curves_multi(results_dict, out_name="roc_curves_4arm"):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    mean_fpr = np.linspace(0, 1, 200)
    for label, results in results_dict.items():
        color = ARM_COLORS[label]
        tprs, aucs_fold = [], []
        for y_true, y_score in zip(results["fold_y_true"], results["fold_y_score"]):
            fpr, tpr, _ = roc_curve(y_true, y_score)
            aucs_fold.append(auc(fpr, tpr))
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
        mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
        mean_auc, std_auc = np.mean(aucs_fold), np.std(aucs_fold)
        ax.plot(mean_fpr, mean_tpr, lw=2.2, color=color,
                label=f"{label} (AUC = {mean_auc:.3f} ± {std_auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray", label="Chance")
    ax.set_xlabel("False Positive Rate (1 - Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.set_title("ROC Curves: All Four Arms")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


def plot_metric_comparison_multi(results_dict, out_name="metric_comparison_4arm"):
    metric_names = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
    display_names = ["AUC", "Sensitivity", "Specificity", "F1", "Accuracy"]
    labels = list(results_dict.keys())
    n_arms = len(labels)
    x = np.arange(len(metric_names))
    width = 0.8 / n_arms
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, label in enumerate(labels):
        means = [np.nanmean(results_dict[label][m]) for m in metric_names]
        stds = [np.nanstd(results_dict[label][m]) for m in metric_names]
        offset = (i - (n_arms - 1) / 2) * width
        ax.bar(x + offset, means, width, yerr=stds, capsize=3,
               label=label, color=ARM_COLORS[label], alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(display_names)
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.2)
    ax.set_title("Cross-Validated Performance: All Four Arms")
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


def plot_fold_auc_distribution_multi(results_dict, out_name="fold_auc_distribution_4arm"):
    labels = list(results_dict.keys())
    data = [results_dict[label]["auc"] for label in labels]
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5, showmeans=True)
    for patch, label in zip(bp["boxes"], labels):
        patch.set_facecolor(ARM_COLORS[label]); patch.set_alpha(0.4)
    rng = np.random.default_rng(0)
    for i, (label, vals) in enumerate(zip(labels, data), start=1):
        jitter = rng.uniform(-0.06, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color=ARM_COLORS[label],
                   edgecolor="black", linewidth=0.5, zorder=3, s=35)
    ax.set_ylabel("Fold-wise AUC")
    ax.set_title("Distribution of Per-Fold AUC: All Four Arms")
    ax.set_ylim(0, 1.05)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


plot_confusion_matrices_grid(results_by_arm)
plot_roc_curves_multi(results_by_arm)
plot_metric_comparison_multi(results_by_arm)
plot_fold_auc_distribution_multi(results_by_arm)

print(f"\nAll figures written to ./{FIGURES_DIR}/ (PNG @600dpi + vector PDF).")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Percentile-Masked Static FC vs. Plain Static FC (SVM)
#
# This extends the percentile-masked pipeline with a BASELINE:
# plain static Pearson FC (no phase-coherence masking at all).
# Both classifiers are evaluated on the SAME StratifiedKFold splits,
# using the SAME model class (SVC) and the SAME train_one_fold_svm(),
# so the only thing that differs between the two rows in the final
# comparison table is the feature-construction step. This isolates
# the effect of percentile phase-masking from confounds like
# different folds, different classifiers, or different tuning.
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score, accuracy_score,
                              roc_curve, auc, precision_recall_curve, average_precision_score,
                              ConfusionMatrixDisplay)
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Journal-friendly plotting defaults (Times-like serif, high DPI, vector-safe)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})


# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = "pitt"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared by both classifiers)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4a. PERCENTILE-MASKED STATIC FC FEATURE (unchanged from before)
# =====================================================
PHASE_TR = 1.5             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band
PERCENTILE = 25.0          # keep the strongest (100 - PERCENTILE)% of phase-coherence edges


def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_matrix(ts_subject, L, R):
    """
    Shared helper: builds the plain static Pearson FC matrix for one
    subject over its real (non-padded) timepoints only. Used by BOTH
    the masked pipeline and the plain-FC baseline, so the underlying
    FC values being compared are guaranteed identical -- the only
    difference is whether masking is applied afterward.
    """
    T = ts_subject.shape[0]
    w = np.zeros(T, dtype=np.float32)
    w[:L] = 1.0
    fc_mat = weighted_corrcoef(ts_subject, w)
    fc_mat = (fc_mat + fc_mat.T) / 2.0  # force exact symmetry
    return fc_mat


def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4):
    """
    Per subject: static Pearson FC + static phase-coherence, combined
    via percentile masking (keep FC value only where |phase-coherence|
    clears the subject's own top-`percentile` threshold; else zero).
    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)

        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))
            diff = phase[:, :, None] - phase[:, None, :]
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)

        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold
        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)

        feats[i] = combined_mat[triu_idx]

    return feats


# =====================================================
# 4b. PLAIN STATIC FC FEATURE (baseline / comparison arm)
# =====================================================
# No phase-coherence, no masking: every edge's static Pearson FC value
# is kept as-is. This isolates whether percentile phase-masking adds
# (or costs) anything relative to the unmasked correlation matrix.
# =====================================================

def compute_static_fc_features(X_raw, real_lengths):
    """
    Per subject: plain static Pearson FC matrix (upper triangle,
    flattened), no masking of any kind.
    Returns: (N, n_pairs) float32 -- same shape/layout as the masked
    feature, so both can be fed through the identical SVM pipeline.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        feats[i] = fc_mat[triu_idx]

    return feats


# =====================================================
# 5. SHARED SVM TRAIN/EVAL (identical for both feature types)
# =====================================================

def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    """
    Fits an SVM on whichever feature matrix is passed in. Features are
    standardized (fit on training subjects only, applied to val --
    fold-safe) before fitting. class_weight='balanced' handles the
    ASD/TD class imbalance.

    kernel="linear" is the default: with n_pairs often >> N_subjects,
    linear SVMs tend to generalize better and keep the model
    interpretable.

    AUC is computed from decision_function rather than predict_proba
    (avoids the extra internal CV pass Platt scaling would need).

    Returns: (model, scaler, val_auc, val_preds)
    """
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)

    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)

    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)

    return model, scaler, val_auc, val_preds


def evaluate_classifier(X, y, real_lengths, skf, feature_fn, feature_kwargs=None,
                         C=1.0, kernel="linear", label="classifier"):
    """
    Generic fold-runner: applies `feature_fn` (either the masked or the
    plain static FC feature function) per fold, then trains/evaluates
    an SVM. Because `skf` is instantiated once and reused for both
    calls, train/val subject splits are IDENTICAL across the two
    arms of the comparison -- no fold-assignment confound.
    """
    feature_kwargs = feature_kwargs or {}
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    # Per-fold raw outputs, kept for downstream figures (ROC/PR curves,
    # pooled confusion matrix). Stored as one array/list entry per fold
    # rather than concatenated here, since ROC-per-fold plotting needs
    # fold boundaries preserved; pooling (for the confusion matrix) is
    # done later with np.concatenate.
    fold_y_true, fold_y_score, fold_y_pred = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- [{label}] Fold {fold+1}: computing features ---")

        X_train_feat = feature_fn(X_train_raw, rl_train, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, **feature_kwargs)

        model, scaler, auc_val, val_preds = train_one_fold_svm(
            X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel
        )
        # decision_function scores, recomputed here only for storage
        # (train_one_fold_svm already used them internally for AUC);
        # cheap since it's just a forward pass through the fitted SVM.
        val_scores = model.decision_function(scaler.transform(X_val_feat))

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)
        fold_y_true.append(y_val); fold_y_score.append(val_scores); fold_y_pred.append(val_preds)

        print(f"[{label}] Fold {fold+1}: AUC={auc_val:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {label}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs,
            "fold_y_true": fold_y_true, "fold_y_score": fold_y_score, "fold_y_pred": fold_y_pred}


# =====================================================
# 6. RUN BOTH ARMS ON THE SAME FOLDS, THEN COMPARE
# =====================================================
print("\n" + "=" * 60)
print(f"ARM 1/2: Percentile-Masked Static FC (SVM, top {100-PERCENTILE:.0f}% phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")

masked_fc_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_percentile_masked_fc_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE),
    label="Percentile-Masked Static FC"
)

print("\n" + "=" * 60)
print("ARM 2/2: Plain Static FC baseline (SVM, no phase masking)")
print("=" * 60)

static_fc_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_static_fc_features,
    feature_kwargs={},
    label="Plain Static FC (baseline)"
)


# =====================================================
# 7. HEAD-TO-HEAD COMPARISON TABLE + PAIRED SIGNIFICANCE TEST
# =====================================================
print("\n" + "=" * 70)
print("HEAD-TO-HEAD: Percentile-Masked FC  vs.  Plain Static FC (baseline)")
print("=" * 70)

metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
header = f"{'Metric':<14}{'Masked FC':>16}{'Static FC':>16}{'Delta (M-S)':>16}{'Paired p':>12}"
print(header)
print("-" * len(header))

for m in metrics:
    masked_vals = np.array(masked_fc_results[m], dtype=np.float64)
    static_vals = np.array(static_fc_results[m], dtype=np.float64)

    masked_mean, masked_std = np.nanmean(masked_vals), np.nanstd(masked_vals)
    static_mean, static_std = np.nanmean(static_vals), np.nanstd(static_vals)
    delta = masked_mean - static_mean

    # Paired t-test across folds (same folds for both arms -> valid pairing).
    # Only meaningful with >=2 folds and no NaNs; guarded accordingly.
    valid = ~(np.isnan(masked_vals) | np.isnan(static_vals))
    if valid.sum() >= 2 and np.std(masked_vals[valid] - static_vals[valid]) > 0:
        _, p_val = stats.ttest_rel(masked_vals[valid], static_vals[valid])
        p_str = f"{p_val:.3f}"
    else:
        p_str = "n/a"

    print(f"{m:<14}{masked_mean:>9.3f}±{masked_std:<5.3f}"
          f"{static_mean:>9.3f}±{static_std:<5.3f}"
          f"{delta:>+15.3f}  {p_str:>10}")

print("-" * len(header))
print("Positive Delta => percentile-masked FC outperforms plain static FC on that metric.")
print("Paired p-value is a per-fold paired t-test (n=folds); with only "
      f"{N_SPLITS} folds this is a rough guide, not a rigorous significance claim -- "
      "treat p-values near the 0.05 boundary cautiously given the small n.")


# =====================================================
# 8. PUBLICATION FIGURES
# =====================================================
# All figures are saved to ./figures at 600 DPI (PNG) and also as
# vector PDF, so they can be dropped into a journal manuscript at any
# print size without pixelation. Both arms use the SAME color coding
# throughout: masked FC = C0 (blue), static FC baseline = C1 (orange).
# =====================================================

COLOR_MASKED = "#1f77b4"
COLOR_STATIC = "#ff7f0e"


def _savefig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{name}.png and .pdf")


# ---------- 8a. Confusion matrices (pooled across folds) ----------
# Pooling all folds' held-out predictions gives one confusion matrix
# per arm built from every subject's out-of-fold prediction exactly
# once (since StratifiedKFold partitions the full dataset) -- this is
# the standard way to report a single CV confusion matrix.

def plot_confusion_matrices(results_dict, out_name="confusion_matrices"):
    n_arms = len(results_dict)
    fig, axes = plt.subplots(1, n_arms, figsize=(5.5 * n_arms, 4.6))
    if n_arms == 1:
        axes = [axes]

    for ax, (label, results) in zip(axes, results_dict.items()):
        y_true_pooled = np.concatenate(results["fold_y_true"])
        y_pred_pooled = np.concatenate(results["fold_y_pred"])
        cm = confusion_matrix(y_true_pooled, y_pred_pooled, labels=[0, 1])

        disp = ConfusionMatrixDisplay(cm, display_labels=["TD", "ASD"])
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(label, fontsize=12)
        ax.grid(False)

    fig.suptitle("Pooled Out-of-Fold Confusion Matrices", y=1.03, fontsize=13)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


# ---------- 8b. ROC curves: per-fold + mean ± std band ----------
# Standard "sklearn-style" multi-fold ROC plot: each fold's ROC is
# interpolated onto a common FPR grid so a mean TPR curve (with a
# shaded +/-1 std band) can be drawn alongside the individual fold
# curves. Both arms are overlaid on one axes for direct comparison.

def plot_roc_curves(results_dict, out_name="roc_curves"):
    fig, ax = plt.subplots(figsize=(6, 6))
    mean_fpr = np.linspace(0, 1, 200)
    colors = {list(results_dict.keys())[0]: COLOR_MASKED,
              list(results_dict.keys())[1]: COLOR_STATIC} if len(results_dict) == 2 else {}

    for label, results in results_dict.items():
        color = colors.get(label, None)
        tprs, aucs_fold = [], []

        for i, (y_true, y_score) in enumerate(zip(results["fold_y_true"], results["fold_y_score"])):
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            aucs_fold.append(roc_auc)

            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)

            ax.plot(fpr, tpr, alpha=0.15, lw=1, color=color)

        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        std_tpr = np.std(tprs, axis=0)
        mean_auc = np.mean(aucs_fold)
        std_auc = np.std(aucs_fold)

        ax.plot(mean_fpr, mean_tpr, lw=2.5, color=color,
                label=f"{label} (AUC = {mean_auc:.3f} ± {std_auc:.3f})")
        ax.fill_between(mean_fpr, np.clip(mean_tpr - std_tpr, 0, 1),
                         np.clip(mean_tpr + std_tpr, 0, 1), color=color, alpha=0.15)

    ax.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray", label="Chance")
    ax.set_xlabel("False Positive Rate (1 - Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.set_title("ROC Curves: Percentile-Masked FC vs. Plain Static FC")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


# ---------- 8c. Precision-Recall curves (useful given class imbalance) ----------

def plot_pr_curves(results_dict, out_name="pr_curves"):
    fig, ax = plt.subplots(figsize=(6, 6))
    colors = {list(results_dict.keys())[0]: COLOR_MASKED,
              list(results_dict.keys())[1]: COLOR_STATIC} if len(results_dict) == 2 else {}

    for label, results in results_dict.items():
        color = colors.get(label, None)
        y_true_pooled = np.concatenate(results["fold_y_true"])
        y_score_pooled = np.concatenate(results["fold_y_score"])

        precision, recall, _ = precision_recall_curve(y_true_pooled, y_score_pooled)
        ap = average_precision_score(y_true_pooled, y_score_pooled)

        ax.plot(recall, precision, lw=2.5, color=color, label=f"{label} (AP = {ap:.3f})")

    prevalence = np.mean(np.concatenate(list(results_dict.values())[0]["fold_y_true"]))
    ax.axhline(prevalence, linestyle="--", lw=1, color="gray",
               label=f"Chance (prevalence = {prevalence:.2f})")
    ax.set_xlabel("Recall (Sensitivity)")
    ax.set_ylabel("Precision (PPV)")
    ax.set_title("Precision-Recall Curves (pooled out-of-fold)")
    ax.legend(loc="lower left", fontsize=9)
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


# ---------- 8d. Metric comparison bar chart (mean ± std across folds) ----------

def plot_metric_comparison(results_dict, out_name="metric_comparison"):
    metric_names = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
    display_names = ["AUC", "Sensitivity", "Specificity", "F1", "Accuracy"]
    labels = list(results_dict.keys())

    x = np.arange(len(metric_names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))

    for i, label in enumerate(labels):
        means = [np.nanmean(results_dict[label][m]) for m in metric_names]
        stds = [np.nanstd(results_dict[label][m]) for m in metric_names]
        color = COLOR_MASKED if i == 0 else COLOR_STATIC
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, means, width, yerr=stds, capsize=4,
                       label=label, color=color, alpha=0.85)
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(display_names)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.15)
    ax.set_title("Cross-Validated Performance: Masked FC vs. Static FC Baseline")
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


# ---------- 8e. Per-fold AUC distribution (boxplot / stripplot) ----------

def plot_fold_auc_distribution(results_dict, out_name="fold_auc_distribution"):
    labels = list(results_dict.keys())
    data = [results_dict[label]["auc"] for label in labels]
    colors = [COLOR_MASKED, COLOR_STATIC]

    fig, ax = plt.subplots(figsize=(5.5, 5))
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5, showmeans=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.4)

    rng = np.random.default_rng(0)
    for i, (label, vals) in enumerate(zip(labels, data), start=1):
        jitter = rng.uniform(-0.06, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color=colors[i - 1],
                   edgecolor="black", linewidth=0.5, zorder=3, s=40)

    ax.set_ylabel("Fold-wise AUC")
    ax.set_title("Distribution of Per-Fold AUC Across CV Folds")
    ax.set_ylim(0, 1.05)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)


# ---------- Run all figure-generation ----------
results_by_arm = {
    "Percentile-Masked FC": masked_fc_results,
    "Plain Static FC": static_fc_results,
}

plot_confusion_matrices(results_by_arm)
plot_roc_curves(results_by_arm)
plot_pr_curves(results_by_arm)
plot_metric_comparison(results_by_arm)
plot_fold_auc_distribution(results_by_arm)

print(f"\nAll figures written to ./{FIGURES_DIR}/ (PNG @600dpi + vector PDF).")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> FIVE-ARM ABLATION STUDY (SVM)
#
#   Arm 1: Static FC baseline (no phase information at all)
#   Arm 2: Percentile-Masked FC -- SIGNED coherence (keeps most
#          in-phase edges; anti-phase edges are masked out)
#   Arm 3: Percentile-Masked FC -- |coherence| magnitude (keeps
#          strongest-magnitude edges regardless of sign; the
#          original masking definition)
#   Arm 4: Phase-Coherence-ONLY -- SIGNED (no FC/amplitude info)
#   Arm 5: Phase-Coherence-ONLY -- |coherence| magnitude (no
#          FC/amplitude info)
#
# All five arms are evaluated on the SAME StratifiedKFold splits,
# using the SAME model class (SVC) and the SAME train_one_fold_svm(),
# so the only thing that differs between rows in the final comparison
# table is the feature-construction step. This gives a full 2x2
# ablation (masked-FC vs. coherence-only) x (signed vs. magnitude)
# anchored against the plain-FC baseline -- report all five, or
# whichever subset is most relevant, in the paper's ablation table.
# =====================================================
# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score, accuracy_score,
                              roc_curve, auc, precision_recall_curve, average_precision_score,
                              ConfusionMatrixDisplay)
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Journal-friendly plotting defaults (Times-like serif, high DPI, vector-safe)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}
NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"
if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")
if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)

def parse_site_from_filename(name):
    return name.split("_")[0]

def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results
    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue
            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]
            real_length = ts.shape[0]
            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]
            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")
    return results

asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())

# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared by all classifiers)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# =====================================================
# 4a. SHARED HELPERS: static FC + static phase coherence
# =====================================================
PHASE_TR = 2             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band
PERCENTILE = 25.0          # keep the top (100 - PERCENTILE)% strongest-coherence edges

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)
    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum
    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)

def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)

def compute_static_fc_matrix(ts_subject, L, R):
    """
    Shared helper: builds the plain static Pearson FC matrix for one
    subject over its real (non-padded) timepoints only. Used across
    every arm that involves FC, so the underlying FC values are
    guaranteed identical -- only the feature-construction step differs.
    """
    T = ts_subject.shape[0]
    w = np.zeros(T, dtype=np.float32)
    w[:L] = 1.0
    fc_mat = weighted_corrcoef(ts_subject, w)
    fc_mat = (fc_mat + fc_mat.T) / 2.0  # force exact symmetry
    return fc_mat

def compute_static_phase_coherence_matrix(ts_subject, L, R, TR, band, order=4):
    """
    Shared helper: builds the SIGNED static phase-coherence matrix for
    one subject over its real (non-padded) timepoints only.
    phase_mat[i,j] = mean(cos(phase_i(t) - phase_j(t))) over t, in
    [-1, +1]. +1 = perfectly in-phase, -1 = perfectly anti-phase.
    No np.abs() is applied here -- sign is preserved. Every arm that
    needs |coherence| (magnitude) applies np.abs() explicitly at the
    point of use, so this single function is the one source of truth
    for the underlying coherence values across all arms.
    """
    if L < 10:
        return np.zeros((R, R), dtype=np.float32)
    fs = 1.0 / TR
    ts = ts_subject[:L, :]
    filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
    phase = np.angle(hilbert(filtered, axis=0))
    diff = phase[:, :, None] - phase[:, None, :]
    coh = np.cos(diff)
    phase_mat = coh.mean(axis=0).astype(np.float32)
    phase_mat = (phase_mat + phase_mat.T) / 2.0  # force exact symmetry
    return phase_mat

# =====================================================
# 4b. STATIC FC FEATURE (baseline arm -- no phase info at all)
# =====================================================
def compute_static_fc_features(X_raw, real_lengths):
    """
    Per subject: plain static Pearson FC matrix (upper triangle,
    flattened), no masking of any kind. Anchor arm for the ablation.
    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        feats[i] = fc_mat[triu_idx]
    return feats

# =====================================================
# 4c. PERCENTILE-MASKED FC FEATURES -- SIGNED vs. MAGNITUDE
# =====================================================
def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4, use_abs=False):
    """
    Per subject: static Pearson FC combined with static phase-coherence
    via percentile masking (keep FC value only where coherence clears
    the subject's own top-`percentile` threshold; else zero).

    use_abs=False (default): threshold on SIGNED coherence -> keeps
        the most strongly IN-PHASE edges. Strongly anti-phase edges
        (coherence near -1) are masked OUT even if their magnitude of
        coupling is large.
    use_abs=True: threshold on |coherence| (magnitude) -> keeps the
        strongest-magnitude edges regardless of sign, so both strongly
        in-phase (+1) and strongly anti-phase (-1) edges are kept.
        This is the original masking definition.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        phase_mat = compute_static_phase_coherence_matrix(X_raw[i], L, R, TR, band, order=order)

        thresholding_mat = np.abs(phase_mat) if use_abs else phase_mat
        thresholding_triu = thresholding_mat[triu_idx]
        threshold = np.percentile(thresholding_triu, percentile)
        mask = thresholding_mat >= threshold

        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)
        feats[i] = combined_mat[triu_idx]
    return feats

# =====================================================
# 4d. PHASE-COHERENCE-ONLY FEATURES -- SIGNED vs. MAGNITUDE
# =====================================================
def compute_phase_coherence_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                      order=4, use_abs=False):
    """
    Per subject: static phase-coherence matrix (upper triangle,
    flattened), no FC/amplitude information involved anywhere.

    use_abs=False (default): keeps the SIGNED coherence value per edge
        (range [-1, +1]).
    use_abs=True: keeps |coherence| per edge (range [0, 1]), discarding
        the in-phase/anti-phase distinction.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)
    for i in range(N):
        L = int(real_lengths[i])
        phase_mat = compute_static_phase_coherence_matrix(X_raw[i], L, R, TR, band, order=order)
        if use_abs:
            phase_mat = np.abs(phase_mat)
        feats[i] = phase_mat[triu_idx]
    return feats

# =====================================================
# 5. SHARED SVM TRAIN/EVAL (identical for all feature types)
# =====================================================
def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    """
    Fits an SVM on whichever feature matrix is passed in. Features are
    standardized (fit on training subjects only, applied to val --
    fold-safe) before fitting. class_weight='balanced' handles the
    ASD/TD class imbalance.
    kernel="linear" is the default: with n_pairs often >> N_subjects,
    linear SVMs tend to generalize better and keep the model
    interpretable.
    AUC is computed from decision_function rather than predict_proba
    (avoids the extra internal CV pass Platt scaling would need).
    Returns: (model, scaler, val_auc, val_preds)
    """
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)
    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)
    return model, scaler, val_auc, val_preds

def evaluate_classifier(X, y, real_lengths, skf, feature_fn, feature_kwargs=None,
                         C=1.0, kernel="linear", label="classifier"):
    """
    Generic fold-runner: applies `feature_fn` per fold, then trains/
    evaluates an SVM. Because `skf` is instantiated once and reused
    across all calls, train/val subject splits are IDENTICAL across
    every arm of the comparison -- no fold-assignment confound.
    """
    feature_kwargs = feature_kwargs or {}
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    fold_y_true, fold_y_score, fold_y_pred = [], [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- [{label}] Fold {fold+1}: computing features ---")
        X_train_feat = feature_fn(X_train_raw, rl_train, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, **feature_kwargs)

        model, scaler, auc_val, val_preds = train_one_fold_svm(
            X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel
        )
        val_scores = model.decision_function(scaler.transform(X_val_feat))

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc_val); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)
        fold_y_true.append(y_val); fold_y_score.append(val_scores); fold_y_pred.append(val_preds)

        print(f"[{label}] Fold {fold+1}: AUC={auc_val:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {label}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs,
            "fold_y_true": fold_y_true, "fold_y_score": fold_y_score, "fold_y_pred": fold_y_pred}

# =====================================================
# 6. RUN ALL FIVE ARMS ON THE SAME FOLDS, THEN COMPARE
# =====================================================
print("\n" + "=" * 60)
print("ARM 1/5: Static FC baseline (SVM, no phase information)")
print("=" * 60)
static_fc_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_static_fc_features,
    feature_kwargs={},
    label="Static FC (baseline)"
)

print("\n" + "=" * 60)
print(f"ARM 2/5: Percentile-Masked FC -- SIGNED coherence (top {100-PERCENTILE:.0f}% most in-phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
masked_fc_signed_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_percentile_masked_fc_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE, use_abs=False),
    label="Masked FC (signed coherence)"
)

print("\n" + "=" * 60)
print(f"ARM 3/5: Percentile-Masked FC -- |coherence| magnitude (top {100-PERCENTILE:.0f}% strongest edges)")
print("=" * 60)
masked_fc_abs_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_percentile_masked_fc_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE, use_abs=True),
    label="Masked FC (|coherence| magnitude)"
)

print("\n" + "=" * 60)
print("ARM 4/5: Phase-Coherence-ONLY -- SIGNED (no FC/amplitude information)")
print("=" * 60)
phase_only_signed_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_phase_coherence_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, use_abs=False),
    label="Phase Coherence Only (signed)"
)

print("\n" + "=" * 60)
print("ARM 5/5: Phase-Coherence-ONLY -- |coherence| magnitude (no FC/amplitude information)")
print("=" * 60)
phase_only_abs_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_phase_coherence_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, use_abs=True),
    label="Phase Coherence Only (|coherence| magnitude)"
)

# =====================================================
# 7. HEAD-TO-HEAD COMPARISON TABLE + PAIRWISE SIGNIFICANCE TESTS
# =====================================================
# results_by_arm is the single source of truth for section 7 and 8:
# add/remove arms here and both the table and the figures follow.
results_by_arm = {
    "Static FC (baseline)": static_fc_results,
    "Masked FC (signed)": masked_fc_signed_results,
    "Masked FC (|coh|)": masked_fc_abs_results,
    "Coherence Only (signed)": phase_only_signed_results,
    "Coherence Only (|coh|)": phase_only_abs_results,
}
ARM_LABELS = list(results_by_arm.keys())
metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]

print("\n" + "=" * 70)
print("ABLATION SUMMARY: Mean ± Std across folds, all five arms")
print("=" * 70)
header = f"{'Metric':<14}" + "".join(f"{lbl:>26}" for lbl in ARM_LABELS)
print(header)
print("-" * len(header))
for m in metrics:
    row = f"{m:<14}"
    for lbl in ARM_LABELS:
        vals = np.array(results_by_arm[lbl][m], dtype=np.float64)
        row += f"{np.nanmean(vals):>19.3f}±{np.nanstd(vals):<5.3f}"
    print(row)
print("-" * len(header))

print("\n" + "=" * 70)
print("PAIRWISE COMPARISONS (paired t-test across folds, same folds every arm)")
print("=" * 70)
pair_header = f"{'Metric':<14}{'Arm A':>24}{'Arm B':>24}{'Delta (A-B)':>14}{'Paired p':>12}"
print(pair_header)
print("-" * len(pair_header))
for m in metrics:
    for lbl_a, lbl_b in itertools.combinations(ARM_LABELS, 2):
        a_vals = np.array(results_by_arm[lbl_a][m], dtype=np.float64)
        b_vals = np.array(results_by_arm[lbl_b][m], dtype=np.float64)
        valid = ~(np.isnan(a_vals) | np.isnan(b_vals))
        delta = np.nanmean(a_vals) - np.nanmean(b_vals)
        if valid.sum() >= 2 and np.std(a_vals[valid] - b_vals[valid]) > 0:
            _, p_val = stats.ttest_rel(a_vals[valid], b_vals[valid])
            p_str = f"{p_val:.3f}"
        else:
            p_str = "n/a"
        print(f"{m:<14}{lbl_a:>24}{lbl_b:>24}{delta:>+14.3f}{p_str:>12}")
print("-" * len(pair_header))
print("Positive Delta => Arm A outperforms Arm B on that metric.")
print("Paired p-value is a per-fold paired t-test (n=folds); with only "
      f"{N_SPLITS} folds this is a rough guide, not a rigorous significance claim -- "
      "treat p-values near the 0.05 boundary cautiously given the small n. For a "
      "paper, consider also reporting a permutation test (label shuffling) given "
      "the small number of folds. With 5 arms there are "
      f"{len(list(itertools.combinations(ARM_LABELS, 2)))} pairwise tests per metric -- "
      "if you report all of them, consider noting the multiple-comparisons caveat "
      "(e.g. Bonferroni/Holm) or focus the paper's main table on the comparisons "
      "that matter most (e.g. each ablation vs. the Static FC baseline).")

# =====================================================
# 8. PUBLICATION FIGURES
# =====================================================
# All figures are saved to ./figures at 600 DPI (PNG) and also as
# vector PDF. Colors are assigned by arm order so any number of arms
# (currently 5) is handled generically.
# =====================================================
ARM_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

def _color_for(label):
    idx = ARM_LABELS.index(label) if label in ARM_LABELS else 0
    return ARM_COLORS[idx % len(ARM_COLORS)]

def _savefig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.pdf"), bbox_inches="tight")
    print(f"Saved figures/{name}.png and .pdf")

# ---------- 8a. Confusion matrices (pooled across folds) ----------
def plot_confusion_matrices(results_dict, out_name="confusion_matrices"):
    n_arms = len(results_dict)
    fig, axes = plt.subplots(1, n_arms, figsize=(4.6 * n_arms, 4.4))
    if n_arms == 1:
        axes = [axes]
    for ax, (label, results) in zip(axes, results_dict.items()):
        y_true_pooled = np.concatenate(results["fold_y_true"])
        y_pred_pooled = np.concatenate(results["fold_y_pred"])
        cm = confusion_matrix(y_true_pooled, y_pred_pooled, labels=[0, 1])
        disp = ConfusionMatrixDisplay(cm, display_labels=["TD", "ASD"])
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(label, fontsize=9)
        ax.grid(False)
    fig.suptitle("Pooled Out-of-Fold Confusion Matrices", y=1.04, fontsize=13)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)

# ---------- 8b. ROC curves: per-fold + mean ± std band ----------
def plot_roc_curves(results_dict, out_name="roc_curves"):
    fig, ax = plt.subplots(figsize=(7, 7))
    mean_fpr = np.linspace(0, 1, 200)
    for label, results in results_dict.items():
        color = _color_for(label)
        tprs, aucs_fold = [], []
        for i, (y_true, y_score) in enumerate(zip(results["fold_y_true"], results["fold_y_score"])):
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            aucs_fold.append(roc_auc)
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
            ax.plot(fpr, tpr, alpha=0.10, lw=1, color=color)
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        std_tpr = np.std(tprs, axis=0)
        mean_auc = np.mean(aucs_fold)
        std_auc = np.std(aucs_fold)
        ax.plot(mean_fpr, mean_tpr, lw=2.5, color=color,
                label=f"{label} (AUC = {mean_auc:.3f} ± {std_auc:.3f})")
        ax.fill_between(mean_fpr, np.clip(mean_tpr - std_tpr, 0, 1),
                         np.clip(mean_tpr + std_tpr, 0, 1), color=color, alpha=0.10)
    ax.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray", label="Chance")
    ax.set_xlabel("False Positive Rate (1 - Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.set_title("ROC Curves: Ablation Comparison")
    ax.legend(loc="lower right", fontsize=7.5)
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)

# ---------- 8c. Precision-Recall curves (useful given class imbalance) ----------
def plot_pr_curves(results_dict, out_name="pr_curves"):
    fig, ax = plt.subplots(figsize=(7, 7))
    for label, results in results_dict.items():
        color = _color_for(label)
        y_true_pooled = np.concatenate(results["fold_y_true"])
        y_score_pooled = np.concatenate(results["fold_y_score"])
        precision, recall, _ = precision_recall_curve(y_true_pooled, y_score_pooled)
        ap = average_precision_score(y_true_pooled, y_score_pooled)
        ax.plot(recall, precision, lw=2.5, color=color, label=f"{label} (AP = {ap:.3f})")
    prevalence = np.mean(np.concatenate(list(results_dict.values())[0]["fold_y_true"]))
    ax.axhline(prevalence, linestyle="--", lw=1, color="gray",
               label=f"Chance (prevalence = {prevalence:.2f})")
    ax.set_xlabel("Recall (Sensitivity)")
    ax.set_ylabel("Precision (PPV)")
    ax.set_title("Precision-Recall Curves (pooled out-of-fold)")
    ax.legend(loc="lower left", fontsize=7.5)
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.01, 1.01)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)

# ---------- 8d. Metric comparison bar chart (mean ± std across folds) ----------
def plot_metric_comparison(results_dict, out_name="metric_comparison"):
    metric_names = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
    display_names = ["AUC", "Sensitivity", "Specificity", "F1", "Accuracy"]
    labels = list(results_dict.keys())
    n_arms = len(labels)
    x = np.arange(len(metric_names))
    width = 0.8 / n_arms
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, label in enumerate(labels):
        means = [np.nanmean(results_dict[label][m]) for m in metric_names]
        stds = [np.nanstd(results_dict[label][m]) for m in metric_names]
        color = _color_for(label)
        offset = (i - (n_arms - 1) / 2) * width
        bars = ax.bar(x + offset, means, width, yerr=stds, capsize=3,
                       label=label, color=color, alpha=0.85)
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=6.5, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(display_names)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.3)
    ax.set_title("Cross-Validated Performance: Ablation Comparison")
    ax.legend(loc="upper right", fontsize=7.5, ncol=2)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)

# ---------- 8e. Per-fold AUC distribution (boxplot / stripplot) ----------
def plot_fold_auc_distribution(results_dict, out_name="fold_auc_distribution"):
    labels = list(results_dict.keys())
    data = [results_dict[label]["auc"] for label in labels]
    colors = [_color_for(label) for label in labels]
    fig, ax = plt.subplots(figsize=(7.5, 5))
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5, showmeans=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.4)
    rng = np.random.default_rng(0)
    for i, (label, vals) in enumerate(zip(labels, data), start=1):
        jitter = rng.uniform(-0.06, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color=colors[i - 1],
                   edgecolor="black", linewidth=0.5, zorder=3, s=40)
    ax.set_ylabel("Fold-wise AUC")
    ax.set_title("Distribution of Per-Fold AUC Across CV Folds")
    ax.set_ylim(0, 1.05)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=8)
    fig.tight_layout()
    _savefig(fig, out_name)
    plt.close(fig)

# ---------- Run all figure-generation ----------
plot_confusion_matrices(results_by_arm)
plot_roc_curves(results_by_arm)
plot_pr_curves(results_by_arm)
plot_metric_comparison(results_by_arm)
plot_fold_auc_distribution(results_by_arm)

print(f"\nAll figures written to ./{FIGURES_DIR}/ (PNG @600dpi + vector PDF).")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Percentile-Masked Static FC vs. Plain Static FC (SVM)
#
# This extends the percentile-masked pipeline with a BASELINE:
# plain static Pearson FC (no phase-coherence masking at all).
# Both classifiers are evaluated on the SAME StratifiedKFold splits,
# using the SAME model class (SVC) and the SAME train_one_fold_svm(),
# so the only thing that differs between the two rows in the final
# comparison table is the feature-construction step. This isolates
# the effect of percentile phase-masking from confounds like
# different folds, different classifiers, or different tuning.
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert
from scipy import stats


# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared by both classifiers)
# =====================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


# =====================================================
# 4a. PERCENTILE-MASKED STATIC FC FEATURE (unchanged from before)
# =====================================================
PHASE_TR = 2.0             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band
PERCENTILE = 25.0          # keep the strongest (100 - PERCENTILE)% of phase-coherence edges


def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_matrix(ts_subject, L, R):
    """
    Shared helper: builds the plain static Pearson FC matrix for one
    subject over its real (non-padded) timepoints only. Used by BOTH
    the masked pipeline and the plain-FC baseline, so the underlying
    FC values being compared are guaranteed identical -- the only
    difference is whether masking is applied afterward.
    """
    T = ts_subject.shape[0]
    w = np.zeros(T, dtype=np.float32)
    w[:L] = 1.0
    fc_mat = weighted_corrcoef(ts_subject, w)
    fc_mat = (fc_mat + fc_mat.T) / 2.0  # force exact symmetry
    return fc_mat


def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4):
    """
    Per subject: static Pearson FC + static phase-coherence, combined
    via percentile masking (keep FC value only where |phase-coherence|
    clears the subject's own top-`percentile` threshold; else zero).
    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)

        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))
            diff = phase[:, :, None] - phase[:, None, :]
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)

        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold
        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)

        feats[i] = combined_mat[triu_idx]

    return feats


# =====================================================
# 4b. PLAIN STATIC FC FEATURE (baseline / comparison arm)
# =====================================================
# No phase-coherence, no masking: every edge's static Pearson FC value
# is kept as-is. This isolates whether percentile phase-masking adds
# (or costs) anything relative to the unmasked correlation matrix.
# =====================================================

def compute_static_fc_features(X_raw, real_lengths):
    """
    Per subject: plain static Pearson FC matrix (upper triangle,
    flattened), no masking of any kind.
    Returns: (N, n_pairs) float32 -- same shape/layout as the masked
    feature, so both can be fed through the identical SVM pipeline.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        fc_mat = compute_static_fc_matrix(X_raw[i], L, R)
        feats[i] = fc_mat[triu_idx]

    return feats


# =====================================================
# 5. SHARED SVM TRAIN/EVAL (identical for both feature types)
# =====================================================

def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    """
    Fits an SVM on whichever feature matrix is passed in. Features are
    standardized (fit on training subjects only, applied to val --
    fold-safe) before fitting. class_weight='balanced' handles the
    ASD/TD class imbalance.

    kernel="linear" is the default: with n_pairs often >> N_subjects,
    linear SVMs tend to generalize better and keep the model
    interpretable.

    AUC is computed from decision_function rather than predict_proba
    (avoids the extra internal CV pass Platt scaling would need).

    Returns: (model, scaler, val_auc, val_preds)
    """
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)

    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)

    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)

    return model, scaler, val_auc, val_preds


def evaluate_classifier(X, y, real_lengths, skf, feature_fn, feature_kwargs=None,
                         C=1.0, kernel="linear", label="classifier"):
    """
    Generic fold-runner: applies `feature_fn` (either the masked or the
    plain static FC feature function) per fold, then trains/evaluates
    an SVM. Because `skf` is instantiated once and reused for both
    calls, train/val subject splits are IDENTICAL across the two
    arms of the comparison -- no fold-assignment confound.
    """
    feature_kwargs = feature_kwargs or {}
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- [{label}] Fold {fold+1}: computing features ---")

        X_train_feat = feature_fn(X_train_raw, rl_train, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, **feature_kwargs)

        model, scaler, auc, val_preds = train_one_fold_svm(
            X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel
        )

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"[{label}] Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {label}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


# =====================================================
# 6. RUN BOTH ARMS ON THE SAME FOLDS, THEN COMPARE
# =====================================================
print("\n" + "=" * 60)
print(f"ARM 1/2: Percentile-Masked Static FC (SVM, top {100-PERCENTILE:.0f}% phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")

masked_fc_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_percentile_masked_fc_features,
    feature_kwargs=dict(TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE),
    label="Percentile-Masked Static FC"
)

print("\n" + "=" * 60)
print("ARM 2/2: Plain Static FC baseline (SVM, no phase masking)")
print("=" * 60)

static_fc_results = evaluate_classifier(
    X, y, real_lengths, skf,
    feature_fn=compute_static_fc_features,
    feature_kwargs={},
    label="Plain Static FC (baseline)"
)


# =====================================================
# 7. HEAD-TO-HEAD COMPARISON TABLE + PAIRED SIGNIFICANCE TEST
# =====================================================
print("\n" + "=" * 70)
print("HEAD-TO-HEAD: Percentile-Masked FC  vs.  Plain Static FC (baseline)")
print("=" * 70)

metrics = ["auc", "sensitivity", "specificity", "f1", "accuracy"]
header = f"{'Metric':<14}{'Masked FC':>16}{'Static FC':>16}{'Delta (M-S)':>16}{'Paired p':>12}"
print(header)
print("-" * len(header))

for m in metrics:
    masked_vals = np.array(masked_fc_results[m], dtype=np.float64)
    static_vals = np.array(static_fc_results[m], dtype=np.float64)

    masked_mean, masked_std = np.nanmean(masked_vals), np.nanstd(masked_vals)
    static_mean, static_std = np.nanmean(static_vals), np.nanstd(static_vals)
    delta = masked_mean - static_mean

    # Paired t-test across folds (same folds for both arms -> valid pairing).
    # Only meaningful with >=2 folds and no NaNs; guarded accordingly.
    valid = ~(np.isnan(masked_vals) | np.isnan(static_vals))
    if valid.sum() >= 2 and np.std(masked_vals[valid] - static_vals[valid]) > 0:
        _, p_val = stats.ttest_rel(masked_vals[valid], static_vals[valid])
        p_str = f"{p_val:.3f}"
    else:
        p_str = "n/a"

    print(f"{m:<14}{masked_mean:>9.3f}±{masked_std:<5.3f}"
          f"{static_mean:>9.3f}±{static_std:<5.3f}"
          f"{delta:>+15.3f}  {p_str:>10}")

print("-" * len(header))
print("Positive Delta => percentile-masked FC outperforms plain static FC on that metric.")
print("Paired p-value is a per-fold paired t-test (n=folds); with only "
      f"{N_SPLITS} folds this is a rough guide, not a rigorous significance claim -- "
      "treat p-values near the 0.05 boundary cautiously given the small n.")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Percentile-Masked Static FC Classifier
#
# Combination rule: for each subject, keep the static Pearson FC value
# at an edge ONLY IF the static phase-coherence at that edge is in the
# top X percentile (i.e. "significant" relative to that subject's own
# phase-coherence distribution); otherwise the edge is zeroed out.
# Phase-coherence acts purely as a per-subject SELECTOR of which FC
# edges to keep -- the retained FC values are passed through
# unchanged, never blended or scaled.
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. PERCENTILE-MASKED STATIC FC FEATURE
# =====================================================
# For each subject:
#   1. static Pearson FC matrix -- ordinary whole-scan correlation,
#      computed once over ALL real (non-padded) timepoints.
#   2. static phase-coherence matrix -- band-pass filter + Hilbert
#      transform to get instantaneous phase, then the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is AVERAGED across
#      every real timepoint into one (R, R) matrix.
#   3. COMBINE via percentile masking: compute the top-`percentile`
#      threshold of |phase-coherence| (per subject, from the upper
#      triangle only), then keep the FC value at an edge ONLY IF its
#      |phase-coherence| clears that threshold -- otherwise zero it
#      out. The retained FC values are passed through UNCHANGED (no
#      blending/scaling); phase-coherence is purely a selector deciding
#      which edges survive.
#
# TR CAVEAT: this dataset resamples every subject to a FIXED NUMBER of
# timepoints (TARGET_TIME), not a fixed PHYSICAL DURATION -- so if your
# included ABIDE sites don't share (approximately) the same TR, the
# band-pass band refers to a different physical frequency band per
# subject. Verify TR consistency (via ABIDE phenotypic metadata) before
# trusting these features, or restrict SITE_FILTER to a single-TR site.
# =====================================================

PHASE_TR = 2.0             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed
PERCENTILE = 20.0          # keep the strongest (100 - PERCENTILE)% of phase-coherence edges


def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4):
    """
    Per subject: builds the static Pearson FC matrix and the static
    phase-coherence matrix, then combines them via PERCENTILE MASKING
    -- keeps the FC value at an edge only if that edge's
    |phase-coherence| is in the top `percentile` percent (per subject,
    computed from the upper triangle only so the trivial diagonal
    doesn't bias the threshold); all other edges are zeroed.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson FC matrix (full R x R, not yet flattened) ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        fc_mat = weighted_corrcoef(X_raw[i], w)
        # Force EXACT symmetry: fc_mat is symmetric by mathematical
        # construction, but floating-point matrix multiplication
        # doesn't guarantee the computed result is bit-for-bit
        # symmetric -- tiny rounding differences would otherwise leak
        # into the mask comparison below.
        fc_mat = (fc_mat + fc_mat.T) / 2.0

        # --- static phase-coherence matrix (full R x R, not yet flattened) ---
        if L < 10:  # too short to filter/estimate phase meaningfully
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))  # (L, R)
            diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)  # (R, R)

        # --- percentile-mask combination, BEFORE flattening ---
        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold  # (R, R) bool, symmetric (phase_mat is symmetric)
        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)

        feats[i] = combined_mat[triu_idx]

    return feats


class fMRIMaskedFCClassifier(nn.Module):
    """
    Single-branch classifier for the percentile-masked FC feature. A
    linear projection down to a small hidden size, then a compact MLP
    head classifies from there.
    """
    def __init__(self, n_pairs, proj_dim=16, hidden_dim=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(n_pairs, proj_dim), nn.GELU())
        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        h = self.proj(x)
        return self.classifier(h)


def train_one_fold_masked_fc(X_train, y_train, X_val, y_val, n_pairs, epochs=60, lr=1e-4):
    model = fMRIMaskedFCClassifier(n_pairs=n_pairs).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_masked_fc_classifier(X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND,
                                   percentile=PERCENTILE):
    """
    Per fold: computes the percentile-masked FC feature for train and
    val (deterministic per-subject, no fitting step needed), then
    trains/evaluates the single-branch classifier. No feature
    selection is applied -- the full masked feature vector is used
    as-is.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing percentile-masked FC (top {100-percentile:.0f}% phase edges) ---")

        X_train_feat = compute_percentile_masked_fc_features(X_train_raw, rl_train, TR, band, percentile)
        X_val_feat = compute_percentile_masked_fc_features(X_val_raw, rl_val, TR, band, percentile)

        model, auc = train_one_fold_masked_fc(
            X_train_feat, y_train, X_val_feat, y_val, n_pairs=X_train_feat.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_feat).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Percentile-Masked Static FC Classifier: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print(f"Training Percentile-Masked Static FC Classifier (top {100-PERCENTILE:.0f}% phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
masked_fc_results = evaluate_masked_fc_classifier(
    X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE
)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Percentile-Masked Static FC Classifier (SVM)
#
# Combination rule: for each subject, keep the static Pearson FC value
# at an edge ONLY IF the static phase-coherence at that edge is in the
# top X percentile (i.e. "significant" relative to that subject's own
# phase-coherence distribution); otherwise the edge is zeroed out.
# Phase-coherence acts purely as a per-subject SELECTOR of which FC
# edges to keep -- the retained FC values are passed through
# unchanged, never blended or scaled.
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP
# =====================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =====================================================
# 4. PERCENTILE-MASKED STATIC FC FEATURE
# =====================================================
# For each subject:
#   1. static Pearson FC matrix -- ordinary whole-scan correlation,
#      computed once over ALL real (non-padded) timepoints.
#   2. static phase-coherence matrix -- band-pass filter + Hilbert
#      transform to get instantaneous phase, then the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is AVERAGED across
#      every real timepoint into one (R, R) matrix.
#   3. COMBINE via percentile masking: compute the top-`percentile`
#      threshold of |phase-coherence| (per subject, from the upper
#      triangle only), then keep the FC value at an edge ONLY IF its
#      |phase-coherence| clears that threshold -- otherwise zero it
#      out. The retained FC values are passed through UNCHANGED (no
#      blending/scaling); phase-coherence is purely a selector deciding
#      which edges survive.
#
# TR CAVEAT: this dataset resamples every subject to a FIXED NUMBER of
# timepoints (TARGET_TIME), not a fixed PHYSICAL DURATION -- so if your
# included ABIDE sites don't share (approximately) the same TR, the
# band-pass band refers to a different physical frequency band per
# subject. Verify TR consistency (via ABIDE phenotypic metadata) before
# trusting these features, or restrict SITE_FILTER to a single-TR site.
# =====================================================

PHASE_TR = 2.0             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed
PERCENTILE = 15.0          # keep the strongest (100 - PERCENTILE)% of phase-coherence edges


def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4):
    """
    Per subject: builds the static Pearson FC matrix and the static
    phase-coherence matrix, then combines them via PERCENTILE MASKING
    -- keeps the FC value at an edge only if that edge's
    |phase-coherence| is in the top `percentile` percent (per subject,
    computed from the upper triangle only so the trivial diagonal
    doesn't bias the threshold); all other edges are zeroed.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson FC matrix (full R x R, not yet flattened) ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        fc_mat = weighted_corrcoef(X_raw[i], w)
        # Force EXACT symmetry: fc_mat is symmetric by mathematical
        # construction, but floating-point matrix multiplication
        # doesn't guarantee the computed result is bit-for-bit
        # symmetric -- tiny rounding differences would otherwise leak
        # into the mask comparison below.
        fc_mat = (fc_mat + fc_mat.T) / 2.0

        # --- static phase-coherence matrix (full R x R, not yet flattened) ---
        if L < 10:  # too short to filter/estimate phase meaningfully
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))  # (L, R)
            diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)  # (R, R)

        # --- percentile-mask combination, BEFORE flattening ---
        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold  # (R, R) bool, symmetric (phase_mat is symmetric)
        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)

        feats[i] = combined_mat[triu_idx]

    return feats


def train_one_fold_svm(X_train, y_train, X_val, y_val, C=1.0, kernel="linear", gamma="scale"):
    """
    Fits an SVM on the percentile-masked FC feature for this fold.
    Features are standardized (fit on training subjects only, applied
    to val -- fold-safe) before fitting, since SVMs are scale-sensitive
    and the masked feature mixes real correlation values with hard
    zeros. class_weight='balanced' handles the ASD/TD class imbalance
    (the earlier PyTorch models did this via a weighted loss).

    kernel="linear" is the default: with n_pairs often >> N_subjects
    (a classic high-dimension/low-sample-size regime for FC data),
    linear SVMs tend to generalize better than RBF and keep the model
    interpretable (the fitted weight vector is directly an importance
    score per ROI-pair). Switch to kernel="rbf" if you want to allow
    non-linear decision boundaries.

    AUC is computed from decision_function (the signed distance to the
    separating hyperplane) rather than predict_proba, since Platt
    scaling (probability=True) requires an extra internal CV pass and
    isn't needed just to rank subjects for ROC-AUC.

    Returns: (model, scaler, val_auc, val_preds)
    """
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)

    model = SVC(C=C, kernel=kernel, gamma=gamma, class_weight="balanced")
    model.fit(Xtr, y_train)

    val_scores = model.decision_function(Xva)
    val_auc = roc_auc_score(y_val, val_scores)
    val_preds = model.predict(Xva)

    return model, scaler, val_auc, val_preds


def evaluate_masked_fc_classifier(X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND,
                                   percentile=PERCENTILE, C=1.0, kernel="linear"):
    """
    Per fold: computes the percentile-masked FC feature for train and
    val (deterministic per-subject, no fitting step needed), then
    trains/evaluates an SVM. No feature selection is applied -- the
    full masked feature vector is used as-is (after standardization).
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing percentile-masked FC (top {100-percentile:.0f}% phase edges) ---")

        X_train_feat = compute_percentile_masked_fc_features(X_train_raw, rl_train, TR, band, percentile)
        X_val_feat = compute_percentile_masked_fc_features(X_val_raw, rl_val, TR, band, percentile)

        model, scaler, auc, val_preds = train_one_fold_svm(
            X_train_feat, y_train, X_val_feat, y_val, C=C, kernel=kernel
        )

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Percentile-Masked Static FC Classifier (SVM): Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print(f"Training Percentile-Masked Static FC Classifier (SVM, top {100-PERCENTILE:.0f}% phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
masked_fc_results = evaluate_masked_fc_classifier(
    X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE
)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Percentile-Masked Static FC Features
#            -> Stacked Denoising Autoencoder + MLP Classifier
#               (method from Heinsfeld et al. 2018, NeuroImage: Clinical)
#
# Feature extraction (unchanged from original code):
# Combination rule: for each subject, keep the static Pearson FC value
# at an edge ONLY IF the static phase-coherence at that edge is in the
# top X percentile (i.e. "significant" relative to that subject's own
# phase-coherence distribution); otherwise the edge is zeroed out.
# Phase-coherence acts purely as a per-subject SELECTOR of which FC
# edges to keep -- the retained FC values are passed through
# unchanged, never blended or scaled.
#
# Classifier (replaced to match the paper):
# 1. Two STACKED DENOISING AUTOENCODERS are pretrained unsupervised
#    on the training fold's flattened FC feature vectors:
#       AE1: n_pairs -> 1000 -> n_pairs   (input corruption p=0.2)
#       AE2: 1000    -> 600  -> 1000      (input corruption p=0.3)
# 2. The encoder weights from AE1 and AE2 initialize a Multi-Layer
#    Perceptron with architecture n_pairs -> 1000 -> 600 -> 2 (softmax).
# 3. The MLP is fine-tuned end-to-end in a supervised fashion
#    (cross-entropy loss) to classify ASD vs TD -- this fine-tuning
#    step uses the autoencoder weights purely as an initialization
#    (transfer learning), exactly as described in the paper.
#
# All pretraining and fine-tuning are done PER FOLD using only that
# fold's training data, to avoid leakage into the validation set.
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from scipy.signal import butter, filtfilt, hilbert

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# 2. DATA LOADER (unchanged)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}
NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"
if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }
    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue
        try:
            ts = np.loadtxt(file)
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue
            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T
            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]
            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)
        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())

# =====================================================
# 3. STRATIFIED K-FOLD SETUP
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# =====================================================
# 4. PERCENTILE-MASKED STATIC FC FEATURE (unchanged)
# =====================================================
PHASE_TR = 2.0             # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed
PERCENTILE = 30.0          # keep the strongest (100 - PERCENTILE)% of phase-coherence edges


def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total
    weight is (near) zero, returns an all-zero matrix rather than
    dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)
    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum
    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_percentile_masked_fc_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND,
                                           percentile=PERCENTILE, order=4):
    """
    Per subject: builds the static Pearson FC matrix and the static
    phase-coherence matrix, then combines them via PERCENTILE MASKING
    -- keeps the FC value at an edge only if that edge's
    |phase-coherence| is in the top `percentile` percent (per subject,
    computed from the upper triangle only so the trivial diagonal
    doesn't bias the threshold); all other edges are zeroed.
    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson FC matrix ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        fc_mat = weighted_corrcoef(X_raw[i], w)
        fc_mat = (fc_mat + fc_mat.T) / 2.0

        # --- static phase-coherence matrix ---
        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))
            diff = phase[:, :, None] - phase[:, None, :]
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)

        # --- percentile-mask combination, BEFORE flattening ---
        abs_phase_triu = np.abs(phase_mat[triu_idx])
        threshold = np.percentile(abs_phase_triu, percentile)
        mask = np.abs(phase_mat) >= threshold
        combined_mat = np.where(mask, fc_mat, 0.0).astype(np.float32)
        feats[i] = combined_mat[triu_idx]

    return feats


# =====================================================
# 5. STACKED DENOISING AUTOENCODER + MLP CLASSIFIER
#    (method used in Heinsfeld et al. 2018)
# =====================================================
AE1_HIDDEN = 1000     # matches paper: n_pairs -> 1000 -> n_pairs
AE2_HIDDEN = 600       # matches paper: 1000 -> 600 -> 1000
AE1_CORRUPTION = 0.2  # binomial corruption prob, n=1 p=0.8 kept -> 20% dropped (paper)
AE2_CORRUPTION = 0.3# binomial corruption prob, n=1 p=0.7 kept -> 30% dropped (paper)
AE_EPOCHS = 100
AE_LR = 1e-3
AE_BATCH_SIZE = 32

FINE_TUNE_EPOCHS = 150
FINE_TUNE_LR = 1e-4


class DenoisingAutoencoder(nn.Module):
    """
    Single-layer denoising autoencoder. Input is stochastically
    corrupted (binomial masking, i.e. random elements zeroed) before
    being encoded; the decoder attempts to reconstruct the *clean*
    (uncorrupted) input from the corrupted encoding. This forces the
    encoder to learn a robust, denoised representation -- exactly the
    procedure described in Vincent et al. (2008, 2010) and used by
    Heinsfeld et al. (2018) for ASD/TD classification.
    """
    def __init__(self, input_dim, hidden_dim, corruption_prob):
        super().__init__()
        self.corruption_prob = corruption_prob
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.act = nn.Tanh()

    def corrupt(self, x):
        if self.training and self.corruption_prob > 0:
            mask = (torch.rand_like(x) > self.corruption_prob).float()
            return x * mask
        return x

    def encode(self, x):
        return self.act(self.encoder(x))

    def forward(self, x):
        x_corrupted = self.corrupt(x)
        h = self.encode(x_corrupted)
        x_recon = self.decoder(h)
        return x_recon, h


def train_denoising_autoencoder(X_train, input_dim, hidden_dim, corruption_prob,
                                 epochs=AE_EPOCHS, lr=AE_LR, batch_size=AE_BATCH_SIZE):
    """
    Unsupervised pretraining of a single denoising autoencoder,
    minimizing mean-squared reconstruction error (as in the paper).
    Labels are never used here.
    """
    model = DenoisingAutoencoder(input_dim, hidden_dim, corruption_prob).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    Xtr = torch.tensor(X_train, dtype=torch.float32).to(device)
    n = Xtr.shape[0]

    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            batch = Xtr[idx]
            optimizer.zero_grad()
            recon, _ = model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch.shape[0]
        epoch_loss /= n
    return model


class DNNClassifier(nn.Module):
    """
    Multi-layer perceptron with architecture
        n_pairs -> 1000 -> 600 -> 2 (softmax)
    matching the fine-tuning network in Heinsfeld et al. (2018).
    Weights for the first two layers are meant to be initialized from
    the pretrained denoising autoencoder encoders (transfer learning);
    the whole network is then fine-tuned end-to-end in a supervised
    fashion via cross-entropy loss.
    """
    def __init__(self, input_dim, hidden1=AE1_HIDDEN, hidden2=AE2_HIDDEN, n_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, n_classes)
        self.act = nn.Tanh()
        self.dropout = nn.Dropout(0.3)

    def init_from_autoencoders(self, ae1, ae2):
        with torch.no_grad():
            self.fc1.weight.copy_(ae1.encoder.weight)
            self.fc1.bias.copy_(ae1.encoder.bias)
            self.fc2.weight.copy_(ae2.encoder.weight)
            self.fc2.bias.copy_(ae2.encoder.bias)

    def forward(self, x):
        h1 = self.dropout(self.act(self.fc1(x)))
        h2 = self.dropout(self.act(self.fc2(h1)))
        logits = self.fc3(h2)
        return logits


def pretrain_stacked_autoencoders(X_train_scaled, n_pairs):
    """
    Stage 1 (unsupervised): trains AE1 (n_pairs -> 1000 -> n_pairs)
    on the raw feature vectors, then trains AE2 (1000 -> 600 -> 1000)
    on AE1's clean (uncorrupted) hidden encoding -- i.e. layer-wise
    greedy pretraining of the stack, as in the paper.
    """
    print("  Pretraining Autoencoder 1 (n_pairs -> 1000 -> n_pairs, corruption=0.20)...")
    ae1 = train_denoising_autoencoder(
        X_train_scaled, input_dim=n_pairs, hidden_dim=AE1_HIDDEN,
        corruption_prob=AE1_CORRUPTION
    )

    ae1.eval()
    with torch.no_grad():
        Xtr_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
        H1_train = ae1.encode(Xtr_t).cpu().numpy()

    print("  Pretraining Autoencoder 2 (1000 -> 600 -> 1000, corruption=0.30)...")
    ae2 = train_denoising_autoencoder(
        H1_train, input_dim=AE1_HIDDEN, hidden_dim=AE2_HIDDEN,
        corruption_prob=AE2_CORRUPTION
    )

    return ae1, ae2


def fine_tune_classifier(ae1, ae2, X_train_scaled, y_train, X_val_scaled, y_val, n_pairs,
                          epochs=FINE_TUNE_EPOCHS, lr=FINE_TUNE_LR):
    """
    Stage 2 (supervised fine-tuning): builds the MLP classifier,
    initializes its first two layers from the pretrained autoencoder
    encoders, then fine-tunes the whole network end-to-end to
    minimize cross-entropy classification error (softmax output),
    exactly as described in the paper.
    """
    model = DNNClassifier(input_dim=n_pairs).to(device)
    model.init_from_autoencoders(ae1, ae2)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)

    best_auc, best_state, patience, bad_epochs = 0.0, None, 15, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_dae_mlp_classifier(X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND,
                                 percentile=PERCENTILE):
    """
    Per fold:
      1. Compute the percentile-masked FC feature (same as original code).
      2. Standardize features (fit on train only).
      3. Pretrain the stacked denoising autoencoders (unsupervised, train only).
      4. Initialize and fine-tune the MLP classifier (supervised).
      5. Evaluate on the held-out validation fold.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold + 1}: computing percentile-masked FC (top {100 - percentile:.0f}% phase edges) ---")
        X_train_feat = compute_percentile_masked_fc_features(X_train_raw, rl_train, TR, band, percentile)
        X_val_feat = compute_percentile_masked_fc_features(X_val_raw, rl_val, TR, band, percentile)
        n_pairs = X_train_feat.shape[1]

        # Standardize (fit on train fold only, to avoid leakage)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_feat).astype(np.float32)
        X_val_scaled = scaler.transform(X_val_feat).astype(np.float32)

        print("  Stacked denoising autoencoder pretraining (unsupervised)...")
        ae1, ae2 = pretrain_stacked_autoencoders(X_train_scaled, n_pairs)

        print("  Fine-tuning MLP classifier (supervised)...")
        model, auc = fine_tune_classifier(
            ae1, ae2, X_train_scaled, y_train, X_val_scaled, y_val, n_pairs
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold + 1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Stacked Denoising Autoencoder + MLP Classifier: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print(f"Training Stacked Denoising Autoencoder + MLP Classifier (Heinsfeld et al. 2018 method)")
print(f"Using percentile-masked FC features (top {100 - PERCENTILE:.0f}% phase edges)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")

dae_mlp_results = evaluate_dae_mlp_classifier(
    X, y, real_lengths, skf, TR=PHASE_TR, band=PHASE_BAND, percentile=PERCENTILE
)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Static FC + Static Phase-Coherence Fusion (learned gate)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from scipy.signal import butter, filtfilt, hilbert

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. WEIGHTED FC UTILITIES (Pearson + shrinkage-regularized partial)
# =====================================================
# General-purpose weighted functional-connectivity estimators. A weight
# vector of all-ones over a subject's real (non-padded) timepoints
# gives ordinary whole-scan static FC -- that's how these are used
# below (section 6). They also generalize to arbitrary non-negative
# per-timepoint weights (e.g. posterior probabilities, cluster
# membership masks) if you later reintroduce any dynamic/state-based
# weighting scheme.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero, returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Weighted PARTIAL correlation matrix, via a shrinkage-regularized
    precision matrix.

    data: (T, n_rois), weights: (T,) non-negative.

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total weight)
    comfortably exceeds n_rois. With whole-brain ROI counts (e.g. 200)
    and short scans (T ~ 78), the raw weighted covariance matrix can be
    singular or near-singular -- naive inversion would silently produce
    unstable, unusable partial correlations. To guard against this, the
    covariance diagonal is shrunk toward a scaled identity before
    inversion (a ridge-regularized precision matrix, in the same spirit
    as Ledoit-Wolf shrinkage / graphical lasso), which guarantees
    invertibility and stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain settings; will likely be
    unstable). Higher values shrink more aggressively toward
    independence (a diagonal-only precision matrix), trading some
    sensitivity for stability. 0.1 is a reasonable starting point;
    increase if you still see extreme values or instability (e.g. try
    0.2-0.3 for whole-brain/200-ROI settings), decrease if you've
    restricted to a small ROI subset (e.g. a single network with
    T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


# =====================================================
# 5. F-SCORE FEATURE SELECTION UTILITY
# =====================================================

def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=1500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on FC vectors, computed using ONLY the training subjects of the
    current fold (fold-safe).

    X_train_states: (N_train, n_groups, n_pairs) -- one FC vector of
                    n_pairs per "group" (e.g. a branch/source; for the
                    static case below, n_groups=1).

    The F-score for each ROI-pair is computed SEPARATELY for every
    group, and these per-group F-scores are then AVERAGED ACROSS GROUPS
    to get one final importance score per pair. When n_groups=1 (the
    static case), this trivially reduces to plain per-pair F-scoring.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_groups, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER GROUP, then stack
    per_group_f_scores = np.zeros((n_groups, n_pairs), dtype=np.float64)
    for g in range(n_groups):
        group_features = X_train_states[:, g, :]  # (N_train, n_pairs)
        f_scores_g, _ = f_classif(group_features, y_train)
        per_group_f_scores[g] = np.nan_to_num(f_scores_g, nan=0.0)

    # average the F-scores ACROSS GROUPS (not the raw features)
    avg_f_scores = per_group_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


# =====================================================
# 6. STATIC FC + STATIC PHASE-COHERENCE FUSION (learned gate)
# =====================================================
# Each subject is reduced to exactly TWO static (whole-scan) feature
# vectors -- no HMM, no clustering, no per-state/dynamic structure:
#   1. static FC: ordinary Pearson/partial-correlation FC, computed
#      once over the ENTIRE (real, non-padded) scan -- reuses
#      weighted_corrcoef/weighted_partial_corrcoef with a weight vector
#      of all-ones over real timepoints (i.e. the "one giant state =
#      the whole scan" special case).
#   2. static phase-coherence FC: band-pass filter + Hilbert transform
#      to get instantaneous phase, then the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is simply AVERAGED
#      across every real timepoint into one (R, R) matrix. No k-means,
#      no per-fold fitting needed for this step -- it's a deterministic
#      per-subject computation. This is the direct phase-based analogue
#      of static Pearson FC.
#
# These two static vectors are fused via a learned GATE
# (fMRIStaticFusionGated): each branch gets its own linear projection,
# then a small gate network (one linear + sigmoid layer, looking at
# both projections together) decides how much to trust each branch,
# per subject -- h = g*h_fc + (1-g)*h_phase. This is directly
# interpretable (inspect g to see which view the model relied on for a
# given subject) and uses far fewer parameters than a self-attention
# block, since there's no multi-head Q/K/V machinery, just one small
# linear layer producing the blend weight.
# =====================================================

# --- placeholder: SET THIS to your actual (site-consistent) TR in seconds ---
PHASE_TR = 2.0           # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_features(X_raw, real_lengths, fc_method="pearson", shrinkage_alpha=0.1):
    """
    Static whole-scan FC per subject: Pearson or shrinkage-regularized
    partial correlation, computed once over ALL real (non-padded)
    timepoints (weight = 1 for real timepoints, 0 for padding) -- no
    per-state/dynamic structure at all.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc = weighted_corrcoef(X_raw[i], w)
        feats[i] = fc[triu_idx]

    return feats


def compute_static_phase_coherence_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND, order=4):
    """
    Static grand-average phase-coherence FC per subject: band-pass
    filter + Hilbert transform to get instantaneous phase, compute the
    instantaneous phase-coherence matrix at EVERY real timepoint, then
    AVERAGE these matrices across all real timepoints into one (R, R)
    matrix -- no clustering step. This collapses the whole scan's phase
    relationships into one static summary, the phase-based analogue of
    static Pearson FC.

    Returns: (N, n_pairs) float32

    IMPORTANT TR CAVEAT: this dataset resamples every subject to a
    FIXED NUMBER of timepoints (TARGET_TIME), not a fixed PHYSICAL
    DURATION -- so if your included ABIDE sites don't share
    (approximately) the same TR, the band here refers to a different
    physical frequency band per subject. Verify TR consistency (via
    ABIDE phenotypic metadata) before trusting these features, or
    restrict SITE_FILTER to a single-TR site.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
        coh = np.cos(diff)                              # (L, R, R)
        grand_avg_coh = coh.mean(axis=0)                # (R, R) -- average over time

        feats[i] = grand_avg_coh[triu_idx]

    return feats


def select_top_pairs_2d(X_train_2d, y_train, k):
    """
    Thin wrapper around select_top_fc_pairs_by_fscore for the static
    (single-vector-per-subject, no group axis) case -- reshapes to
    (N, 1, n_pairs) so the existing per-group-then-average F-score
    logic still applies (trivially, since there's only one group).
    """
    top_idx, scores = select_top_fc_pairs_by_fscore(X_train_2d[:, None, :], y_train, k=k)
    return top_idx, scores


class fMRIStaticFusionGated(nn.Module):
    """
    Fuses two STATIC representations per subject via a learned GATE
    rather than concatenation or attention. Each branch (static FC,
    static phase-coherence) gets its own linear projection into a
    shared d-dimensional space; a small gate network then looks at
    both projected vectors together and outputs g in (0, 1) --
    per-dimension by default (vector gate) -- which blends them:

        h = g * h_fc + (1 - g) * h_phase

    g close to 1 means "for this subject/dimension, trust the FC view";
    close to 0 means "trust the phase-coherence view". This is directly
    interpretable per subject (call forward with return_gate=True to
    inspect it) and uses far fewer parameters than a self-attention
    block, since the gate is just one small linear+sigmoid layer rather
    than full multi-head Q/K/V projections plus a feedforward block.
    """
    def __init__(self, n_pairs_fc, n_pairs_phase, proj_dim=16, hidden_dim=32,
                 dropout=0.5, n_classes=2, vector_gate=True):
        super().__init__()
        self.fc_proj = nn.Sequential(nn.Linear(n_pairs_fc, proj_dim), nn.GELU())
        self.phase_proj = nn.Sequential(nn.Linear(n_pairs_phase, proj_dim), nn.GELU())

        # gate looks at BOTH projected branches to decide how to blend them
        gate_out_dim = proj_dim if vector_gate else 1
        self.gate = nn.Linear(proj_dim * 2, gate_out_dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x_fc, x_phase, return_gate=False):
        # x_fc: (B, n_pairs_fc), x_phase: (B, n_pairs_phase)
        h_fc = self.fc_proj(x_fc)        # (B, proj_dim)
        h_phase = self.phase_proj(x_phase)  # (B, proj_dim)

        gate_input = torch.cat([h_fc, h_phase], dim=1)  # (B, 2*proj_dim)
        g = torch.sigmoid(self.gate(gate_input))        # (B, proj_dim) or (B, 1)

        h = g * h_fc + (1 - g) * h_phase  # (B, proj_dim) -- broadcasts if g is (B,1)
        logits = self.classifier(h)

        if return_gate:
            return logits, g
        return logits


def train_one_fold_static_fusion(X_train_fc, X_train_phase, y_train,
                                  X_val_fc, X_val_phase, y_val,
                                  n_pairs_fc, n_pairs_phase, epochs=60, lr=1e-4):
    model = fMRIStaticFusionGated(n_pairs_fc=n_pairs_fc, n_pairs_phase=n_pairs_phase).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr_fc = torch.tensor(X_train_fc).to(device)
    Xtr_phase = torch.tensor(X_train_phase).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva_fc = torch.tensor(X_val_fc).to(device)
    Xva_phase = torch.tensor(X_val_phase).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr_fc, Xtr_phase)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_fc, Xva_phase)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_static_fusion_transformer(X, y, real_lengths, skf, k_fc_select=1500,
                                        fc_method="pearson", shrinkage_alpha=0.1,
                                        TR=PHASE_TR, band=PHASE_BAND):
    """
    Per fold: computes static FC and static phase-coherence features
    (both deterministic per-subject, no fold-fitting needed for the
    features themselves), runs F-score selection SEPARATELY on each
    branch (fold-safe: fit on training subjects only), then trains the
    gated static fusion model.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing static FC + static phase-coherence features ---")

        X_train_fc = compute_static_fc_features(X_train_raw, rl_train, fc_method, shrinkage_alpha)
        X_val_fc = compute_static_fc_features(X_val_raw, rl_val, fc_method, shrinkage_alpha)

        X_train_phase = compute_static_phase_coherence_features(X_train_raw, rl_train, TR, band)
        X_val_phase = compute_static_phase_coherence_features(X_val_raw, rl_val, TR, band)

        if k_fc_select is not None:
            fc_idx, _ = select_top_pairs_2d(X_train_fc, y_train, k_fc_select)
            X_train_fc = X_train_fc[:, fc_idx]
            X_val_fc = X_val_fc[:, fc_idx]

            phase_idx, _ = select_top_pairs_2d(X_train_phase, y_train, k_fc_select)
            X_train_phase = X_train_phase[:, phase_idx]
            X_val_phase = X_val_phase[:, phase_idx]

            print(f"F-score selection: static-FC kept {len(fc_idx)}, "
                  f"static-phase kept {len(phase_idx)} ROI-pair features "
                  f"(selected independently per branch)")

        model, auc = train_one_fold_static_fusion(
            X_train_fc, X_train_phase, y_train, X_val_fc, X_val_phase, y_val,
            n_pairs_fc=X_train_fc.shape[1], n_pairs_phase=X_train_phase.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_fc_t = torch.tensor(X_val_fc).to(device)
            Xva_phase_t = torch.tensor(X_val_phase).to(device)
            val_logits = model(Xva_fc_t, Xva_phase_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Static FC + Static Phase-Coherence Fusion: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Static FC + Static Phase-Coherence Fusion (learned gate)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
static_fusion_results = evaluate_static_fusion_transformer(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", shrinkage_alpha=0.1
)


# =====================================================
# 7. ELEMENT-WISE MATRIX FUSION (Pearson FC ⊙/+ phase-coherence FC)
# =====================================================
# Rather than fusing the two static views AFTER collapsing each to a
# flat vector (section 6's gated model, which never sees the (R, R)
# structure), this section combines the two (R, R) matrices directly,
# edge by edge, into ONE fused FC matrix per subject -- before any
# flattening or modeling happens. This collapses the whole pipeline
# back to a SINGLE feature branch, so no dual-branch fusion model is
# needed downstream at all: just the original single-FC classifier
# pattern.
#
# Two combination methods are supported:
#   "hadamard"    : combined = FC_pearson * FC_phase (element-wise).
#                   An edge only stays strong if BOTH views agree it's
#                   strong -- tends to suppress noise that only one
#                   method happens to pick up, at the cost of also
#                   suppressing genuine signal only one view captures.
#   "weighted_avg": combined = alpha*FC_pearson + (1-alpha)*FC_phase.
#                   A softer blend; alpha=0.5 weighs both views
#                   equally, alpha>0.5 favors Pearson, alpha<0.5 favors
#                   phase-coherence.
# =====================================================

def compute_combined_static_fc_features(X_raw, real_lengths, fc_method="pearson",
                                         shrinkage_alpha=0.1, TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5, order=4):
    """
    For each subject: computes the static Pearson/partial FC matrix AND
    the static phase-coherence matrix (same definitions as
    compute_static_fc_features / compute_static_phase_coherence_features),
    then combines them ELEMENT-WISE into a single (R, R) matrix before
    flattening -- i.e. fusion happens at the edge level, not after
    vectorization.

    combine_method: "hadamard" (element-wise product) or "weighted_avg"
                     (alpha*pearson + (1-alpha)*phase).
    alpha: only used for "weighted_avg" -- weight given to the FC branch.

    Returns: (N, n_pairs) float32 -- a SINGLE combined feature vector
             per subject (not two branches).
    """
    if combine_method not in ("hadamard", "weighted_avg"):
        raise ValueError(f"Unknown combine_method: {combine_method!r} "
                          f"(expected 'hadamard' or 'weighted_avg')")

    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson/partial FC matrix (full R x R, not yet flattened) ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc_mat = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc_mat = weighted_corrcoef(X_raw[i], w)
        # Force EXACT symmetry: fc_mat is symmetric by mathematical
        # construction (it comes from A^T @ W @ A), but floating-point
        # matrix multiplication doesn't guarantee the computed result is
        # bit-for-bit symmetric -- tiny rounding differences between
        # fc_mat[a,b] and fc_mat[b,a] would otherwise silently leak into
        # combined_mat via the element-wise product below.
        fc_mat = (fc_mat + fc_mat.T) / 2.0

        # --- static phase-coherence matrix (full R x R, not yet flattened) ---
        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))  # (L, R)
            diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)  # (R, R)

        # --- element-wise combination, BEFORE flattening ---
        if combine_method == "hadamard":
            combined_mat = fc_mat * phase_mat
        else:  # weighted_avg
            combined_mat = alpha * fc_mat + (1 - alpha) * phase_mat

        feats[i] = combined_mat[triu_idx]

    return feats


class fMRICombinedStaticClassifier(nn.Module):
    """
    Single-branch classifier for the element-wise-combined static FC
    feature -- no dual-branch fusion needed, since combination already
    happened at the matrix level in compute_combined_static_fc_features.
    A linear projection down to a small hidden size, then a compact MLP
    head classifies from there.
    """
    def __init__(self, n_pairs, proj_dim=16, hidden_dim=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(n_pairs, proj_dim), nn.GELU())
        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        h = self.proj(x)
        return self.classifier(h)


def train_one_fold_combined_static(X_train, y_train, X_val, y_val, n_pairs, epochs=60, lr=1e-4):
    model = fMRICombinedStaticClassifier(n_pairs=n_pairs).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_combined_static_classifier(X, y, real_lengths, skf, k_fc_select=1500,
                                         fc_method="pearson", shrinkage_alpha=0.1,
                                         TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5):
    """
    Per fold: computes the element-wise-combined static FC feature
    (single vector per subject), F-score-selects it (fold-safe: fit on
    training subjects only), then trains the single-branch classifier.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing element-wise combined FC ({combine_method}) ---")

        X_train_combined = compute_combined_static_fc_features(
            X_train_raw, rl_train, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )
        X_val_combined = compute_combined_static_fc_features(
            X_val_raw, rl_val, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )

        if k_fc_select is not None:
            top_idx, _ = select_top_pairs_2d(X_train_combined, y_train, k_fc_select)
            X_train_combined = X_train_combined[:, top_idx]
            X_val_combined = X_val_combined[:, top_idx]
            print(f"F-score selection: kept {len(top_idx)} ROI-pair features "
                  f"(from the combined matrix)")

        model, auc = train_one_fold_combined_static(
            X_train_combined, y_train, X_val_combined, y_val,
            n_pairs=X_train_combined.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_combined).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== Element-Wise Combined FC ({combine_method}): Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Element-Wise Combined FC Classifier (hadamard product)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
combined_matrix_results = evaluate_combined_static_classifier(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", combine_method="hadamard"
)


# =====================================================
# 8. INDIVIDUAL-FEATURE BASELINES + COMPARISON vs COMBINED FC
# =====================================================
# Runs the SAME single-branch classifier (fMRICombinedStaticClassifier /
# train_one_fold_combined_static) on each static feature INDIVIDUALLY --
# static FC alone, static phase-coherence alone -- so the combined
# (Hadamard) result above can be judged against its two ingredients
# rather than in isolation. Reuses the exact same model/training code;
# only the feature-extraction function fed in differs, which keeps the
# comparison controlled (same architecture, same training procedure,
# same folds -- only the input feature changes).
# =====================================================

def evaluate_single_static_feature(X, y, real_lengths, skf, feature_fn, feature_name,
                                    k_fc_select=1500, **feature_kwargs):
    """
    Generic single-branch evaluator: computes ONE static feature type
    per subject via feature_fn(X_raw, real_lengths, **feature_kwargs),
    F-score-selects it (fold-safe), then trains/evaluates the same
    single-branch classifier used for the combined-matrix feature.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing {feature_name} ---")

        X_train_feat = feature_fn(X_train_raw, rl_train, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, **feature_kwargs)

        if k_fc_select is not None:
            top_idx, _ = select_top_pairs_2d(X_train_feat, y_train, k_fc_select)
            X_train_feat = X_train_feat[:, top_idx]
            X_val_feat = X_val_feat[:, top_idx]
            print(f"F-score selection: kept {len(top_idx)} ROI-pair features")

        model, auc = train_one_fold_combined_static(
            X_train_feat, y_train, X_val_feat, y_val, n_pairs=X_train_feat.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_feat).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {feature_name}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training baseline: Static FC (Pearson) alone")
print("=" * 60)
static_fc_only_results = evaluate_single_static_feature(
    X, y, real_lengths, skf, compute_static_fc_features, "Static FC (Pearson) alone",
    k_fc_select=1500, fc_method="pearson"
)

print("\n" + "=" * 60)
print("Training baseline: Static Phase-Coherence alone")
print("=" * 60)
static_phase_only_results = evaluate_single_static_feature(
    X, y, real_lengths, skf, compute_static_phase_coherence_features, "Static Phase-Coherence alone",
    k_fc_select=1500
)

print("\n" + "=" * 60)
print("COMPARISON: Static FC alone vs Static Phase-Coherence alone vs Combined FC (hadamard)")
print("=" * 60)
print(f"{'Metric':<15}{'FC alone':<18}{'Phase alone':<18}{'Combined':<18}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(static_fc_only_results[metric]); a_std = np.std(static_fc_only_results[metric])
    b_mean = np.mean(static_phase_only_results[metric]); b_std = np.std(static_phase_only_results[metric])
    c_mean = np.mean(combined_matrix_results[metric]); c_std = np.std(combined_matrix_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}   {b_mean:.3f} ± {b_std:.3f}   {c_mean:.3f} ± {c_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from sklearn.cluster import KMeans
from scipy.signal import butter, filtfilt, hilbert
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Posterior-weighted PARTIAL correlation matrix, via a shrinkage-
    regularized precision matrix.

    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total posterior
    weight, which is often << T for less-visited states) comfortably
    exceeds n_rois. With whole-brain ROI counts (e.g. 200) and short
    scans (T ~ 78, further reduced by per-state weighting), the raw
    weighted covariance matrix is frequently singular or near-singular
    -- naive inversion would silently produce unstable, unusable
    partial correlations. To guard against this, the covariance
    diagonal is shrunk toward a scaled identity before inversion (a
    ridge-regularized precision matrix, in the same spirit as Ledoit-Wolf
    shrinkage / graphical lasso), which guarantees invertibility and
    stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain, low-occupancy-state
    settings; will likely be unstable). Higher values shrink more
    aggressively toward independence (a diagonal-only precision
    matrix), trading some sensitivity for stability. 0.1 is a
    reasonable starting point; increase if you still see extreme
    values or instability (e.g. try 0.2-0.3 for whole-brain/200-ROI
    settings), decrease if you've restricted to a small ROI subset
    (e.g. a single network with T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    fc_method: "pearson" (marginal correlation, weighted_corrcoef) or
               "partial" (shrinkage-regularized partial correlation,
               weighted_partial_corrcoef -- estimates DIRECT connectivity
               between ROI pairs, controlling for all other ROIs). See
               weighted_partial_corrcoef's docstring for why shrinkage
               is necessary, especially with whole-brain ROI counts.
    shrinkage_alpha: only used when fc_method="partial". See
               weighted_partial_corrcoef docstring for guidance.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    if fc_method not in ("pearson", "partial"):
        raise ValueError(f"Unknown fc_method: {fc_method!r} (expected 'pearson' or 'partial')")

    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                if fc_method == "partial":
                    fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
                else:
                    fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=1500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on the per-state FC vectors, computed using ONLY the training
    subjects of the current fold (fold-safe -- same discipline as the
    earlier MI-based ROI selection).

    X_train_states: (N_train, n_states, n_pairs) -- posterior-weighted
                    FC features, one vector of n_pairs per state.

    IMPORTANT: the F-score for each ROI-pair is computed SEPARATELY for
    every state (using that state's own FC values against the label),
    and these per-state F-scores are then AVERAGED ACROSS STATES to get
    one final importance score per pair. This is different from (and
    more faithful than) averaging the raw FC values across states
    first -- computing one F-score per state and then averaging the
    SCORES preserves a pair's discriminative signal even if it's only
    strong in one or two specific states, rather than letting it get
    diluted/cancelled out by blending raw signal across states before
    scoring at all.

    The same top-k pair indices (selected via this averaged score) are
    then applied identically to EVERY state's token -- i.e. the
    selected ROI-pair set is shared across all states, so the "state as
    token" structure (K tokens) is preserved; only n_pairs shrinks to k
    for every token.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_states, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER STATE, then stack
    per_state_f_scores = np.zeros((n_states, n_pairs), dtype=np.float64)
    for k_state in range(n_states):
        state_features = X_train_states[:, k_state, :]  # (N_train, n_pairs)
        f_scores_k, _ = f_classif(state_features, y_train)
        per_state_f_scores[k_state] = np.nan_to_num(f_scores_k, nan=0.0)

    # average the F-scores ACROSS STATES (not the raw features) -- this
    # is the key difference from a "blend features, then score once" approach
    avg_f_scores = per_state_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


def evaluate_hmm_state_transformer(X, y, skf, n_states=6, k_fc_select=None,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, OPTIONALLY reduce the n_pairs
    dimension via fold-safe F-score feature selection (fit on training
    subjects only, then applied identically to train and val), then
    train/evaluate the transformer.

    k_fc_select: if None, no feature selection is applied (uses all
                 n_pairs, original behavior). If an int, keeps only the
                 top-k ROI-pair features (by F-score) for every state's
                 token, shared across states.
    fc_method / shrinkage_alpha: passed through to
                 compute_hmm_state_fc_features -- see that function's
                 and weighted_partial_corrcoef's docstrings.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            # --- F-score selection using ONLY this fold's training subjects ---
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6, F-score feature selection)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
#
# k_fc_select: keeps only the top-k ROI-pair FC features (by F-score,
# computed fold-safe on training subjects only) per state token,
# instead of the full n_pairs (e.g. 19,900 for 200 ROIs). Reduces
# dimensionality/overfitting risk before the transformer sees the data.
# Set to None to disable and use all pairs (original behavior).
#
# fc_method="partial": estimates DIRECT ROI-to-ROI connectivity (i.e.
# controls for all other ROIs) rather than marginal Pearson
# correlation, which can't distinguish a direct connection from one
# mediated through a third region. Because this is a whole-brain
# (R=200) setting with short, posterior-weighted effective sample
# sizes per state, shrinkage_alpha=0.1 regularizes the precision-matrix
# inversion to keep it stable -- raise this (e.g. 0.2-0.3) if you
# still see unstable/extreme values, or set fc_method="pearson" to
# revert to the original marginal-correlation behavior.
hmm_state_results = evaluate_hmm_state_transformer(
    X, y, skf, n_states=6, k_fc_select=1500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 5. LEiDA (LEADING EIGENVECTOR DYNAMICS ANALYSIS) STATE-FC TRANSFORMER
# =====================================================
# An alternative to the HMM for defining "states": instead of a
# generative model fit on raw ROI signal, phase-locking states are
# derived directly from signal-processing (band-pass filter -> Hilbert
# phase -> instantaneous phase-coherence -> leading eigenvector per
# timepoint), then clustered with k-means. This gives an independent,
# assumption-different way to define the K state tokens that plug into
# the SAME downstream FC-feature + transformer pipeline as section 4,
# enabling a direct HMM-vs-LEiDA comparison.
#
# READ BEFORE TRUSTING RESULTS -- two dataset-specific caveats:
#
# 1. TR (repetition time) MUST be correct for the band-pass filter to
#    mean what you think it means. This pipeline resamples every
#    subject to a FIXED NUMBER of timepoints (TARGET_TIME), not a fixed
#    PHYSICAL DURATION -- so if your included ABIDE sites don't share
#    (approximately) the same TR, "0.04-0.07 Hz" refers to a different
#    physical frequency band per subject. Check the TR(s) for your
#    included sites (via ABIDE phenotypic metadata) before trusting
#    this. Either set SITE_FILTER to a single-TR site, or resample to a
#    common physical sampling rate upstream of TARGET_TIME truncation,
#    or treat results here as exploratory only.
# 2. Padded (originally-shorter-than-TARGET_TIME) subjects have zeros
#    appended after their real_length. Those padded timepoints are
#    EXCLUDED ENTIRELY from filtering/phase/eigenvector computation
#    (using real_lengths) rather than being band-pass filtered as if
#    they were real signal, which would otherwise create a sharp
#    real-signal-to-zero edge artifact right at the pad boundary.
# =====================================================

# --- placeholder: SET THIS to your actual (site-consistent) TR in seconds ---
LEIDA_TR = 2.0          # <-- VERIFY against ABIDE phenotypic metadata for your sites
LEIDA_BAND = (0.04, 0.07)  # Hz; standard fMRI LEiDA band, adjust if needed


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_leading_eigenvectors(X_raw, real_lengths, TR, band=LEIDA_BAND, order=4):
    """
    Per subject: band-pass filters ROI signals over ONLY the subject's
    real (non-padded) timepoints, Hilbert-transforms to get
    instantaneous phase, builds the instantaneous phase-coherence
    matrix cos(phase_i - phase_j) at every real timepoint, and takes
    its leading eigenvector (largest-eigenvalue eigenvector) -- a
    compact R-dim summary of the dominant coherence pattern at that
    instant. Sign is fixed (flipped so the eigenvector's mean is
    negative) purely for k-means cluster stability, since eigenvectors
    are only defined up to sign.

    Returns: eigenvectors (N, T, n_rois) float32, with rows beyond each
    subject's real_length left as exactly zero (never used as a
    meaningful state assignment downstream).
    """
    N, T, R = X_raw.shape
    fs = 1.0 / TR
    eigvecs = np.zeros((N, T, R), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        # vectorized instantaneous phase-coherence + leading eigenvector
        # across all L real timepoints at once (fast batched eigh)
        diff = phase[:, :, None] - phase[:, None, :]      # (L, R, R)
        coh = np.cos(diff)                                 # (L, R, R)
        evals, evecs = np.linalg.eigh(coh)                 # ascending eigvals
        v = evecs[:, :, -1]                                # (L, R) leading eigenvector per timepoint
        flip = v.mean(axis=1) > 0
        v[flip] *= -1

        eigvecs[i, :L] = v.astype(np.float32)

    return eigvecs


def compute_leida_state_fc_features(X_train_raw, X_val_raw, real_lengths_train, real_lengths_val,
                                     n_states=6, TR=LEIDA_TR, band=LEIDA_BAND,
                                     fc_method="pearson", shrinkage_alpha=0.1,
                                     random_state=42, verbose=True):
    """
    LEiDA-style dynamic FC features, fold-safe (mirrors
    compute_hmm_state_fc_features's leakage discipline exactly, but
    defines states via phase clustering instead of an HMM):

      1. Leading eigenvector of instantaneous phase-coherence, per
         real (non-padded) timepoint (see compute_leading_eigenvectors).
      2. K-MEANS fit on TRAINING SUBJECTS' eigenvectors ONLY (pooled
         across all their real timepoints) -> n_states phase-locking
         states.
      3. Every timepoint (train AND val) hard-assigned to its nearest
         centroid using that SAME fitted k-means (never refit on val).
      4. Per state, per subject: FC computed via
         weighted_corrcoef/weighted_partial_corrcoef, weighting each
         timepoint by its one-hot state membership (i.e. FC computed
         only from timepoints assigned to that state). Padded
         timepoints get zero weight in every state.

    Returns: (X_train_states, X_val_states), each (N, n_states, n_pairs)
             -- same shape/convention as compute_hmm_state_fc_features,
             so this plugs directly into select_top_fc_pairs_by_fscore
             and the existing transformer training code.
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    if verbose:
        print(f"Computing LEiDA leading eigenvectors (TR={TR}s, band={band} Hz)...")
    train_eigvecs = compute_leading_eigenvectors(X_train_raw, real_lengths_train, TR, band)
    val_eigvecs = compute_leading_eigenvectors(X_val_raw, real_lengths_val, TR, band)

    # --- fit k-means on TRAINING subjects' real (non-padded) eigenvectors only ---
    train_mask = np.zeros((N_train, T), dtype=bool)
    for i in range(N_train):
        train_mask[i, :int(real_lengths_train[i])] = True
    train_pool = train_eigvecs[train_mask]  # (sum_real_lengths_train, R)

    kmeans = KMeans(n_clusters=n_states, random_state=random_state, n_init=10)
    kmeans.fit(train_pool)

    if verbose:
        print(f"K-means fit on {train_pool.shape[0]} training timepoints "
              f"({n_states} phase-locking states)")

    def assign_states(eigvecs, real_lengths):
        N = eigvecs.shape[0]
        weights = np.zeros((N, T, n_states), dtype=np.float32)
        for i in range(N):
            L = int(real_lengths[i])
            if L == 0:
                continue
            labels = kmeans.predict(eigvecs[i, :L])  # (L,)
            for k in range(n_states):
                weights[i, :L, k] = (labels == k).astype(np.float32)
        return weights

    train_weights = assign_states(train_eigvecs, real_lengths_train)
    val_weights = assign_states(val_eigvecs, real_lengths_val)

    def build_fc_features(X_raw, weights):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            for k in range(n_states):
                w = weights[i, :, k]
                if w.sum() < 1.0:
                    low_occupancy_flags += 1
                if fc_method == "partial":
                    fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
                else:
                    fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_fc_features(X_train_raw, train_weights)
    X_val_states, low_occ_val = build_fc_features(X_val_raw, val_weights)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (zero timepoints assigned): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


def evaluate_leida_state_transformer(X, y, real_lengths, skf, n_states=6, k_fc_select=None,
                                      fc_method="pearson", shrinkage_alpha=0.1,
                                      TR=LEIDA_TR, band=LEIDA_BAND):
    """
    Same per-fold structure as evaluate_hmm_state_transformer, but
    states come from compute_leida_state_fc_features instead of the
    HMM. Reuses select_top_fc_pairs_by_fscore and train_one_fold_hmm_state
    unchanged -- both are agnostic to how the (n_states, n_pairs)
    features were produced.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: LEiDA state assignment (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_leida_state_fc_features(
            X_train_raw, X_val_raw, rl_train, rl_val,
            n_states=n_states, TR=TR, band=band,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== LEiDA Phase-Locking-State Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training LEiDA Phase-Locking-State Transformer (n_states=6)")
print("=" * 60)
print(f"NOTE: using TR={LEIDA_TR}s, band={LEIDA_BAND} Hz -- verify TR matches "
      f"your included ABIDE site(s) before trusting these results.")
leida_state_results = evaluate_leida_state_transformer(
    X, y, real_lengths, skf, n_states=6, k_fc_select=1500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 6. QUICK COMPARISON -- HMM STATE-FC (section 4) vs LEiDA STATE-FC (section 5)
# =====================================================
print("\n" + "=" * 60)
print("COMPARISON: HMM State-FC (K tokens) vs LEiDA Phase-Locking-State-FC (K tokens)")
print("=" * 60)
print(f"{'Metric':<15}{'HMM State-FC':<20}{'LEiDA State-FC':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(leida_state_results[metric]); b_std = np.std(leida_state_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}")


# =====================================================
# 7. FINAL RESULTS SUMMARY
# =====================================================
print("\n" + "=" * 60)
print("FINAL RESULTS: HMM State-FC Transformer (K state tokens)")
print("=" * 60)
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    mean = np.mean(hmm_state_results[metric])
    std = np.std(hmm_state_results[metric])
    print(f"{metric:<15}: {mean:.3f} ± {std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from sklearn.cluster import KMeans
from scipy.signal import butter, filtfilt, hilbert
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Posterior-weighted PARTIAL correlation matrix, via a shrinkage-
    regularized precision matrix.

    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total posterior
    weight, which is often << T for less-visited states) comfortably
    exceeds n_rois. With whole-brain ROI counts (e.g. 200) and short
    scans (T ~ 78, further reduced by per-state weighting), the raw
    weighted covariance matrix is frequently singular or near-singular
    -- naive inversion would silently produce unstable, unusable
    partial correlations. To guard against this, the covariance
    diagonal is shrunk toward a scaled identity before inversion (a
    ridge-regularized precision matrix, in the same spirit as Ledoit-Wolf
    shrinkage / graphical lasso), which guarantees invertibility and
    stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain, low-occupancy-state
    settings; will likely be unstable). Higher values shrink more
    aggressively toward independence (a diagonal-only precision
    matrix), trading some sensitivity for stability. 0.1 is a
    reasonable starting point; increase if you still see extreme
    values or instability (e.g. try 0.2-0.3 for whole-brain/200-ROI
    settings), decrease if you've restricted to a small ROI subset
    (e.g. a single network with T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    fc_method: "pearson" (marginal correlation, weighted_corrcoef) or
               "partial" (shrinkage-regularized partial correlation,
               weighted_partial_corrcoef -- estimates DIRECT connectivity
               between ROI pairs, controlling for all other ROIs). See
               weighted_partial_corrcoef's docstring for why shrinkage
               is necessary, especially with whole-brain ROI counts.
    shrinkage_alpha: only used when fc_method="partial". See
               weighted_partial_corrcoef docstring for guidance.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    if fc_method not in ("pearson", "partial"):
        raise ValueError(f"Unknown fc_method: {fc_method!r} (expected 'pearson' or 'partial')")

    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                if fc_method == "partial":
                    fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
                else:
                    fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=1500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on the per-state FC vectors, computed using ONLY the training
    subjects of the current fold (fold-safe -- same discipline as the
    earlier MI-based ROI selection).

    X_train_states: (N_train, n_states, n_pairs) -- posterior-weighted
                    FC features, one vector of n_pairs per state.

    IMPORTANT: the F-score for each ROI-pair is computed SEPARATELY for
    every state (using that state's own FC values against the label),
    and these per-state F-scores are then AVERAGED ACROSS STATES to get
    one final importance score per pair. This is different from (and
    more faithful than) averaging the raw FC values across states
    first -- computing one F-score per state and then averaging the
    SCORES preserves a pair's discriminative signal even if it's only
    strong in one or two specific states, rather than letting it get
    diluted/cancelled out by blending raw signal across states before
    scoring at all.

    The same top-k pair indices (selected via this averaged score) are
    then applied identically to EVERY state's token -- i.e. the
    selected ROI-pair set is shared across all states, so the "state as
    token" structure (K tokens) is preserved; only n_pairs shrinks to k
    for every token.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_states, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER STATE, then stack
    per_state_f_scores = np.zeros((n_states, n_pairs), dtype=np.float64)
    for k_state in range(n_states):
        state_features = X_train_states[:, k_state, :]  # (N_train, n_pairs)
        f_scores_k, _ = f_classif(state_features, y_train)
        per_state_f_scores[k_state] = np.nan_to_num(f_scores_k, nan=0.0)

    # average the F-scores ACROSS STATES (not the raw features) -- this
    # is the key difference from a "blend features, then score once" approach
    avg_f_scores = per_state_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


def evaluate_hmm_state_transformer(X, y, skf, n_states=6, k_fc_select=None,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, OPTIONALLY reduce the n_pairs
    dimension via fold-safe F-score feature selection (fit on training
    subjects only, then applied identically to train and val), then
    train/evaluate the transformer.

    k_fc_select: if None, no feature selection is applied (uses all
                 n_pairs, original behavior). If an int, keeps only the
                 top-k ROI-pair features (by F-score) for every state's
                 token, shared across states.
    fc_method / shrinkage_alpha: passed through to
                 compute_hmm_state_fc_features -- see that function's
                 and weighted_partial_corrcoef's docstrings.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            # --- F-score selection using ONLY this fold's training subjects ---
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6, F-score feature selection)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
#
# k_fc_select: keeps only the top-k ROI-pair FC features (by F-score,
# computed fold-safe on training subjects only) per state token,
# instead of the full n_pairs (e.g. 19,900 for 200 ROIs). Reduces
# dimensionality/overfitting risk before the transformer sees the data.
# Set to None to disable and use all pairs (original behavior).
#
# fc_method="partial": estimates DIRECT ROI-to-ROI connectivity (i.e.
# controls for all other ROIs) rather than marginal Pearson
# correlation, which can't distinguish a direct connection from one
# mediated through a third region. Because this is a whole-brain
# (R=200) setting with short, posterior-weighted effective sample
# sizes per state, shrinkage_alpha=0.1 regularizes the precision-matrix
# inversion to keep it stable -- raise this (e.g. 0.2-0.3) if you
# still see unstable/extreme values, or set fc_method="pearson" to
# revert to the original marginal-correlation behavior.
hmm_state_results = evaluate_hmm_state_transformer(
    X, y, skf, n_states=6, k_fc_select=1500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 5. LEiDA (LEADING EIGENVECTOR DYNAMICS ANALYSIS) STATE-FC TRANSFORMER
# =====================================================
# An alternative to the HMM for defining "states": instead of a
# generative model fit on raw ROI signal, phase-locking states are
# derived directly from signal-processing (band-pass filter -> Hilbert
# phase -> instantaneous phase-coherence -> leading eigenvector per
# timepoint), then clustered with k-means. This gives an independent,
# assumption-different way to define the K state tokens that plug into
# the SAME downstream FC-feature + transformer pipeline as section 4,
# enabling a direct HMM-vs-LEiDA comparison.
#
# READ BEFORE TRUSTING RESULTS -- two dataset-specific caveats:
#
# 1. TR (repetition time) MUST be correct for the band-pass filter to
#    mean what you think it means. This pipeline resamples every
#    subject to a FIXED NUMBER of timepoints (TARGET_TIME), not a fixed
#    PHYSICAL DURATION -- so if your included ABIDE sites don't share
#    (approximately) the same TR, "0.04-0.07 Hz" refers to a different
#    physical frequency band per subject. Check the TR(s) for your
#    included sites (via ABIDE phenotypic metadata) before trusting
#    this. Either set SITE_FILTER to a single-TR site, or resample to a
#    common physical sampling rate upstream of TARGET_TIME truncation,
#    or treat results here as exploratory only.
# 2. Padded (originally-shorter-than-TARGET_TIME) subjects have zeros
#    appended after their real_length. Those padded timepoints are
#    EXCLUDED ENTIRELY from filtering/phase/eigenvector computation
#    (using real_lengths) rather than being band-pass filtered as if
#    they were real signal, which would otherwise create a sharp
#    real-signal-to-zero edge artifact right at the pad boundary.
# =====================================================

# --- placeholder: SET THIS to your actual (site-consistent) TR in seconds ---
LEIDA_TR = 2.0          # <-- VERIFY against ABIDE phenotypic metadata for your sites
LEIDA_BAND = (0.04, 0.07)  # Hz; standard fMRI LEiDA band, adjust if needed


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_leading_eigenvectors(X_raw, real_lengths, TR, band=LEIDA_BAND, order=4):
    """
    Per subject: band-pass filters ROI signals over ONLY the subject's
    real (non-padded) timepoints, Hilbert-transforms to get
    instantaneous phase, builds the instantaneous phase-coherence
    matrix cos(phase_i - phase_j) at every real timepoint, and takes
    its leading eigenvector (largest-eigenvalue eigenvector) -- a
    compact R-dim summary of the dominant coherence pattern at that
    instant. Sign is fixed (flipped so the eigenvector's mean is
    negative) purely for k-means cluster stability, since eigenvectors
    are only defined up to sign.

    Returns: eigenvectors (N, T, n_rois) float32, with rows beyond each
    subject's real_length left as exactly zero (never used as a
    meaningful state assignment downstream).
    """
    N, T, R = X_raw.shape
    fs = 1.0 / TR
    eigvecs = np.zeros((N, T, R), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        # vectorized instantaneous phase-coherence + leading eigenvector
        # across all L real timepoints at once (fast batched eigh)
        diff = phase[:, :, None] - phase[:, None, :]      # (L, R, R)
        coh = np.cos(diff)                                 # (L, R, R)
        evals, evecs = np.linalg.eigh(coh)                 # ascending eigvals
        v = evecs[:, :, -1]                                # (L, R) leading eigenvector per timepoint
        flip = v.mean(axis=1) > 0
        v[flip] *= -1

        eigvecs[i, :L] = v.astype(np.float32)

    return eigvecs


def compute_leida_state_fc_features(X_train_raw, X_val_raw, real_lengths_train, real_lengths_val,
                                     n_states=6, TR=LEIDA_TR, band=LEIDA_BAND,
                                     fc_method="pearson", shrinkage_alpha=0.1,
                                     random_state=42, verbose=True):
    """
    LEiDA-style dynamic FC features, fold-safe (mirrors
    compute_hmm_state_fc_features's leakage discipline exactly, but
    defines states via phase clustering instead of an HMM):

      1. Leading eigenvector of instantaneous phase-coherence, per
         real (non-padded) timepoint (see compute_leading_eigenvectors).
      2. K-MEANS fit on TRAINING SUBJECTS' eigenvectors ONLY (pooled
         across all their real timepoints) -> n_states phase-locking
         states.
      3. Every timepoint (train AND val) hard-assigned to its nearest
         centroid using that SAME fitted k-means (never refit on val).
      4. Per state, per subject: FC computed via
         weighted_corrcoef/weighted_partial_corrcoef, weighting each
         timepoint by its one-hot state membership (i.e. FC computed
         only from timepoints assigned to that state). Padded
         timepoints get zero weight in every state.

    Returns: (X_train_states, X_val_states), each (N, n_states, n_pairs)
             -- same shape/convention as compute_hmm_state_fc_features,
             so this plugs directly into select_top_fc_pairs_by_fscore
             and the existing transformer training code.
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    if verbose:
        print(f"Computing LEiDA leading eigenvectors (TR={TR}s, band={band} Hz)...")
    train_eigvecs = compute_leading_eigenvectors(X_train_raw, real_lengths_train, TR, band)
    val_eigvecs = compute_leading_eigenvectors(X_val_raw, real_lengths_val, TR, band)

    # --- fit k-means on TRAINING subjects' real (non-padded) eigenvectors only ---
    train_mask = np.zeros((N_train, T), dtype=bool)
    for i in range(N_train):
        train_mask[i, :int(real_lengths_train[i])] = True
    train_pool = train_eigvecs[train_mask]  # (sum_real_lengths_train, R)

    kmeans = KMeans(n_clusters=n_states, random_state=random_state, n_init=10)
    kmeans.fit(train_pool)

    if verbose:
        print(f"K-means fit on {train_pool.shape[0]} training timepoints "
              f"({n_states} phase-locking states)")

    def assign_states(eigvecs, real_lengths):
        N = eigvecs.shape[0]
        weights = np.zeros((N, T, n_states), dtype=np.float32)
        for i in range(N):
            L = int(real_lengths[i])
            if L == 0:
                continue
            labels = kmeans.predict(eigvecs[i, :L])  # (L,)
            for k in range(n_states):
                weights[i, :L, k] = (labels == k).astype(np.float32)
        return weights

    train_weights = assign_states(train_eigvecs, real_lengths_train)
    val_weights = assign_states(val_eigvecs, real_lengths_val)

    def build_fc_features(X_raw, weights):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            for k in range(n_states):
                w = weights[i, :, k]
                if w.sum() < 1.0:
                    low_occupancy_flags += 1
                if fc_method == "partial":
                    fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
                else:
                    fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_fc_features(X_train_raw, train_weights)
    X_val_states, low_occ_val = build_fc_features(X_val_raw, val_weights)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (zero timepoints assigned): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


def evaluate_leida_state_transformer(X, y, real_lengths, skf, n_states=6, k_fc_select=None,
                                      fc_method="pearson", shrinkage_alpha=0.1,
                                      TR=LEIDA_TR, band=LEIDA_BAND):
    """
    Same per-fold structure as evaluate_hmm_state_transformer, but
    states come from compute_leida_state_fc_features instead of the
    HMM. Reuses select_top_fc_pairs_by_fscore and train_one_fold_hmm_state
    unchanged -- both are agnostic to how the (n_states, n_pairs)
    features were produced.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: LEiDA state assignment (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_leida_state_fc_features(
            X_train_raw, X_val_raw, rl_train, rl_val,
            n_states=n_states, TR=TR, band=band,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== LEiDA Phase-Locking-State Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training LEiDA Phase-Locking-State Transformer (n_states=6)")
print("=" * 60)
print(f"NOTE: using TR={LEIDA_TR}s, band={LEIDA_BAND} Hz -- verify TR matches "
      f"your included ABIDE site(s) before trusting these results.")
leida_state_results = evaluate_leida_state_transformer(
    X, y, real_lengths, skf, n_states=6, k_fc_select=1500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 6. QUICK COMPARISON -- HMM STATE-FC (section 4) vs LEiDA STATE-FC (section 5)
# =====================================================
print("\n" + "=" * 60)
print("COMPARISON: HMM State-FC (K tokens) vs LEiDA Phase-Locking-State-FC (K tokens)")
print("=" * 60)
print(f"{'Metric':<15}{'HMM State-FC':<20}{'LEiDA State-FC':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(leida_state_results[metric]); b_std = np.std(leida_state_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}")


# =====================================================
# 7. COMBINED HMM + LEiDA TOKEN-FUSION TRANSFORMER
# =====================================================
# Rather than treating HMM-defined states and LEiDA-defined states as
# two competing pipelines (section 6's comparison), this section fuses
# them into ONE transformer: HMM's K tokens and LEiDA's K tokens are
# concatenated into a single sequence of 2K tokens, each projected into
# the shared embedding space by its OWN linear layer (since an "HMM
# state" and a "LEiDA state" are unrelated vocabularies -- there's no
# reason to assume "HMM state 3" corresponds to "LEiDA state 3", so
# they must NOT share a projection or a state-identity embedding).
# A learned SOURCE embedding additionally tells the model which tokens
# came from which method. Self-attention across all 2K tokens then lets
# the model learn interactions BETWEEN the two state-definition methods
# directly (e.g. "HMM state 2 combined with LEiDA state 5"), which
# neither an early-fusion (concatenate raw FC vectors, forcing a false
# state-to-state correspondence) nor a late-fusion/ensemble (average
# two independent models' predictions, no cross-interaction at all)
# approach can do.
#
# Both branches remain fully fold-safe and independent of each other:
# the HMM is fit on training subjects only (section 4's function), the
# LEiDA k-means is fit on training subjects only (section 5's
# function), and F-score feature selection is run SEPARATELY for each
# branch (their selected ROI-pair indices need not match -- each
# branch's "n_pairs" columns mean something different after selection).
# =====================================================

class fMRITransformerCombined(nn.Module):
    """
    Token axis = HMM states (K_hmm of them) concatenated with LEiDA
    states (K_leida of them) -> 2K tokens total (when K_hmm == K_leida).
    Each source gets its OWN input projection and its OWN per-state
    identity embedding (they are unrelated vocabularies), plus a shared
    source-type embedding so the model can tell the two apart.
    """
    def __init__(self, n_pairs_hmm, n_pairs_leida, n_states_hmm, n_states_leida,
                 d_model=16, n_heads=2, n_layers=1, dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.hmm_proj = nn.Linear(n_pairs_hmm, d_model)
        self.leida_proj = nn.Linear(n_pairs_leida, d_model)

        self.hmm_state_embedding = nn.Parameter(torch.randn(1, n_states_hmm, d_model) * 0.02)
        self.leida_state_embedding = nn.Parameter(torch.randn(1, n_states_leida, d_model) * 0.02)

        # source-type embedding: one vector per source (HMM vs LEiDA),
        # broadcast across that source's states and the batch dimension
        self.source_embedding = nn.Parameter(torch.randn(2, 1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x_hmm, x_leida, return_attn=False):
        # x_hmm: (B, n_states_hmm, n_pairs_hmm)
        # x_leida: (B, n_states_leida, n_pairs_leida)
        h_hmm = self.hmm_proj(x_hmm) + self.hmm_state_embedding + self.source_embedding[0]
        h_leida = self.leida_proj(x_leida) + self.leida_state_embedding + self.source_embedding[1]

        h = torch.cat([h_hmm, h_leida], dim=1)  # (B, n_states_hmm + n_states_leida, d_model)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_combined(X_train_hmm, X_train_leida, y_train,
                             X_val_hmm, X_val_leida, y_val,
                             n_pairs_hmm, n_pairs_leida, n_states_hmm, n_states_leida,
                             epochs=60, lr=1e-4):
    model = fMRITransformerCombined(
        n_pairs_hmm=n_pairs_hmm, n_pairs_leida=n_pairs_leida,
        n_states_hmm=n_states_hmm, n_states_leida=n_states_leida
    ).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr_hmm = torch.tensor(X_train_hmm).to(device)
    Xtr_leida = torch.tensor(X_train_leida).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva_hmm = torch.tensor(X_val_hmm).to(device)
    Xva_leida = torch.tensor(X_val_leida).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr_hmm, Xtr_leida)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_hmm, Xva_leida)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_combined_transformer(X, y, real_lengths, skf, n_states=6, k_fc_select=1500,
                                   fc_method="partial", shrinkage_alpha=0.1,
                                   TR=LEIDA_TR, band=LEIDA_BAND):
    """
    Per fold: computes BOTH HMM state-FC features (section 4's function)
    and LEiDA state-FC features (section 5's function) -- each fit
    fold-safe on that fold's training subjects only -- then runs F-score
    selection SEPARATELY on each branch (their selected pair indices are
    independent; this is fine since each branch's projection layer is
    also independent), then trains the combined token-fusion transformer.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing HMM + LEiDA state features ---")

        X_train_hmm, X_val_hmm = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )
        X_train_leida, X_val_leida = compute_leida_state_fc_features(
            X_train_raw, X_val_raw, rl_train, rl_val, n_states=n_states,
            TR=TR, band=band, fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            hmm_pair_idx, _ = select_top_fc_pairs_by_fscore(X_train_hmm, y_train, k=k_fc_select)
            X_train_hmm = X_train_hmm[:, :, hmm_pair_idx]
            X_val_hmm = X_val_hmm[:, :, hmm_pair_idx]

            leida_pair_idx, _ = select_top_fc_pairs_by_fscore(X_train_leida, y_train, k=k_fc_select)
            X_train_leida = X_train_leida[:, :, leida_pair_idx]
            X_val_leida = X_val_leida[:, :, leida_pair_idx]

            print(f"F-score selection: HMM kept {len(hmm_pair_idx)}, "
                  f"LEiDA kept {len(leida_pair_idx)} ROI-pair features "
                  f"(selected independently per branch)")

        model, auc = train_one_fold_combined(
            X_train_hmm, X_train_leida, y_train,
            X_val_hmm, X_val_leida, y_val,
            n_pairs_hmm=X_train_hmm.shape[2], n_pairs_leida=X_train_leida.shape[2],
            n_states_hmm=n_states, n_states_leida=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_hmm_t = torch.tensor(X_val_hmm).to(device)
            Xva_leida_t = torch.tensor(X_val_leida).to(device)
            val_logits = model(Xva_hmm_t, Xva_leida_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Combined HMM+LEiDA Token-Fusion Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Combined HMM+LEiDA Token-Fusion Transformer (n_states=6 each)")
print("=" * 60)
combined_results = evaluate_combined_transformer(
    X, y, real_lengths, skf, n_states=6, k_fc_select=1500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 8. FINAL RESULTS SUMMARY -- ALL THREE APPROACHES
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: HMM-only vs LEiDA-only vs Combined Token-Fusion")
print("=" * 60)
print(f"{'Metric':<15}{'HMM only':<20}{'LEiDA only':<20}{'Combined':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(leida_state_results[metric]); b_std = np.std(leida_state_results[metric])
    c_mean = np.mean(combined_results[metric]); c_std = np.std(combined_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}      {c_mean:.3f} ± {c_std:.3f}")


# =====================================================
# 9. STATIC FC + STATIC PHASE-COHERENCE FUSION (non-dynamic baseline)
# =====================================================
# This section deliberately avoids ALL per-state/dynamic structure --
# no HMM, no LEiDA clustering. Each subject is reduced to exactly TWO
# static (whole-scan) feature vectors:
#   1. static FC: ordinary Pearson/partial-correlation FC, computed
#      once over the ENTIRE (real, non-padded) scan -- reuses
#      weighted_corrcoef/weighted_partial_corrcoef with a weight vector
#      of all-ones over real timepoints (i.e. the "one giant state =
#      the whole scan" special case).
#   2. static phase-coherence FC: the same band-pass + Hilbert-phase
#      front-end as LEiDA (section 5), but instead of clustering
#      timepoints into discrete states, the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is simply AVERAGED
#      across every real timepoint into one (R, R) matrix. No k-means,
#      no per-fold fitting needed for this step -- it's a deterministic
#      per-subject computation. This is the direct phase-based analogue
#      of static Pearson FC.
#
# These two static vectors are fused via a minimal 2-token transformer
# (fMRIStaticFusionTransformer): one token per branch, each with its
# own linear projection and its own learned source embedding, self-
# attention over just those 2 tokens. This section is independent of
# sections 4-8 (no HMM/LEiDA dynamic state machinery is used) and can
# be run standalone as a simpler, non-dynamic baseline.
# =====================================================

def compute_static_fc_features(X_raw, real_lengths, fc_method="pearson", shrinkage_alpha=0.1):
    """
    Static whole-scan FC per subject: Pearson or shrinkage-regularized
    partial correlation, computed once over ALL real (non-padded)
    timepoints (weight = 1 for real timepoints, 0 for padding) -- no
    per-state/dynamic structure at all.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc = weighted_corrcoef(X_raw[i], w)
        feats[i] = fc[triu_idx]

    return feats


def compute_static_phase_coherence_features(X_raw, real_lengths, TR=LEIDA_TR, band=LEIDA_BAND, order=4):
    """
    Static grand-average phase-coherence FC per subject: band-pass
    filter + Hilbert transform to get instantaneous phase (same
    front-end as LEiDA), compute the instantaneous phase-coherence
    matrix at EVERY real timepoint, then AVERAGE these matrices across
    all real timepoints into one (R, R) matrix -- no clustering step.
    This collapses the whole scan's phase relationships into one
    static summary, the phase-based analogue of static Pearson FC.

    Returns: (N, n_pairs) float32

    Same TR caveat as section 5's LEiDA code applies -- TR must be
    correct/consistent across included sites for the band-pass filter
    to be meaningful.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
        coh = np.cos(diff)                              # (L, R, R)
        grand_avg_coh = coh.mean(axis=0)                # (R, R) -- average over time

        feats[i] = grand_avg_coh[triu_idx]

    return feats


def select_top_pairs_2d(X_train_2d, y_train, k):
    """
    Thin wrapper around select_top_fc_pairs_by_fscore for the static
    (single-vector-per-subject, no state axis) case -- reshapes to
    (N, 1, n_pairs) so the existing per-state-then-average F-score
    logic still applies (trivially, since there's only one "state").
    """
    top_idx, scores = select_top_fc_pairs_by_fscore(X_train_2d[:, None, :], y_train, k=k)
    return top_idx, scores


class fMRIStaticFusionTransformer(nn.Module):
    """
    Fuses two STATIC (one token each -- no per-state/dynamic structure)
    representations per subject: a static FC vector and a static
    phase-coherence vector. Each gets its own linear projection and its
    own learned source embedding (marking which branch a token came
    from); self-attention over just these 2 tokens lets the model learn
    how to weigh/combine the two static views. Written this way (rather
    than a plain concatenation + MLP) so it can later be extended with
    additional per-state/dynamic tokens (e.g. HMM or LEiDA states from
    sections 4-7) just by concatenating more tokens, without an
    architecture change.
    """
    def __init__(self, n_pairs_fc, n_pairs_phase, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.fc_proj = nn.Linear(n_pairs_fc, d_model)
        self.phase_proj = nn.Linear(n_pairs_phase, d_model)
        # one learned embedding per token (2 tokens total: static FC, static phase)
        self.source_embedding = nn.Parameter(torch.randn(1, 2, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x_fc, x_phase, return_attn=False):
        # x_fc: (B, n_pairs_fc), x_phase: (B, n_pairs_phase) -- single
        # vectors per subject, not sequences
        h_fc = self.fc_proj(x_fc).unsqueeze(1)           # (B, 1, d_model)
        h_phase = self.phase_proj(x_phase).unsqueeze(1)  # (B, 1, d_model)
        h = torch.cat([h_fc, h_phase], dim=1)            # (B, 2, d_model)
        h = h + self.source_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_static_fusion(X_train_fc, X_train_phase, y_train,
                                  X_val_fc, X_val_phase, y_val,
                                  n_pairs_fc, n_pairs_phase, epochs=60, lr=1e-4):
    model = fMRIStaticFusionTransformer(n_pairs_fc=n_pairs_fc, n_pairs_phase=n_pairs_phase).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr_fc = torch.tensor(X_train_fc).to(device)
    Xtr_phase = torch.tensor(X_train_phase).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva_fc = torch.tensor(X_val_fc).to(device)
    Xva_phase = torch.tensor(X_val_phase).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr_fc, Xtr_phase)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_fc, Xva_phase)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_static_fusion_transformer(X, y, real_lengths, skf, k_fc_select=1500,
                                        fc_method="partial", shrinkage_alpha=0.1,
                                        TR=LEIDA_TR, band=LEIDA_BAND):
    """
    Per fold: computes static FC and static phase-coherence features
    (both deterministic per-subject, no fold-fitting needed for the
    features themselves), runs F-score selection SEPARATELY on each
    branch (fold-safe: fit on training subjects only), then trains the
    2-token static fusion transformer.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing static FC + static phase-coherence features ---")

        X_train_fc = compute_static_fc_features(X_train_raw, rl_train, fc_method, shrinkage_alpha)
        X_val_fc = compute_static_fc_features(X_val_raw, rl_val, fc_method, shrinkage_alpha)

        X_train_phase = compute_static_phase_coherence_features(X_train_raw, rl_train, TR, band)
        X_val_phase = compute_static_phase_coherence_features(X_val_raw, rl_val, TR, band)

        if k_fc_select is not None:
            fc_idx, _ = select_top_pairs_2d(X_train_fc, y_train, k_fc_select)
            X_train_fc = X_train_fc[:, fc_idx]
            X_val_fc = X_val_fc[:, fc_idx]

            phase_idx, _ = select_top_pairs_2d(X_train_phase, y_train, k_fc_select)
            X_train_phase = X_train_phase[:, phase_idx]
            X_val_phase = X_val_phase[:, phase_idx]

            print(f"F-score selection: static-FC kept {len(fc_idx)}, "
                  f"static-phase kept {len(phase_idx)} ROI-pair features "
                  f"(selected independently per branch)")

        model, auc = train_one_fold_static_fusion(
            X_train_fc, X_train_phase, y_train, X_val_fc, X_val_phase, y_val,
            n_pairs_fc=X_train_fc.shape[1], n_pairs_phase=X_train_phase.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_fc_t = torch.tensor(X_val_fc).to(device)
            Xva_phase_t = torch.tensor(X_val_phase).to(device)
            val_logits = model(Xva_fc_t, Xva_phase_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Static FC + Static Phase-Coherence Fusion: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Static FC + Static Phase-Coherence Fusion Transformer")
print("=" * 60)
print(f"NOTE: using TR={LEIDA_TR}s, band={LEIDA_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
static_fusion_results = evaluate_static_fusion_transformer(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", shrinkage_alpha=0.1
)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Static FC + Static Phase-Coherence Fusion (learned gate)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from scipy.signal import butter, filtfilt, hilbert

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. WEIGHTED FC UTILITIES (Pearson + shrinkage-regularized partial)
# =====================================================
# General-purpose weighted functional-connectivity estimators. A weight
# vector of all-ones over a subject's real (non-padded) timepoints
# gives ordinary whole-scan static FC -- that's how these are used
# below (section 6). They also generalize to arbitrary non-negative
# per-timepoint weights (e.g. posterior probabilities, cluster
# membership masks) if you later reintroduce any dynamic/state-based
# weighting scheme.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero, returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Weighted PARTIAL correlation matrix, via a shrinkage-regularized
    precision matrix.

    data: (T, n_rois), weights: (T,) non-negative.

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total weight)
    comfortably exceeds n_rois. With whole-brain ROI counts (e.g. 200)
    and short scans (T ~ 78), the raw weighted covariance matrix can be
    singular or near-singular -- naive inversion would silently produce
    unstable, unusable partial correlations. To guard against this, the
    covariance diagonal is shrunk toward a scaled identity before
    inversion (a ridge-regularized precision matrix, in the same spirit
    as Ledoit-Wolf shrinkage / graphical lasso), which guarantees
    invertibility and stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain settings; will likely be
    unstable). Higher values shrink more aggressively toward
    independence (a diagonal-only precision matrix), trading some
    sensitivity for stability. 0.1 is a reasonable starting point;
    increase if you still see extreme values or instability (e.g. try
    0.2-0.3 for whole-brain/200-ROI settings), decrease if you've
    restricted to a small ROI subset (e.g. a single network with
    T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


# =====================================================
# 5. F-SCORE FEATURE SELECTION UTILITY
# =====================================================

def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=1500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on FC vectors, computed using ONLY the training subjects of the
    current fold (fold-safe).

    X_train_states: (N_train, n_groups, n_pairs) -- one FC vector of
                    n_pairs per "group" (e.g. a branch/source; for the
                    static case below, n_groups=1).

    The F-score for each ROI-pair is computed SEPARATELY for every
    group, and these per-group F-scores are then AVERAGED ACROSS GROUPS
    to get one final importance score per pair. When n_groups=1 (the
    static case), this trivially reduces to plain per-pair F-scoring.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_groups, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER GROUP, then stack
    per_group_f_scores = np.zeros((n_groups, n_pairs), dtype=np.float64)
    for g in range(n_groups):
        group_features = X_train_states[:, g, :]  # (N_train, n_pairs)
        f_scores_g, _ = f_classif(group_features, y_train)
        per_group_f_scores[g] = np.nan_to_num(f_scores_g, nan=0.0)

    # average the F-scores ACROSS GROUPS (not the raw features)
    avg_f_scores = per_group_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


# =====================================================
# 6. STATIC FC + STATIC PHASE-COHERENCE FUSION (learned gate)
# =====================================================
# Each subject is reduced to exactly TWO static (whole-scan) feature
# vectors -- no HMM, no clustering, no per-state/dynamic structure:
#   1. static FC: ordinary Pearson/partial-correlation FC, computed
#      once over the ENTIRE (real, non-padded) scan -- reuses
#      weighted_corrcoef/weighted_partial_corrcoef with a weight vector
#      of all-ones over real timepoints (i.e. the "one giant state =
#      the whole scan" special case).
#   2. static phase-coherence FC: band-pass filter + Hilbert transform
#      to get instantaneous phase, then the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is simply AVERAGED
#      across every real timepoint into one (R, R) matrix. No k-means,
#      no per-fold fitting needed for this step -- it's a deterministic
#      per-subject computation. This is the direct phase-based analogue
#      of static Pearson FC.
#
# These two static vectors are fused via a learned GATE
# (fMRIStaticFusionGated): each branch gets its own linear projection,
# then a small gate network (one linear + sigmoid layer, looking at
# both projections together) decides how much to trust each branch,
# per subject -- h = g*h_fc + (1-g)*h_phase. This is directly
# interpretable (inspect g to see which view the model relied on for a
# given subject) and uses far fewer parameters than a self-attention
# block, since there's no multi-head Q/K/V machinery, just one small
# linear layer producing the blend weight.
# =====================================================

# --- placeholder: SET THIS to your actual (site-consistent) TR in seconds ---
PHASE_TR = 2.0           # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_features(X_raw, real_lengths, fc_method="pearson", shrinkage_alpha=0.1):
    """
    Static whole-scan FC per subject: Pearson or shrinkage-regularized
    partial correlation, computed once over ALL real (non-padded)
    timepoints (weight = 1 for real timepoints, 0 for padding) -- no
    per-state/dynamic structure at all.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc = weighted_corrcoef(X_raw[i], w)
        feats[i] = fc[triu_idx]

    return feats


def compute_static_phase_coherence_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND, order=4):
    """
    Static grand-average phase-coherence FC per subject: band-pass
    filter + Hilbert transform to get instantaneous phase, compute the
    instantaneous phase-coherence matrix at EVERY real timepoint, then
    AVERAGE these matrices across all real timepoints into one (R, R)
    matrix -- no clustering step. This collapses the whole scan's phase
    relationships into one static summary, the phase-based analogue of
    static Pearson FC.

    Returns: (N, n_pairs) float32

    IMPORTANT TR CAVEAT: this dataset resamples every subject to a
    FIXED NUMBER of timepoints (TARGET_TIME), not a fixed PHYSICAL
    DURATION -- so if your included ABIDE sites don't share
    (approximately) the same TR, the band here refers to a different
    physical frequency band per subject. Verify TR consistency (via
    ABIDE phenotypic metadata) before trusting these features, or
    restrict SITE_FILTER to a single-TR site.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
        coh = np.cos(diff)                              # (L, R, R)
        grand_avg_coh = coh.mean(axis=0)                # (R, R) -- average over time

        feats[i] = grand_avg_coh[triu_idx]

    return feats


def select_top_pairs_2d(X_train_2d, y_train, k):
    """
    Thin wrapper around select_top_fc_pairs_by_fscore for the static
    (single-vector-per-subject, no group axis) case -- reshapes to
    (N, 1, n_pairs) so the existing per-group-then-average F-score
    logic still applies (trivially, since there's only one group).
    """
    top_idx, scores = select_top_fc_pairs_by_fscore(X_train_2d[:, None, :], y_train, k=k)
    return top_idx, scores


class fMRIStaticFusionGated(nn.Module):
    """
    Fuses two STATIC representations per subject via a learned GATE
    rather than concatenation or attention. Each branch (static FC,
    static phase-coherence) gets its own linear projection into a
    shared d-dimensional space; a small gate network then looks at
    both projected vectors together and outputs g in (0, 1) --
    per-dimension by default (vector gate) -- which blends them:

        h = g * h_fc + (1 - g) * h_phase

    g close to 1 means "for this subject/dimension, trust the FC view";
    close to 0 means "trust the phase-coherence view". This is directly
    interpretable per subject (call forward with return_gate=True to
    inspect it) and uses far fewer parameters than a self-attention
    block, since the gate is just one small linear+sigmoid layer rather
    than full multi-head Q/K/V projections plus a feedforward block.
    """
    def __init__(self, n_pairs_fc, n_pairs_phase, proj_dim=16, hidden_dim=32,
                 dropout=0.5, n_classes=2, vector_gate=True):
        super().__init__()
        self.fc_proj = nn.Sequential(nn.Linear(n_pairs_fc, proj_dim), nn.GELU())
        self.phase_proj = nn.Sequential(nn.Linear(n_pairs_phase, proj_dim), nn.GELU())

        # gate looks at BOTH projected branches to decide how to blend them
        gate_out_dim = proj_dim if vector_gate else 1
        self.gate = nn.Linear(proj_dim * 2, gate_out_dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x_fc, x_phase, return_gate=False):
        # x_fc: (B, n_pairs_fc), x_phase: (B, n_pairs_phase)
        h_fc = self.fc_proj(x_fc)        # (B, proj_dim)
        h_phase = self.phase_proj(x_phase)  # (B, proj_dim)

        gate_input = torch.cat([h_fc, h_phase], dim=1)  # (B, 2*proj_dim)
        g = torch.sigmoid(self.gate(gate_input))        # (B, proj_dim) or (B, 1)

        h = g * h_fc + (1 - g) * h_phase  # (B, proj_dim) -- broadcasts if g is (B,1)
        logits = self.classifier(h)

        if return_gate:
            return logits, g
        return logits


def train_one_fold_static_fusion(X_train_fc, X_train_phase, y_train,
                                  X_val_fc, X_val_phase, y_val,
                                  n_pairs_fc, n_pairs_phase, epochs=60, lr=1e-4):
    model = fMRIStaticFusionGated(n_pairs_fc=n_pairs_fc, n_pairs_phase=n_pairs_phase).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr_fc = torch.tensor(X_train_fc).to(device)
    Xtr_phase = torch.tensor(X_train_phase).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva_fc = torch.tensor(X_val_fc).to(device)
    Xva_phase = torch.tensor(X_val_phase).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr_fc, Xtr_phase)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_fc, Xva_phase)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_static_fusion_transformer(X, y, real_lengths, skf, k_fc_select=1500,
                                        fc_method="pearson", shrinkage_alpha=0.1,
                                        TR=PHASE_TR, band=PHASE_BAND):
    """
    Per fold: computes static FC and static phase-coherence features
    (both deterministic per-subject, no fold-fitting needed for the
    features themselves), runs F-score selection SEPARATELY on each
    branch (fold-safe: fit on training subjects only), then trains the
    gated static fusion model.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing static FC + static phase-coherence features ---")

        X_train_fc = compute_static_fc_features(X_train_raw, rl_train, fc_method, shrinkage_alpha)
        X_val_fc = compute_static_fc_features(X_val_raw, rl_val, fc_method, shrinkage_alpha)

        X_train_phase = compute_static_phase_coherence_features(X_train_raw, rl_train, TR, band)
        X_val_phase = compute_static_phase_coherence_features(X_val_raw, rl_val, TR, band)

        if k_fc_select is not None:
            fc_idx, _ = select_top_pairs_2d(X_train_fc, y_train, k_fc_select)
            X_train_fc = X_train_fc[:, fc_idx]
            X_val_fc = X_val_fc[:, fc_idx]

            phase_idx, _ = select_top_pairs_2d(X_train_phase, y_train, k_fc_select)
            X_train_phase = X_train_phase[:, phase_idx]
            X_val_phase = X_val_phase[:, phase_idx]

            print(f"F-score selection: static-FC kept {len(fc_idx)}, "
                  f"static-phase kept {len(phase_idx)} ROI-pair features "
                  f"(selected independently per branch)")

        model, auc = train_one_fold_static_fusion(
            X_train_fc, X_train_phase, y_train, X_val_fc, X_val_phase, y_val,
            n_pairs_fc=X_train_fc.shape[1], n_pairs_phase=X_train_phase.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_fc_t = torch.tensor(X_val_fc).to(device)
            Xva_phase_t = torch.tensor(X_val_phase).to(device)
            val_logits = model(Xva_fc_t, Xva_phase_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Static FC + Static Phase-Coherence Fusion: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Static FC + Static Phase-Coherence Fusion (learned gate)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
static_fusion_results = evaluate_static_fusion_transformer(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", shrinkage_alpha=0.1
)


# =====================================================
# 7. ELEMENT-WISE MATRIX FUSION (Pearson FC ⊙/+ phase-coherence FC)
# =====================================================
# Rather than fusing the two static views AFTER collapsing each to a
# flat vector (section 6's gated model, which never sees the (R, R)
# structure), this section combines the two (R, R) matrices directly,
# edge by edge, into ONE fused FC matrix per subject -- before any
# flattening or modeling happens. This collapses the whole pipeline
# back to a SINGLE feature branch, so no dual-branch fusion model is
# needed downstream at all: just the original single-FC classifier
# pattern.
#
# Two combination methods are supported:
#   "hadamard"    : combined = FC_pearson * FC_phase (element-wise).
#                   An edge only stays strong if BOTH views agree it's
#                   strong -- tends to suppress noise that only one
#                   method happens to pick up, at the cost of also
#                   suppressing genuine signal only one view captures.
#   "weighted_avg": combined = alpha*FC_pearson + (1-alpha)*FC_phase.
#                   A softer blend; alpha=0.5 weighs both views
#                   equally, alpha>0.5 favors Pearson, alpha<0.5 favors
#                   phase-coherence.
# =====================================================

def compute_combined_static_fc_features(X_raw, real_lengths, fc_method="pearson",
                                         shrinkage_alpha=0.1, TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5, order=4):
    """
    For each subject: computes the static Pearson/partial FC matrix AND
    the static phase-coherence matrix (same definitions as
    compute_static_fc_features / compute_static_phase_coherence_features),
    then combines them ELEMENT-WISE into a single (R, R) matrix before
    flattening -- i.e. fusion happens at the edge level, not after
    vectorization.

    combine_method: "hadamard" (element-wise product) or "weighted_avg"
                     (alpha*pearson + (1-alpha)*phase).
    alpha: only used for "weighted_avg" -- weight given to the FC branch.

    Returns: (N, n_pairs) float32 -- a SINGLE combined feature vector
             per subject (not two branches).
    """
    if combine_method not in ("hadamard", "weighted_avg"):
        raise ValueError(f"Unknown combine_method: {combine_method!r} "
                          f"(expected 'hadamard' or 'weighted_avg')")

    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson/partial FC matrix (full R x R, not yet flattened) ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc_mat = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc_mat = weighted_corrcoef(X_raw[i], w)
        # Force EXACT symmetry: fc_mat is symmetric by mathematical
        # construction (it comes from A^T @ W @ A), but floating-point
        # matrix multiplication doesn't guarantee the computed result is
        # bit-for-bit symmetric -- tiny rounding differences between
        # fc_mat[a,b] and fc_mat[b,a] would otherwise silently leak into
        # combined_mat via the element-wise product below.
        fc_mat = (fc_mat + fc_mat.T) / 2.0

        # --- static phase-coherence matrix (full R x R, not yet flattened) ---
        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))  # (L, R)
            diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)  # (R, R)

        # --- element-wise combination, BEFORE flattening ---
        if combine_method == "hadamard":
            combined_mat = fc_mat * phase_mat
        else:  # weighted_avg
            combined_mat = alpha * fc_mat + (1 - alpha) * phase_mat

        feats[i] = combined_mat[triu_idx]

    return feats


class fMRICombinedStaticClassifier(nn.Module):
    """
    Single-branch classifier for the element-wise-combined static FC
    feature -- no dual-branch fusion needed, since combination already
    happened at the matrix level in compute_combined_static_fc_features.
    A linear projection down to a small hidden size, then a compact MLP
    head classifies from there.
    """
    def __init__(self, n_pairs, proj_dim=16, hidden_dim=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(n_pairs, proj_dim), nn.GELU())
        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        h = self.proj(x)
        return self.classifier(h)


def train_one_fold_combined_static(X_train, y_train, X_val, y_val, n_pairs, epochs=60, lr=1e-4):
    model = fMRICombinedStaticClassifier(n_pairs=n_pairs).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_combined_static_classifier(X, y, real_lengths, skf, k_fc_select=1500,
                                         fc_method="pearson", shrinkage_alpha=0.1,
                                         TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5):
    """
    Per fold: computes the element-wise-combined static FC feature
    (single vector per subject), F-score-selects it (fold-safe: fit on
    training subjects only), then trains the single-branch classifier.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing element-wise combined FC ({combine_method}) ---")

        X_train_combined = compute_combined_static_fc_features(
            X_train_raw, rl_train, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )
        X_val_combined = compute_combined_static_fc_features(
            X_val_raw, rl_val, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )

        if k_fc_select is not None:
            top_idx, _ = select_top_pairs_2d(X_train_combined, y_train, k_fc_select)
            X_train_combined = X_train_combined[:, top_idx]
            X_val_combined = X_val_combined[:, top_idx]
            print(f"F-score selection: kept {len(top_idx)} ROI-pair features "
                  f"(from the combined matrix)")

        model, auc = train_one_fold_combined_static(
            X_train_combined, y_train, X_val_combined, y_val,
            n_pairs=X_train_combined.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_combined).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== Element-Wise Combined FC ({combine_method}): Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Element-Wise Combined FC Classifier (hadamard product)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
combined_matrix_results = evaluate_combined_static_classifier(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", combine_method="hadamard"
)


# =====================================================
# 8. INDIVIDUAL-FEATURE BASELINES + COMPARISON vs COMBINED FC
# =====================================================
# Runs the SAME single-branch classifier (fMRICombinedStaticClassifier /
# train_one_fold_combined_static) on each static feature INDIVIDUALLY --
# static FC alone, static phase-coherence alone -- so the combined
# (Hadamard) result above can be judged against its two ingredients
# rather than in isolation. Reuses the exact same model/training code;
# only the feature-extraction function fed in differs, which keeps the
# comparison controlled (same architecture, same training procedure,
# same folds -- only the input feature changes).
# =====================================================

def evaluate_single_static_feature(X, y, real_lengths, skf, feature_fn, feature_name,
                                    k_fc_select=1500, **feature_kwargs):
    """
    Generic single-branch evaluator: computes ONE static feature type
    per subject via feature_fn(X_raw, real_lengths, **feature_kwargs),
    F-score-selects it (fold-safe), then trains/evaluates the same
    single-branch classifier used for the combined-matrix feature.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing {feature_name} ---")

        X_train_feat = feature_fn(X_train_raw, rl_train, **feature_kwargs)
        X_val_feat = feature_fn(X_val_raw, rl_val, **feature_kwargs)

        if k_fc_select is not None:
            top_idx, _ = select_top_pairs_2d(X_train_feat, y_train, k_fc_select)
            X_train_feat = X_train_feat[:, top_idx]
            X_val_feat = X_val_feat[:, top_idx]
            print(f"F-score selection: kept {len(top_idx)} ROI-pair features")

        model, auc = train_one_fold_combined_static(
            X_train_feat, y_train, X_val_feat, y_val, n_pairs=X_train_feat.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_feat).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== {feature_name}: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training baseline: Static FC (Pearson) alone")
print("=" * 60)
static_fc_only_results = evaluate_single_static_feature(
    X, y, real_lengths, skf, compute_static_fc_features, "Static FC (Pearson) alone",
    k_fc_select=1500, fc_method="pearson"
)

print("\n" + "=" * 60)
print("Training baseline: Static Phase-Coherence alone")
print("=" * 60)
static_phase_only_results = evaluate_single_static_feature(
    X, y, real_lengths, skf, compute_static_phase_coherence_features, "Static Phase-Coherence alone",
    k_fc_select=1500
)

print("\n" + "=" * 60)
print("COMPARISON: Static FC alone vs Static Phase-Coherence alone vs Combined FC (hadamard)")
print("=" * 60)
print(f"{'Metric':<15}{'FC alone':<18}{'Phase alone':<18}{'Combined':<18}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(static_fc_only_results[metric]); a_std = np.std(static_fc_only_results[metric])
    b_mean = np.mean(static_phase_only_results[metric]); b_std = np.std(static_phase_only_results[metric])
    c_mean = np.mean(combined_matrix_results[metric]); c_std = np.std(combined_matrix_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}   {b_mean:.3f} ± {b_std:.3f}   {c_mean:.3f} ± {c_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> Static FC + Static Phase-Coherence Fusion (learned gate)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from scipy.signal import butter, filtfilt, hilbert

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. WEIGHTED FC UTILITIES (Pearson + shrinkage-regularized partial)
# =====================================================
# General-purpose weighted functional-connectivity estimators. A weight
# vector of all-ones over a subject's real (non-padded) timepoints
# gives ordinary whole-scan static FC -- that's how these are used
# below (section 6). They also generalize to arbitrary non-negative
# per-timepoint weights (e.g. posterior probabilities, cluster
# membership masks) if you later reintroduce any dynamic/state-based
# weighting scheme.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative.
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero, returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Weighted PARTIAL correlation matrix, via a shrinkage-regularized
    precision matrix.

    data: (T, n_rois), weights: (T,) non-negative.

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total weight)
    comfortably exceeds n_rois. With whole-brain ROI counts (e.g. 200)
    and short scans (T ~ 78), the raw weighted covariance matrix can be
    singular or near-singular -- naive inversion would silently produce
    unstable, unusable partial correlations. To guard against this, the
    covariance diagonal is shrunk toward a scaled identity before
    inversion (a ridge-regularized precision matrix, in the same spirit
    as Ledoit-Wolf shrinkage / graphical lasso), which guarantees
    invertibility and stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain settings; will likely be
    unstable). Higher values shrink more aggressively toward
    independence (a diagonal-only precision matrix), trading some
    sensitivity for stability. 0.1 is a reasonable starting point;
    increase if you still see extreme values or instability (e.g. try
    0.2-0.3 for whole-brain/200-ROI settings), decrease if you've
    restricted to a small ROI subset (e.g. a single network with
    T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


# =====================================================
# 5. F-SCORE FEATURE SELECTION UTILITY
# =====================================================

def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=1500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on FC vectors, computed using ONLY the training subjects of the
    current fold (fold-safe).

    X_train_states: (N_train, n_groups, n_pairs) -- one FC vector of
                    n_pairs per "group" (e.g. a branch/source; for the
                    static case below, n_groups=1).

    The F-score for each ROI-pair is computed SEPARATELY for every
    group, and these per-group F-scores are then AVERAGED ACROSS GROUPS
    to get one final importance score per pair. When n_groups=1 (the
    static case), this trivially reduces to plain per-pair F-scoring.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_groups, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER GROUP, then stack
    per_group_f_scores = np.zeros((n_groups, n_pairs), dtype=np.float64)
    for g in range(n_groups):
        group_features = X_train_states[:, g, :]  # (N_train, n_pairs)
        f_scores_g, _ = f_classif(group_features, y_train)
        per_group_f_scores[g] = np.nan_to_num(f_scores_g, nan=0.0)

    # average the F-scores ACROSS GROUPS (not the raw features)
    avg_f_scores = per_group_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


# =====================================================
# 6. STATIC FC + STATIC PHASE-COHERENCE FUSION (learned gate)
# =====================================================
# Each subject is reduced to exactly TWO static (whole-scan) feature
# vectors -- no HMM, no clustering, no per-state/dynamic structure:
#   1. static FC: ordinary Pearson/partial-correlation FC, computed
#      once over the ENTIRE (real, non-padded) scan -- reuses
#      weighted_corrcoef/weighted_partial_corrcoef with a weight vector
#      of all-ones over real timepoints (i.e. the "one giant state =
#      the whole scan" special case).
#   2. static phase-coherence FC: band-pass filter + Hilbert transform
#      to get instantaneous phase, then the instantaneous
#      phase-coherence matrix cos(phase_i - phase_j) is simply AVERAGED
#      across every real timepoint into one (R, R) matrix. No k-means,
#      no per-fold fitting needed for this step -- it's a deterministic
#      per-subject computation. This is the direct phase-based analogue
#      of static Pearson FC.
#
# These two static vectors are fused via a learned GATE
# (fMRIStaticFusionGated): each branch gets its own linear projection,
# then a small gate network (one linear + sigmoid layer, looking at
# both projections together) decides how much to trust each branch,
# per subject -- h = g*h_fc + (1-g)*h_phase. This is directly
# interpretable (inspect g to see which view the model relied on for a
# given subject) and uses far fewer parameters than a self-attention
# block, since there's no multi-head Q/K/V machinery, just one small
# linear layer producing the blend weight.
# =====================================================

# --- placeholder: SET THIS to your actual (site-consistent) TR in seconds ---
PHASE_TR = 2.0           # <-- VERIFY against ABIDE phenotypic metadata for your sites
PHASE_BAND = (0.04, 0.07)  # Hz; standard fMRI phase-coherence band, adjust if needed


def bandpass_filter_multichannel(ts, low, high, fs, order=4):
    """Zero-phase Butterworth band-pass filter, applied per-column (per-ROI)."""
    nyq = fs / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, ts, axis=0)


def compute_static_fc_features(X_raw, real_lengths, fc_method="pearson", shrinkage_alpha=0.1):
    """
    Static whole-scan FC per subject: Pearson or shrinkage-regularized
    partial correlation, computed once over ALL real (non-padded)
    timepoints (weight = 1 for real timepoints, 0 for padding) -- no
    per-state/dynamic structure at all.

    Returns: (N, n_pairs) float32
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc = weighted_corrcoef(X_raw[i], w)
        feats[i] = fc[triu_idx]

    return feats


def compute_static_phase_coherence_features(X_raw, real_lengths, TR=PHASE_TR, band=PHASE_BAND, order=4):
    """
    Static grand-average phase-coherence FC per subject: band-pass
    filter + Hilbert transform to get instantaneous phase, compute the
    instantaneous phase-coherence matrix at EVERY real timepoint, then
    AVERAGE these matrices across all real timepoints into one (R, R)
    matrix -- no clustering step. This collapses the whole scan's phase
    relationships into one static summary, the phase-based analogue of
    static Pearson FC.

    Returns: (N, n_pairs) float32

    IMPORTANT TR CAVEAT: this dataset resamples every subject to a
    FIXED NUMBER of timepoints (TARGET_TIME), not a fixed PHYSICAL
    DURATION -- so if your included ABIDE sites don't share
    (approximately) the same TR, the band here refers to a different
    physical frequency band per subject. Verify TR consistency (via
    ABIDE phenotypic metadata) before trusting these features, or
    restrict SITE_FILTER to a single-TR site.
    """
    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])
        if L < 10:  # too short to filter/estimate phase meaningfully
            continue
        ts = X_raw[i, :L, :]
        filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
        phase = np.angle(hilbert(filtered, axis=0))  # (L, R)

        diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
        coh = np.cos(diff)                              # (L, R, R)
        grand_avg_coh = coh.mean(axis=0)                # (R, R) -- average over time

        feats[i] = grand_avg_coh[triu_idx]

    return feats


def select_top_pairs_2d(X_train_2d, y_train, k):
    """
    Thin wrapper around select_top_fc_pairs_by_fscore for the static
    (single-vector-per-subject, no group axis) case -- reshapes to
    (N, 1, n_pairs) so the existing per-group-then-average F-score
    logic still applies (trivially, since there's only one group).
    """
    top_idx, scores = select_top_fc_pairs_by_fscore(X_train_2d[:, None, :], y_train, k=k)
    return top_idx, scores


class fMRIStaticFusionGated(nn.Module):
    """
    Fuses two STATIC representations per subject via a learned GATE
    rather than concatenation or attention. Each branch (static FC,
    static phase-coherence) gets its own linear projection into a
    shared d-dimensional space; a small gate network then looks at
    both projected vectors together and outputs g in (0, 1) --
    per-dimension by default (vector gate) -- which blends them:

        h = g * h_fc + (1 - g) * h_phase

    g close to 1 means "for this subject/dimension, trust the FC view";
    close to 0 means "trust the phase-coherence view". This is directly
    interpretable per subject (call forward with return_gate=True to
    inspect it) and uses far fewer parameters than a self-attention
    block, since the gate is just one small linear+sigmoid layer rather
    than full multi-head Q/K/V projections plus a feedforward block.
    """
    def __init__(self, n_pairs_fc, n_pairs_phase, proj_dim=16, hidden_dim=32,
                 dropout=0.5, n_classes=2, vector_gate=True):
        super().__init__()
        self.fc_proj = nn.Sequential(nn.Linear(n_pairs_fc, proj_dim), nn.GELU())
        self.phase_proj = nn.Sequential(nn.Linear(n_pairs_phase, proj_dim), nn.GELU())

        # gate looks at BOTH projected branches to decide how to blend them
        gate_out_dim = proj_dim if vector_gate else 1
        self.gate = nn.Linear(proj_dim * 2, gate_out_dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x_fc, x_phase, return_gate=False):
        # x_fc: (B, n_pairs_fc), x_phase: (B, n_pairs_phase)
        h_fc = self.fc_proj(x_fc)        # (B, proj_dim)
        h_phase = self.phase_proj(x_phase)  # (B, proj_dim)

        gate_input = torch.cat([h_fc, h_phase], dim=1)  # (B, 2*proj_dim)
        g = torch.sigmoid(self.gate(gate_input))        # (B, proj_dim) or (B, 1)

        h = g * h_fc + (1 - g) * h_phase  # (B, proj_dim) -- broadcasts if g is (B,1)
        logits = self.classifier(h)

        if return_gate:
            return logits, g
        return logits


def train_one_fold_static_fusion(X_train_fc, X_train_phase, y_train,
                                  X_val_fc, X_val_phase, y_val,
                                  n_pairs_fc, n_pairs_phase, epochs=60, lr=1e-4):
    model = fMRIStaticFusionGated(n_pairs_fc=n_pairs_fc, n_pairs_phase=n_pairs_phase).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr_fc = torch.tensor(X_train_fc).to(device)
    Xtr_phase = torch.tensor(X_train_phase).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva_fc = torch.tensor(X_val_fc).to(device)
    Xva_phase = torch.tensor(X_val_phase).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr_fc, Xtr_phase)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva_fc, Xva_phase)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_static_fusion_transformer(X, y, real_lengths, skf, k_fc_select=1500,
                                        fc_method="pearson", shrinkage_alpha=0.1,
                                        TR=PHASE_TR, band=PHASE_BAND):
    """
    Per fold: computes static FC and static phase-coherence features
    (both deterministic per-subject, no fold-fitting needed for the
    features themselves), runs F-score selection SEPARATELY on each
    branch (fold-safe: fit on training subjects only), then trains the
    gated static fusion model.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing static FC + static phase-coherence features ---")

        X_train_fc = compute_static_fc_features(X_train_raw, rl_train, fc_method, shrinkage_alpha)
        X_val_fc = compute_static_fc_features(X_val_raw, rl_val, fc_method, shrinkage_alpha)

        X_train_phase = compute_static_phase_coherence_features(X_train_raw, rl_train, TR, band)
        X_val_phase = compute_static_phase_coherence_features(X_val_raw, rl_val, TR, band)

        if k_fc_select is not None:
            fc_idx, _ = select_top_pairs_2d(X_train_fc, y_train, k_fc_select)
            X_train_fc = X_train_fc[:, fc_idx]
            X_val_fc = X_val_fc[:, fc_idx]

            phase_idx, _ = select_top_pairs_2d(X_train_phase, y_train, k_fc_select)
            X_train_phase = X_train_phase[:, phase_idx]
            X_val_phase = X_val_phase[:, phase_idx]

            print(f"F-score selection: static-FC kept {len(fc_idx)}, "
                  f"static-phase kept {len(phase_idx)} ROI-pair features "
                  f"(selected independently per branch)")

        model, auc = train_one_fold_static_fusion(
            X_train_fc, X_train_phase, y_train, X_val_fc, X_val_phase, y_val,
            n_pairs_fc=X_train_fc.shape[1], n_pairs_phase=X_train_phase.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_fc_t = torch.tensor(X_val_fc).to(device)
            Xva_phase_t = torch.tensor(X_val_phase).to(device)
            val_logits = model(Xva_fc_t, Xva_phase_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Static FC + Static Phase-Coherence Fusion: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Static FC + Static Phase-Coherence Fusion (learned gate)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
static_fusion_results = evaluate_static_fusion_transformer(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", shrinkage_alpha=0.1
)


# =====================================================
# 7. ELEMENT-WISE MATRIX FUSION (Pearson FC ⊙/+ phase-coherence FC)
# =====================================================
# Rather than fusing the two static views AFTER collapsing each to a
# flat vector (section 6's gated model, which never sees the (R, R)
# structure), this section combines the two (R, R) matrices directly,
# edge by edge, into ONE fused FC matrix per subject -- before any
# flattening or modeling happens. This collapses the whole pipeline
# back to a SINGLE feature branch, so no dual-branch fusion model is
# needed downstream at all: just the original single-FC classifier
# pattern.
#
# Two combination methods are supported:
#   "hadamard"    : combined = FC_pearson * FC_phase (element-wise).
#                   An edge only stays strong if BOTH views agree it's
#                   strong -- tends to suppress noise that only one
#                   method happens to pick up, at the cost of also
#                   suppressing genuine signal only one view captures.
#   "weighted_avg": combined = alpha*FC_pearson + (1-alpha)*FC_phase.
#                   A softer blend; alpha=0.5 weighs both views
#                   equally, alpha>0.5 favors Pearson, alpha<0.5 favors
#                   phase-coherence.
# =====================================================

def compute_combined_static_fc_features(X_raw, real_lengths, fc_method="pearson",
                                         shrinkage_alpha=0.1, TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5, order=4):
    """
    For each subject: computes the static Pearson/partial FC matrix AND
    the static phase-coherence matrix (same definitions as
    compute_static_fc_features / compute_static_phase_coherence_features),
    then combines them ELEMENT-WISE into a single (R, R) matrix before
    flattening -- i.e. fusion happens at the edge level, not after
    vectorization.

    combine_method: "hadamard" (element-wise product) or "weighted_avg"
                     (alpha*pearson + (1-alpha)*phase).
    alpha: only used for "weighted_avg" -- weight given to the FC branch.

    Returns: (N, n_pairs) float32 -- a SINGLE combined feature vector
             per subject (not two branches).
    """
    if combine_method not in ("hadamard", "weighted_avg"):
        raise ValueError(f"Unknown combine_method: {combine_method!r} "
                          f"(expected 'hadamard' or 'weighted_avg')")

    N, T, R = X_raw.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])
    fs = 1.0 / TR
    feats = np.zeros((N, n_pairs), dtype=np.float32)

    for i in range(N):
        L = int(real_lengths[i])

        # --- static Pearson/partial FC matrix (full R x R, not yet flattened) ---
        w = np.zeros(T, dtype=np.float32)
        w[:L] = 1.0
        if fc_method == "partial":
            fc_mat = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
        else:
            fc_mat = weighted_corrcoef(X_raw[i], w)

        # --- static phase-coherence matrix (full R x R, not yet flattened) ---
        if L < 10:
            phase_mat = np.zeros((R, R), dtype=np.float32)
        else:
            ts = X_raw[i, :L, :]
            filtered = bandpass_filter_multichannel(ts, band[0], band[1], fs, order=order)
            phase = np.angle(hilbert(filtered, axis=0))  # (L, R)
            diff = phase[:, :, None] - phase[:, None, :]  # (L, R, R)
            coh = np.cos(diff)
            phase_mat = coh.mean(axis=0).astype(np.float32)  # (R, R)

        # --- element-wise combination, BEFORE flattening ---
        if combine_method == "hadamard":
            combined_mat = fc_mat * phase_mat
        else:  # weighted_avg
            combined_mat = alpha * fc_mat + (1 - alpha) * phase_mat

        feats[i] = combined_mat[triu_idx]

    return feats


class fMRICombinedStaticClassifier(nn.Module):
    """
    Single-branch classifier for the element-wise-combined static FC
    feature -- no dual-branch fusion needed, since combination already
    happened at the matrix level in compute_combined_static_fc_features.
    A linear projection down to a small hidden size, then a compact MLP
    head classifies from there.
    """
    def __init__(self, n_pairs, proj_dim=16, hidden_dim=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(n_pairs, proj_dim), nn.GELU())
        self.classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        h = self.proj(x)
        return self.classifier(h)


def train_one_fold_combined_static(X_train, y_train, X_val, y_val, n_pairs, epochs=60, lr=1e-4):
    model = fMRICombinedStaticClassifier(n_pairs=n_pairs).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_combined_static_classifier(X, y, real_lengths, skf, k_fc_select=1500,
                                         fc_method="pearson", shrinkage_alpha=0.1,
                                         TR=PHASE_TR, band=PHASE_BAND,
                                         combine_method="hadamard", alpha=0.5):
    """
    Per fold: computes the element-wise-combined static FC feature
    (single vector per subject), F-score-selects it (fold-safe: fit on
    training subjects only), then trains the single-branch classifier.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]
        rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

        print(f"\n--- Fold {fold+1}: computing element-wise combined FC ({combine_method}) ---")

        X_train_combined = compute_combined_static_fc_features(
            X_train_raw, rl_train, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )
        X_val_combined = compute_combined_static_fc_features(
            X_val_raw, rl_val, fc_method, shrinkage_alpha, TR, band, combine_method, alpha
        )

        if k_fc_select is not None:
            top_idx, _ = select_top_pairs_2d(X_train_combined, y_train, k_fc_select)
            X_train_combined = X_train_combined[:, top_idx]
            X_val_combined = X_val_combined[:, top_idx]
            print(f"F-score selection: kept {len(top_idx)} ROI-pair features "
                  f"(from the combined matrix)")

        model, auc = train_one_fold_combined_static(
            X_train_combined, y_train, X_val_combined, y_val,
            n_pairs=X_train_combined.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_combined).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print(f"\n===== Element-Wise Combined FC ({combine_method}): Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Element-Wise Combined FC Classifier (hadamard product)")
print("=" * 60)
print(f"NOTE: using TR={PHASE_TR}s, band={PHASE_BAND} Hz for the phase branch -- "
      f"verify TR matches your included ABIDE site(s) before trusting these results.")
combined_matrix_results = evaluate_combined_static_classifier(
    X, y, real_lengths, skf, k_fc_select=1500,
    fc_method="pearson", combine_method="hadamard"
)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=3500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on the per-state FC vectors, computed using ONLY the training
    subjects of the current fold (fold-safe -- same discipline as the
    earlier MI-based ROI selection).

    X_train_states: (N_train, n_states, n_pairs) -- posterior-weighted
                    FC features, one vector of n_pairs per state.

    IMPORTANT: the F-score for each ROI-pair is computed SEPARATELY for
    every state (using that state's own FC values against the label),
    and these per-state F-scores are then AVERAGED ACROSS STATES to get
    one final importance score per pair. This is different from (and
    more faithful than) averaging the raw FC values across states
    first -- computing one F-score per state and then averaging the
    SCORES preserves a pair's discriminative signal even if it's only
    strong in one or two specific states, rather than letting it get
    diluted/cancelled out by blending raw signal across states before
    scoring at all.

    The same top-k pair indices (selected via this averaged score) are
    then applied identically to EVERY state's token -- i.e. the
    selected ROI-pair set is shared across all states, so the "state as
    token" structure (K tokens) is preserved; only n_pairs shrinks to k
    for every token.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_states, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER STATE, then stack
    per_state_f_scores = np.zeros((n_states, n_pairs), dtype=np.float64)
    for k_state in range(n_states):
        state_features = X_train_states[:, k_state, :]  # (N_train, n_pairs)
        f_scores_k, _ = f_classif(state_features, y_train)
        per_state_f_scores[k_state] = np.nan_to_num(f_scores_k, nan=0.0)

    # average the F-scores ACROSS STATES (not the raw features) -- this
    # is the key difference from a "blend features, then score once" approach
    avg_f_scores = per_state_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


def evaluate_hmm_state_transformer(X, y, skf, n_states=6, k_fc_select=None):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, OPTIONALLY reduce the n_pairs
    dimension via fold-safe F-score feature selection (fit on training
    subjects only, then applied identically to train and val), then
    train/evaluate the transformer.

    k_fc_select: if None, no feature selection is applied (uses all
                 n_pairs, original behavior). If an int, keeps only the
                 top-k ROI-pair features (by F-score) for every state's
                 token, shared across states.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states
        )

        if k_fc_select is not None:
            # --- F-score selection using ONLY this fold's training subjects ---
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6, F-score feature selection)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
#
# k_fc_select: keeps only the top-k ROI-pair FC features (by F-score,
# computed fold-safe on training subjects only) per state token,
# instead of the full n_pairs (e.g. 19,900 for 200 ROIs). Reduces
# dimensionality/overfitting risk before the transformer sees the data.
# Set to None to disable and use all pairs (original behavior).
hmm_state_results = evaluate_hmm_state_transformer(
    X, y, skf, n_states=6, k_fc_select=3500
)


# =====================================================
# 5. HMM STATE-POSTERIOR SEQUENCE TRANSFORMER (Option 1)
# =====================================================
# Section 4 collapsed each subject's scan into exactly K tokens (one
# per HMM state), using posterior-weighted FC -- state IDENTITY becomes
# the token axis, and the exact moment-to-moment ORDER of state
# transitions is discarded (state 3's token looks the same whether it
# was visited early, late, or in several separate bursts).
#
# THIS section keeps every timepoint as its own token (sequence length
# = T, same as your very first raw-signal transformer), but replaces
# the noisy, high-dimensional raw ROI signal at each timepoint with the
# HMM's compact, denoised K-dimensional STATE-POSTERIOR vector at that
# same moment (e.g. "80% state 3, 15% state 1, 5% elsewhere"). Since
# timepoint ORDER is meaningful here, positional encoding is used again
# -- structurally this is close to your original time-token
# transformer, except the per-timestep input is HMM-denoised rather
# than raw signal.
#
# WHAT THIS DIRECTLY TESTS: does the exact SEQUENCE of state
# transitions over time carry classification signal beyond what
# aggregate per-state occupancy/FC (section 4) already captures? If
# this version outperforms section 4, timing/order of transitions
# matters. If it performs similarly or worse, aggregate per-state
# statistics were already capturing what mattered, and the extra
# temporal detail here is mostly noise for this task.
#
# LEAKAGE: same discipline as section 4 -- the HMM is fit ONLY on each
# fold's training subjects, then used (unchanged) to compute posterior
# sequences for both train and validation subjects in that fold.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed because each
    token here is a TIMEPOINT, and timepoint order is meaningful."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_hmm_posterior_sequences(X_train_raw, X_val_raw, n_states=6,
                                     n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (same
    leakage-safe approach as section 4), then computes the FULL
    per-timepoint posterior sequence (T, n_states) for every train and
    val subject using that fitted model.

    Returns: (X_train_post, X_val_post), each of shape (N, T, n_states).
    Note dtype is explicitly cast to float32 -- hmmlearn's
    predict_proba returns float64, which silently breaks PyTorch's
    matmul (it requires matching dtypes) if not cast first.
    """
    N_train, T, R = X_train_raw.shape

    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_posterior_seqs(X_raw):
        N = X_raw.shape[0]
        posteriors = np.stack(
            [model.predict_proba(X_raw[i]) for i in range(N)]
        ).astype(np.float32)  # (N, T, n_states)
        return posteriors

    X_train_post = build_posterior_seqs(X_train_raw)
    X_val_post = build_posterior_seqs(X_val_raw)

    return X_train_post, X_val_post


class HMMPosteriorSeqTransformer(nn.Module):
    """
    Token axis = TIMEPOINTS (in order, T of them). Each token's feature
    is that timepoint's HMM state-posterior vector (n_states-dim),
    instead of the raw n_rois-dim signal.
    """
    def __init__(self, n_states, seq_len, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_states, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(seq_len, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, T, n_states)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_posterior_seq(X_train_post, y_train, X_val_post, y_val,
                                      n_states, seq_len, epochs=60, lr=1e-4):
    model = HMMPosteriorSeqTransformer(n_states=n_states, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_post).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_post).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute per-timepoint posterior SEQUENCES for train and val
    subjects using that fitted model, then train/evaluate the
    time-token transformer on those sequences.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    seq_len = X.shape[1]

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_post, X_val_post = compute_hmm_posterior_sequences(
            X_train_raw, X_val_raw, n_states=n_states
        )

        model, auc = train_one_fold_hmm_posterior_seq(
            X_train_post, y_train, X_val_post, y_val,
            n_states=n_states, seq_len=seq_len
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_post).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM Posterior-Sequence Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM Posterior-Sequence Transformer (n_states=6)")
print("=" * 60)
hmm_posterior_seq_results = evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6)


# =====================================================
# 6. QUICK COMPARISON -- STATE-FC TOKENS (section 4) vs POSTERIOR-SEQUENCE (section 5)
# =====================================================
print("\n" + "=" * 60)
print("COMPARISON: HMM State-FC (K tokens) vs HMM Posterior-Sequence (T tokens)")
print("=" * 60)
print(f"{'Metric':<15}{'State-FC (K tok)':<20}{'Posterior-Seq (T tok)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(hmm_posterior_seq_results[metric]); b_std = np.std(hmm_posterior_seq_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def weighted_partial_corrcoef(data, weights, shrinkage_alpha=0.1, eps=1e-8):
    """
    Posterior-weighted PARTIAL correlation matrix, via a shrinkage-
    regularized precision matrix.

    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).

    WHY REGULARIZATION IS NECESSARY: partial correlation requires
    inverting the (weighted) covariance matrix. That inversion is only
    well-behaved when the EFFECTIVE sample size (here, total posterior
    weight, which is often << T for less-visited states) comfortably
    exceeds n_rois. With whole-brain ROI counts (e.g. 200) and short
    scans (T ~ 78, further reduced by per-state weighting), the raw
    weighted covariance matrix is frequently singular or near-singular
    -- naive inversion would silently produce unstable, unusable
    partial correlations. To guard against this, the covariance
    diagonal is shrunk toward a scaled identity before inversion (a
    ridge-regularized precision matrix, in the same spirit as Ledoit-Wolf
    shrinkage / graphical lasso), which guarantees invertibility and
    stabilizes the estimate at the cost of some bias.

    shrinkage_alpha: float in [0, 1). 0 = no regularization (raw
    inverse -- NOT recommended for whole-brain, low-occupancy-state
    settings; will likely be unstable). Higher values shrink more
    aggressively toward independence (a diagonal-only precision
    matrix), trading some sensitivity for stability. 0.1 is a
    reasonable starting point; increase if you still see extreme
    values or instability (e.g. try 0.2-0.3 for whole-brain/200-ROI
    settings), decrease if you've restricted to a small ROI subset
    (e.g. a single network with T > n_rois).

    If total weight is (near) zero, returns an all-zero matrix (same
    convention as weighted_corrcoef).
    """
    n_rois = data.shape[1]
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((n_rois, n_rois), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    # Ridge shrinkage toward a scaled identity, so the matrix stays
    # invertible even when effective sample size < n_rois.
    diag_mean = np.trace(cov) / n_rois
    cov_shrunk = (1 - shrinkage_alpha) * cov + shrinkage_alpha * diag_mean * np.eye(n_rois)

    try:
        precision = np.linalg.inv(cov_shrunk)
    except np.linalg.LinAlgError:
        # Extremely rare given the shrinkage above, but fall back to a
        # pseudo-inverse rather than crashing a whole fold.
        precision = np.linalg.pinv(cov_shrunk)

    diag_p = np.diag(precision)
    denom = np.sqrt(np.outer(diag_p, diag_p))
    denom[denom < eps] = eps
    pcorr = -precision / denom
    np.fill_diagonal(pcorr, 1.0)
    return np.nan_to_num(pcorr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    fc_method: "pearson" (marginal correlation, weighted_corrcoef) or
               "partial" (shrinkage-regularized partial correlation,
               weighted_partial_corrcoef -- estimates DIRECT connectivity
               between ROI pairs, controlling for all other ROIs). See
               weighted_partial_corrcoef's docstring for why shrinkage
               is necessary, especially with whole-brain ROI counts.
    shrinkage_alpha: only used when fc_method="partial". See
               weighted_partial_corrcoef docstring for guidance.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    if fc_method not in ("pearson", "partial"):
        raise ValueError(f"Unknown fc_method: {fc_method!r} (expected 'pearson' or 'partial')")

    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                if fc_method == "partial":
                    fc = weighted_partial_corrcoef(X_raw[i], w, shrinkage_alpha=shrinkage_alpha)
                else:
                    fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=2500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on the per-state FC vectors, computed using ONLY the training
    subjects of the current fold (fold-safe -- same discipline as the
    earlier MI-based ROI selection).

    X_train_states: (N_train, n_states, n_pairs) -- posterior-weighted
                    FC features, one vector of n_pairs per state.

    IMPORTANT: the F-score for each ROI-pair is computed SEPARATELY for
    every state (using that state's own FC values against the label),
    and these per-state F-scores are then AVERAGED ACROSS STATES to get
    one final importance score per pair. This is different from (and
    more faithful than) averaging the raw FC values across states
    first -- computing one F-score per state and then averaging the
    SCORES preserves a pair's discriminative signal even if it's only
    strong in one or two specific states, rather than letting it get
    diluted/cancelled out by blending raw signal across states before
    scoring at all.

    The same top-k pair indices (selected via this averaged score) are
    then applied identically to EVERY state's token -- i.e. the
    selected ROI-pair set is shared across all states, so the "state as
    token" structure (K tokens) is preserved; only n_pairs shrinks to k
    for every token.

    Returns: (top_pair_idx sorted ascending, averaged f_scores array)
    """
    N_train, n_states, n_pairs = X_train_states.shape

    # compute one F-score vector (length n_pairs) PER STATE, then stack
    per_state_f_scores = np.zeros((n_states, n_pairs), dtype=np.float64)
    for k_state in range(n_states):
        state_features = X_train_states[:, k_state, :]  # (N_train, n_pairs)
        f_scores_k, _ = f_classif(state_features, y_train)
        per_state_f_scores[k_state] = np.nan_to_num(f_scores_k, nan=0.0)

    # average the F-scores ACROSS STATES (not the raw features) -- this
    # is the key difference from a "blend features, then score once" approach
    avg_f_scores = per_state_f_scores.mean(axis=0)  # (n_pairs,)

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(avg_f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), avg_f_scores


def evaluate_hmm_state_transformer(X, y, skf, n_states=6, k_fc_select=None,
                                    fc_method="pearson", shrinkage_alpha=0.1):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, OPTIONALLY reduce the n_pairs
    dimension via fold-safe F-score feature selection (fit on training
    subjects only, then applied identically to train and val), then
    train/evaluate the transformer.

    k_fc_select: if None, no feature selection is applied (uses all
                 n_pairs, original behavior). If an int, keeps only the
                 top-k ROI-pair features (by F-score) for every state's
                 token, shared across states.
    fc_method / shrinkage_alpha: passed through to
                 compute_hmm_state_fc_features -- see that function's
                 and weighted_partial_corrcoef's docstrings.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states,
            fc_method=fc_method, shrinkage_alpha=shrinkage_alpha
        )

        if k_fc_select is not None:
            # --- F-score selection using ONLY this fold's training subjects ---
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6, F-score feature selection)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
#
# k_fc_select: keeps only the top-k ROI-pair FC features (by F-score,
# computed fold-safe on training subjects only) per state token,
# instead of the full n_pairs (e.g. 19,900 for 200 ROIs). Reduces
# dimensionality/overfitting risk before the transformer sees the data.
# Set to None to disable and use all pairs (original behavior).
#
# fc_method="partial": estimates DIRECT ROI-to-ROI connectivity (i.e.
# controls for all other ROIs) rather than marginal Pearson
# correlation, which can't distinguish a direct connection from one
# mediated through a third region. Because this is a whole-brain
# (R=200) setting with short, posterior-weighted effective sample
# sizes per state, shrinkage_alpha=0.1 regularizes the precision-matrix
# inversion to keep it stable -- raise this (e.g. 0.2-0.3) if you
# still see unstable/extreme values, or set fc_method="pearson" to
# revert to the original marginal-correlation behavior.
hmm_state_results = evaluate_hmm_state_transformer(
    X, y, skf, n_states=6, k_fc_select=2500,
    fc_method="partial", shrinkage_alpha=0.1
)


# =====================================================
# 5. FINAL RESULTS SUMMARY
# =====================================================
print("\n" + "=" * 60)
print("FINAL RESULTS: HMM State-FC Transformer (K state tokens)")
print("=" * 60)
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    mean = np.mean(hmm_state_results[metric])
    std = np.std(hmm_state_results[metric])
    print(f"{metric:<15}: {mean:.3f} ± {std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> SVM-FC baseline -> Dynamic FC-Window
# Transformer (+ grid search over window_size / stride)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)


# =====================================================
# 5. DYNAMIC FC-WINDOW TRANSFORMER
# =====================================================
# A static FC matrix collapses the ENTIRE scan into one correlation
# matrix per subject -- robust to scanner/site noise, but throws away
# any information about how connectivity changes over the course of the
# scan (e.g. transient synchronization, state changes).
#
# Here, each subject's scan is split into overlapping time windows; FC is
# computed WITHIN each window (still a robust, correlation-based feature),
# and the SEQUENCE of window-FC vectors becomes the transformer's input.
# The token axis is genuinely TEMPORAL (windows in order), so positional
# encoding is used -- order matters here.
#
# Net effect: keeps the SVM's robust correlation-based feature type,
# while recovering the temporal dynamics a single static FC matrix
# cannot represent.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed because WINDOW
    tokens have a genuine temporal order (unlike, e.g., ROI tokens)."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_dynamic_fc_windows(X, window_size=30, stride=15, verbose=True):
    """
    X: (N, T, n_rois) raw (already per-ROI standardized) timeseries.
    Returns: (N, n_windows, n_roi_pairs) -- for each subject, a SEQUENCE
             of flattened upper-triangle FC vectors, one per sliding window.

    n_windows is determined by T, window_size, stride and is IDENTICAL
    for every subject (since every subject shares the same T after your
    loader's resample/pad step) -- this keeps the sequence length fixed,
    which the transformer requires.
    """
    N, T, R = X.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    starts = list(range(0, T - window_size + 1, stride))
    n_windows = len(starts)

    if n_windows == 0:
        raise ValueError(
            f"window_size={window_size} is larger than seq_len={T} -- "
            f"reduce window_size or stride."
        )

    windows_fc = np.zeros((N, n_windows, n_pairs), dtype=np.float32)
    nan_window_count = 0

    for i in range(N):
        for w, start in enumerate(starts):
            seg = X[i, start:start + window_size, :]
            fc = np.corrcoef(seg.T)
            if np.isnan(fc).any():
                nan_window_count += 1
                fc = np.nan_to_num(fc, nan=0.0)
            windows_fc[i, w] = fc[triu_idx]

    if verbose:
        print(f"n_windows per subject : {n_windows} (window_size={window_size}, stride={stride})")
        print(f"n_roi_pairs per window: {n_pairs}")
        print(f"Windows with NaN FC (fixed via nan_to_num): {nan_window_count} / {N * n_windows}")

    return windows_fc


class fMRITransformerDynamicFC(nn.Module):
    """
    Token axis = time WINDOWS (in order). Each token's feature is that
    window's flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_windows, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(n_windows, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_windows, n_pairs)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_dynamic_fc(X_train_win, y_train, X_val_win, y_val,
                               n_pairs, n_windows, epochs=60, lr=1e-4):
    model = fMRITransformerDynamicFC(n_pairs=n_pairs, n_windows=n_windows).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_win).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_win).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_dynamic_fc_transformer(X, y, skf, window_size=20, stride=10):
    """
    Dynamic-FC-window features are computed ONCE up front (per subject,
    from that subject's own raw timeseries only -- no cross-subject
    statistics involved, so this is leakage-safe to precompute outside
    the fold loop).
    """
    print(f"\nComputing dynamic FC windows (window_size={window_size}, stride={stride})...")
    X_windows = compute_dynamic_fc_windows(X, window_size=window_size, stride=stride)
    n_windows = X_windows.shape[1]
    n_pairs = X_windows.shape[2]

    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_win, X_val_win = X_windows[tr_idx], X_windows[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_dynamic_fc(
            X_train_win, y_train, X_val_win, y_val,
            n_pairs=n_pairs, n_windows=n_windows
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_win).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Dynamic-FC-Window Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Dynamic-FC-Window Transformer (default window_size=20, stride=10)")
print("=" * 60)
# window_size/stride are chosen relative to your seq_len (TARGET_TIME).
# With TARGET_TIME=178: window_size=20, stride=10 gives (178-20)/10+1 = 16 windows.
# Adjust these if your TARGET_TIME changes -- too few windows (e.g. <4)
# defeats the purpose (not enough sequence to attend over); too many
# windows with small stride increases compute and overlap redundancy.
transformer_dynamicfc_results = evaluate_dynamic_fc_transformer(
    X, y, skf, window_size=30, stride=15
)


# =====================================================
# 6. FINAL SUMMARY -- SVM-FC vs DYNAMIC-FC TRANSFORMER (default params)
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs Dynamic-FC-Window Transformer (default params)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'TF(dynamic-FC)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]); svm_std = np.std(svm_results[metric])
    tf_mean = np.mean(transformer_dynamicfc_results[metric]); tf_std = np.std(transformer_dynamicfc_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      {tf_mean:.3f} ± {tf_std:.3f}")


# =====================================================
# 7. GRID SEARCH -- BEST (window_size, stride) FOR DYNAMIC-FC TRANSFORMER
# =====================================================
# Reuses evaluate_dynamic_fc_transformer as-is for each candidate
# (window_size, stride) pair, on the SAME skf folds each time so results
# are directly comparable. Selection is based on mean AUC across folds.
#
# IMPORTANT CAVEAT: because this search reuses validation-fold AUC to
# pick the winning (window_size, stride), the reported AUC for the BEST
# combination is optimistically biased (the same folds used to pick the
# winner are also used to report its score). Treat the winning
# combination's exact number as an upper estimate, not a final
# generalization guarantee -- if this needs to go in a report, ideally
# re-validate the chosen (window_size, stride) on a fresh held-out split
# not used in the search.
# =====================================================

def grid_search_dynamic_fc(X, y, skf, window_sizes, strides, min_windows=4):
    """
    Tries every (window_size, stride) combination from the given lists.
    Skips combinations that would produce fewer than `min_windows`
    windows (too short a sequence for attention to do anything useful)
    or where window_size > seq_len (invalid).

    Returns a list of result dicts, one per valid combination tried,
    each containing the params and the full metric results, sorted by
    mean AUC descending.
    """
    seq_len = X.shape[1]
    grid_results = []

    for window_size in window_sizes:
        for stride in strides:
            if window_size > seq_len:
                print(f"Skipping window_size={window_size} (> seq_len={seq_len})")
                continue

            n_windows = len(range(0, seq_len - window_size + 1, stride))
            if n_windows < min_windows:
                print(f"Skipping window_size={window_size}, stride={stride} "
                      f"-> only {n_windows} windows (< min_windows={min_windows})")
                continue

            print("\n" + "-" * 60)
            print(f"Trying window_size={window_size}, stride={stride} "
                  f"(-> {n_windows} windows)")
            print("-" * 60)

            results = evaluate_dynamic_fc_transformer(
                X, y, skf, window_size=window_size, stride=stride
            )

            grid_results.append({
                "window_size": window_size,
                "stride": stride,
                "n_windows": n_windows,
                "auc_mean": np.mean(results["auc"]),
                "auc_std": np.std(results["auc"]),
                "sensitivity_mean": np.mean(results["sensitivity"]),
                "specificity_mean": np.mean(results["specificity"]),
                "f1_mean": np.mean(results["f1"]),
                "accuracy_mean": np.mean(results["accuracy"]),
                "full_results": results,
            })

    grid_results.sort(key=lambda r: r["auc_mean"], reverse=True)
    return grid_results


def print_grid_search_summary(grid_results):
    print("\n" + "=" * 60)
    print("GRID SEARCH SUMMARY (sorted by mean AUC, best first)")
    print("=" * 60)
    header = f"{'window':<8}{'stride':<8}{'n_win':<7}{'AUC':<16}{'Sens':<8}{'Spec':<8}{'F1':<8}{'Acc':<8}"
    print(header)
    for r in grid_results:
        print(f"{r['window_size']:<8}{r['stride']:<8}{r['n_windows']:<7}"
              f"{r['auc_mean']:.3f}±{r['auc_std']:.3f}    "
              f"{r['sensitivity_mean']:.3f}   "
              f"{r['specificity_mean']:.3f}   "
              f"{r['f1_mean']:.3f}   "
              f"{r['accuracy_mean']:.3f}")

    if grid_results:
        best = grid_results[0]
        print(f"\nBest combination: window_size={best['window_size']}, "
              f"stride={best['stride']}  (AUC={best['auc_mean']:.3f} ± {best['auc_std']:.3f})")
        print("NOTE: this AUC is selected FROM this same grid search's fold "
              "results -- re-validate on a fresh split before treating it as "
              "a final generalization estimate (see caveat above section 7).")


# --- Candidate grids ---
# Adjust these based on your actual seq_len (TARGET_TIME). For seq_len=178,
# window_sizes up to ~40 and strides down to ~5 are reasonable.
# min_windows=4 filters out combinations that would leave too short a
# sequence to be useful.
WINDOW_SIZES = [10, 15, 20, 30, 40]
STRIDES = [5, 10, 15, 20]

print("\n" + "=" * 60)
print("GRID SEARCH: Dynamic-FC-Window Transformer (window_size x stride)")
print("=" * 60)
dynamic_fc_grid_results = grid_search_dynamic_fc(
    X, y, skf, window_sizes=WINDOW_SIZES, strides=STRIDES, min_windows=4
)
print_grid_search_summary(dynamic_fc_grid_results)


# =====================================================
# 8. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_state_transformer(X, y, skf, n_states=6):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, then train/evaluate the transformer.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states
        )
        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
hmm_state_results = evaluate_hmm_state_transformer(X, y, skf, n_states=6)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=5,
                                    n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_state_transformer(X, y, skf, n_states=5):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, then train/evaluate the transformer.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states
        )
        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
hmm_state_results = evaluate_hmm_state_transformer(X, y, skf, n_states=5)

In [ ]:
# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import f_classif
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(116)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def select_top_fc_pairs_by_fscore(X_train_states, y_train, k=500):
    """
    F-score (ANOVA F-value, sklearn's f_classif) based feature selection
    on the FLATTENED per-state FC vectors, computed using ONLY the
    training subjects of the current fold (fold-safe -- exactly the
    same discipline as the earlier MI-based ROI selection).

    X_train_states: (N_train, n_states, n_pairs) -- posterior-weighted
                    FC features, one vector of n_pairs per state.

    Each ROI-pair's F-score is computed against the label using that
    pair's value AVERAGED ACROSS STATES per subject (i.e. one summary
    number per pair per subject) -- mirroring how MI-based ROI
    selection earlier used a mean-over-time summary before ranking.
    This keeps the selection a single, shared set of pair indices,
    applied identically to every state's token so the "state as token"
    structure (K tokens) is preserved -- only n_pairs shrinks to k.

    Returns: (top_pair_idx sorted ascending, full f_scores array)
    """
    N_train, n_states, n_pairs = X_train_states.shape

    # summary feature per pair per subject: mean across the K states
    pair_features = X_train_states.mean(axis=1)  # (N_train, n_pairs)

    f_scores, _ = f_classif(pair_features, y_train)
    f_scores = np.nan_to_num(f_scores, nan=0.0)  # guard against constant-feature pairs

    k = min(k, n_pairs)
    top_pair_idx = np.argsort(f_scores)[::-1][:k]
    return sorted(top_pair_idx.tolist()), f_scores


def evaluate_hmm_state_transformer(X, y, skf, n_states=6, k_fc_select=None):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, OPTIONALLY reduce the n_pairs
    dimension via fold-safe F-score feature selection (fit on training
    subjects only, then applied identically to train and val), then
    train/evaluate the transformer.

    k_fc_select: if None, no feature selection is applied (uses all
                 n_pairs, original behavior). If an int, keeps only the
                 top-k ROI-pair features (by F-score) for every state's
                 token, shared across states.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_pairs_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states
        )

        if k_fc_select is not None:
            # --- F-score selection using ONLY this fold's training subjects ---
            top_pair_idx, _ = select_top_fc_pairs_by_fscore(
                X_train_states, y_train, k=k_fc_select
            )
            selected_pairs_per_fold.append(top_pair_idx)

            X_train_states = X_train_states[:, :, top_pair_idx]
            X_val_states = X_val_states[:, :, top_pair_idx]

            print(f"F-score selection: kept {len(top_pair_idx)} ROI-pair "
                  f"features (shared across all {n_states} state tokens)")

        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    result = {"auc": aucs, "sensitivity": senss, "specificity": specs,
              "f1": f1s, "accuracy": accs}
    if k_fc_select is not None:
        result["selected_pairs_per_fold"] = selected_pairs_per_fold
    return result


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6, F-score feature selection)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
#
# k_fc_select: keeps only the top-k ROI-pair FC features (by F-score,
# computed fold-safe on training subjects only) per state token,
# instead of the full n_pairs (e.g. 19,900 for 200 ROIs). Reduces
# dimensionality/overfitting risk before the transformer sees the data.
# Set to None to disable and use all pairs (original behavior).
hmm_state_results = evaluate_hmm_state_transformer(
    X, y, skf, n_states=6, k_fc_select=500
)


# =====================================================
# 5. HMM STATE-POSTERIOR SEQUENCE TRANSFORMER (Option 1)
# =====================================================
# Section 4 collapsed each subject's scan into exactly K tokens (one
# per HMM state), using posterior-weighted FC -- state IDENTITY becomes
# the token axis, and the exact moment-to-moment ORDER of state
# transitions is discarded (state 3's token looks the same whether it
# was visited early, late, or in several separate bursts).
#
# THIS section keeps every timepoint as its own token (sequence length
# = T, same as your very first raw-signal transformer), but replaces
# the noisy, high-dimensional raw ROI signal at each timepoint with the
# HMM's compact, denoised K-dimensional STATE-POSTERIOR vector at that
# same moment (e.g. "80% state 3, 15% state 1, 5% elsewhere"). Since
# timepoint ORDER is meaningful here, positional encoding is used again
# -- structurally this is close to your original time-token
# transformer, except the per-timestep input is HMM-denoised rather
# than raw signal.
#
# WHAT THIS DIRECTLY TESTS: does the exact SEQUENCE of state
# transitions over time carry classification signal beyond what
# aggregate per-state occupancy/FC (section 4) already captures? If
# this version outperforms section 4, timing/order of transitions
# matters. If it performs similarly or worse, aggregate per-state
# statistics were already capturing what mattered, and the extra
# temporal detail here is mostly noise for this task.
#
# LEAKAGE: same discipline as section 4 -- the HMM is fit ONLY on each
# fold's training subjects, then used (unchanged) to compute posterior
# sequences for both train and validation subjects in that fold.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed because each
    token here is a TIMEPOINT, and timepoint order is meaningful."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_hmm_posterior_sequences(X_train_raw, X_val_raw, n_states=6,
                                     n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (same
    leakage-safe approach as section 4), then computes the FULL
    per-timepoint posterior sequence (T, n_states) for every train and
    val subject using that fitted model.

    Returns: (X_train_post, X_val_post), each of shape (N, T, n_states).
    Note dtype is explicitly cast to float32 -- hmmlearn's
    predict_proba returns float64, which silently breaks PyTorch's
    matmul (it requires matching dtypes) if not cast first.
    """
    N_train, T, R = X_train_raw.shape

    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_posterior_seqs(X_raw):
        N = X_raw.shape[0]
        posteriors = np.stack(
            [model.predict_proba(X_raw[i]) for i in range(N)]
        ).astype(np.float32)  # (N, T, n_states)
        return posteriors

    X_train_post = build_posterior_seqs(X_train_raw)
    X_val_post = build_posterior_seqs(X_val_raw)

    return X_train_post, X_val_post


class HMMPosteriorSeqTransformer(nn.Module):
    """
    Token axis = TIMEPOINTS (in order, T of them). Each token's feature
    is that timepoint's HMM state-posterior vector (n_states-dim),
    instead of the raw n_rois-dim signal.
    """
    def __init__(self, n_states, seq_len, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_states, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(seq_len, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, T, n_states)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_posterior_seq(X_train_post, y_train, X_val_post, y_val,
                                      n_states, seq_len, epochs=60, lr=1e-4):
    model = HMMPosteriorSeqTransformer(n_states=n_states, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_post).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_post).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute per-timepoint posterior SEQUENCES for train and val
    subjects using that fitted model, then train/evaluate the
    time-token transformer on those sequences.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    seq_len = X.shape[1]

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_post, X_val_post = compute_hmm_posterior_sequences(
            X_train_raw, X_val_raw, n_states=n_states
        )

        model, auc = train_one_fold_hmm_posterior_seq(
            X_train_post, y_train, X_val_post, y_val,
            n_states=n_states, seq_len=seq_len
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_post).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM Posterior-Sequence Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM Posterior-Sequence Transformer (n_states=6)")
print("=" * 60)
hmm_posterior_seq_results = evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6)


# =====================================================
# 6. QUICK COMPARISON -- STATE-FC TOKENS (section 4) vs POSTERIOR-SEQUENCE (section 5)
# =====================================================
print("\n" + "=" * 60)
print("COMPARISON: HMM State-FC (K tokens) vs HMM Posterior-Sequence (T tokens)")
print("=" * 60)
print(f"{'Metric':<15}{'State-FC (K tok)':<20}{'Posterior-Seq (T tok)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(hmm_posterior_seq_results[metric]); b_std = np.std(hmm_posterior_seq_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}")
hmm_posterior_seq_results = evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6)


# =====================================================
# 6. QUICK COMPARISON -- STATE-FC TOKENS (section 4) vs POSTERIOR-SEQUENCE (section 5)
# =====================================================
print("\n" + "=" * 60)
print("COMPARISON: HMM State-FC (K tokens) vs HMM Posterior-Sequence (T tokens)")
print("=" * 60)
print(f"{'Metric':<15}{'State-FC (K tok)':<20}{'Posterior-Seq (T tok)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    a_mean = np.mean(hmm_state_results[metric]); a_std = np.std(hmm_state_results[metric])
    b_mean = np.mean(hmm_posterior_seq_results[metric]); b_std = np.std(hmm_posterior_seq_results[metric])
    print(f"{metric:<15}{a_mean:.3f} ± {a_std:.3f}      {b_mean:.3f} ± {b_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from hmmlearn import hmm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. HMM STATE-SPECIFIC FC TRANSFORMER
# =====================================================
# Instead of segmenting each subject's scan with an ARBITRARY fixed
# sliding window, a Hidden Markov Model is fit to learn K latent brain
# states directly from the data. For each subject, we get a posterior
# probability of being in each state at every timepoint, and use these
# posteriors as SOFT WEIGHTS to compute a state-specific FC matrix per
# state (a posterior-weighted correlation, not a hard in/out split --
# this avoids the noise problem of states with very few timepoints).
#
# Each subject then becomes a fixed-length sequence of exactly K tokens
# (one per state), each token being that state's flattened upper-
# triangle FC vector. Since K is fixed and states are SHARED across all
# subjects (the HMM is fit once per fold, on that fold's training
# subjects only), "State 3" means the same thing for every subject --
# unlike a window index, which has no consistent meaning across
# subjects. A learned per-state identity embedding (analogous to the
# ROI-identity embedding used elsewhere in this project) lets the
# transformer distinguish "State 1" from "State 2", etc.
#
# NOTE ON LEAKAGE: the HMM is fit ONLY on the training subjects of each
# fold (never on validation subjects), then used to compute posteriors
# for BOTH train and validation subjects in that fold -- this mirrors
# how the MI-based ROI selection was kept fold-safe earlier in this
# project.
# =====================================================

def weighted_corrcoef(data, weights, eps=1e-8):
    """
    Posterior-weighted Pearson correlation matrix.
    data: (T, n_rois), weights: (T,) non-negative (state posterior probs).
    Returns (n_rois, n_rois) weighted correlation matrix. If total weight
    is (near) zero -- i.e. this state was essentially never visited --
    returns an all-zero matrix rather than dividing by ~0.
    """
    w = weights.reshape(-1, 1)
    w_sum = w.sum()
    if w_sum < eps:
        return np.zeros((data.shape[1], data.shape[1]), dtype=np.float32)

    mu = (w * data).sum(axis=0) / w_sum
    centered = data - mu
    cov = (centered * w).T @ centered / w_sum

    std = np.sqrt(np.diag(cov))
    denom = np.outer(std, std)
    denom[denom < eps] = eps
    corr = cov / denom
    return np.nan_to_num(corr, nan=0.0).astype(np.float32)


def compute_hmm_state_fc_features(X_train_raw, X_val_raw, n_states=6,
                                    n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (concatenated
    across subjects, using hmmlearn's `lengths` argument so subjects are
    not treated as one continuous timeseries), then computes
    posterior-weighted, per-state FC features for both train and val
    subjects using that same fitted model.

    Returns: (X_train_states, X_val_states) each of shape
             (N, n_states, n_roi_pairs).
    """
    N_train, T, R = X_train_raw.shape
    N_val = X_val_raw.shape[0]
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    # --- fit HMM on training subjects only ---
    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_features(X_raw):
        N = X_raw.shape[0]
        feats = np.zeros((N, n_states, n_pairs), dtype=np.float32)
        low_occupancy_flags = 0
        for i in range(N):
            posteriors = model.predict_proba(X_raw[i])  # (T, n_states)
            for k in range(n_states):
                w = posteriors[:, k]
                if w.sum() < 1.0:  # effectively < ~1 full timepoint of weight
                    low_occupancy_flags += 1
                fc = weighted_corrcoef(X_raw[i], w)
                feats[i, k] = fc[triu_idx]
        return feats, low_occupancy_flags

    X_train_states, low_occ_train = build_features(X_train_raw)
    X_val_states, low_occ_val = build_features(X_val_raw)

    if verbose:
        total_state_slots = (N_train + N_val) * n_states
        print(f"Low-occupancy (state,subject) pairs (total weight < 1.0 timepoint): "
              f"{low_occ_train + low_occ_val} / {total_state_slots}")

    return X_train_states, X_val_states


class fMRITransformerHMMState(nn.Module):
    """
    Token axis = HMM STATES (K of them, shared identity across all
    subjects within a fold). Each token's feature is that state's
    posterior-weighted, flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_states, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        # learned state-identity embedding -- analogous to the ROI-identity
        # embedding used elsewhere: state order/index is arbitrary, but
        # consistent within a fold (same fitted HMM for train + val), so
        # the model can still learn "state 3 tends to look like X".
        self.state_embedding = nn.Parameter(torch.randn(1, n_states, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_states, n_pairs)
        h = self.input_proj(x)
        h = h + self.state_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_state(X_train_states, y_train, X_val_states, y_val,
                              n_pairs, n_states, epochs=60, lr=1e-4):
    model = fMRITransformerHMMState(n_pairs=n_pairs, n_states=n_states).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_states).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_states).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_state_transformer(X, y, skf, n_states=6):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute posterior-weighted per-state FC features for train and val
    subjects using that fitted model, then train/evaluate the transformer.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_states, X_val_states = compute_hmm_state_fc_features(
            X_train_raw, X_val_raw, n_states=n_states
        )
        n_pairs = X_train_states.shape[2]

        model, auc = train_one_fold_hmm_state(
            X_train_states, y_train, X_val_states, y_val,
            n_pairs=n_pairs, n_states=n_states
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_states).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM State-FC Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM State-FC Transformer (n_states=6)")
print("=" * 60)
# n_states is a real hyperparameter -- too few states may not capture
# meaningful distinct connectivity patterns; too many, combined with a
# short scan (small T), risks very low per-state occupancy (noisy
# weighted-FC estimates even with soft weighting). 6 is a reasonable
# starting point for scans in the ~78-178 timepoint range.
hmm_state_results = evaluate_hmm_state_transformer(X, y, skf, n_states=6)


# =====================================================
# 5. HMM STATE-POSTERIOR SEQUENCE TRANSFORMER (Option 1)
# =====================================================
# Section 4 collapsed each subject's scan into exactly K tokens (one
# per HMM state), using posterior-weighted FC -- state IDENTITY becomes
# the token axis, and the exact moment-to-moment ORDER of state
# transitions is discarded (state 3's token looks the same whether it
# was visited early, late, or in several separate bursts).
#
# THIS section keeps every timepoint as its own token (sequence length
# = T, same as your very first raw-signal transformer), but replaces
# the noisy, high-dimensional raw ROI signal at each timepoint with the
# HMM's compact, denoised K-dimensional STATE-POSTERIOR vector at that
# same moment (e.g. "80% state 3, 15% state 1, 5% elsewhere"). Since
# timepoint ORDER is meaningful here, positional encoding is used again
# -- structurally this is close to your original time-token
# transformer, except the per-timestep input is HMM-denoised rather
# than raw signal.
#
# WHAT THIS DIRECTLY TESTS: does the exact SEQUENCE of state
# transitions over time carry classification signal beyond what
# aggregate per-state occupancy/FC (section 4) already captures? If
# this version outperforms section 4, timing/order of transitions
# matters. If it performs similarly or worse, aggregate per-state
# statistics were already capturing what mattered, and the extra
# temporal detail here is mostly noise for this task.
#
# LEAKAGE: same discipline as section 4 -- the HMM is fit ONLY on each
# fold's training subjects, then used (unchanged) to compute posterior
# sequences for both train and validation subjects in that fold.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed because each
    token here is a TIMEPOINT, and timepoint order is meaningful."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_hmm_posterior_sequences(X_train_raw, X_val_raw, n_states=6,
                                     n_iter=100, random_state=42, verbose=True):
    """
    Fits a single GaussianHMM on the TRAINING subjects only (same
    leakage-safe approach as section 4), then computes the FULL
    per-timepoint posterior sequence (T, n_states) for every train and
    val subject using that fitted model.

    Returns: (X_train_post, X_val_post), each of shape (N, T, n_states).
    Note dtype is explicitly cast to float32 -- hmmlearn's
    predict_proba returns float64, which silently breaks PyTorch's
    matmul (it requires matching dtypes) if not cast first.
    """
    N_train, T, R = X_train_raw.shape

    concat_train = X_train_raw.reshape(N_train * T, R)
    lengths_train = [T] * N_train

    model = hmm.GaussianHMM(
        n_components=n_states, covariance_type="diag",
        n_iter=n_iter, random_state=random_state
    )
    model.fit(concat_train, lengths_train)

    if verbose:
        print(f"HMM fit on {N_train} training subjects "
              f"({n_states} states, converged={model.monitor_.converged})")

    def build_posterior_seqs(X_raw):
        N = X_raw.shape[0]
        posteriors = np.stack(
            [model.predict_proba(X_raw[i]) for i in range(N)]
        ).astype(np.float32)  # (N, T, n_states)
        return posteriors

    X_train_post = build_posterior_seqs(X_train_raw)
    X_val_post = build_posterior_seqs(X_val_raw)

    return X_train_post, X_val_post


class HMMPosteriorSeqTransformer(nn.Module):
    """
    Token axis = TIMEPOINTS (in order, T of them). Each token's feature
    is that timepoint's HMM state-posterior vector (n_states-dim),
    instead of the raw n_rois-dim signal.
    """
    def __init__(self, n_states, seq_len, d_model=16, n_heads=2, n_layers=1,
                 dim_ff=32, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_states, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(seq_len, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, T, n_states)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_hmm_posterior_seq(X_train_post, y_train, X_val_post, y_val,
                                      n_states, seq_len, epochs=60, lr=1e-4):
    model = HMMPosteriorSeqTransformer(n_states=n_states, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_post).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_post).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_hmm_posterior_seq_transformer(X, y, skf, n_states=6):
    """
    For each fold: fit the HMM on that fold's TRAINING subjects only,
    compute per-timepoint posterior SEQUENCES for train and val
    subjects using that fitted model, then train/evaluate the
    time-token transformer on those sequences.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    seq_len = X.shape[1]

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        print(f"\n--- Fold {fold+1}: fitting HMM (n_states={n_states}) ---")
        X_train_post, X_val_post = compute_hmm_posterior_sequences(
            X_train_raw, X_val_raw, n_states=n_states
        )

        model, auc = train_one_fold_hmm_posterior_seq(
            X_train_post, y_train, X_val_post, y_val,
            n_states=n_states, seq_len=seq_len
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_post).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== HMM Posterior-Sequence Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training HMM Posterior-Sequence Transformer (n_states=6)")
print("=" * 60)# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> HMM State-Specific FC Transformer
# =====================================================



In [ ]:
# =====================================================
# Load ABIDE ROI Time Series (Corrected)
# =====================================================

import os
import glob
import numpy as np
from sklearn.preprocessing import StandardScaler

ASD_DATA_PATH = "/content/drive/MyDrive/FC/entiredataset_rois_aal/asd/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/entiredataset_rois_aal/td/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 116
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

# =====================================================
# AAL-116 Network definitions (0-based indexing)
# =====================================================

NETWORKS = {
    "WHOLE_BRAIN": list(range(116)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("ROI Indices      :", SELECTED_ROIS)
print("=" * 60)


def parse_site_from_filename(name):
    """
    Extract actual site name from filename regardless of SITE_FILTER.
    ABIDE filenames typically look like: NYU_0051091_rois_aal.1D
    """
    return name.split("_")[0]


def load_subjects(folder, label):
    """
    Loads all .1D ROI time series from `folder`, applies:
      - site filtering
      - ROI selection
      - per-ROI z-scoring
      - resample-to-TARGET_TIME (interpolation if too long, zero-pad if too short)
    Returns a dict of lists so nothing is silently misaligned.
    """
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        # site filtering (always based on actual parsed site, not the filter string)
        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            # handle (ROI, time) vs (time, ROI) orientation
            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            # z-score each ROI's time series (per-subject standardization)
            ts = StandardScaler().fit_transform(ts)

            # select ROI subset
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                # interpolate down to TARGET_TIME — real_length stays TARGET_TIME
                # since every resulting timestep is real signal (no padding)
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                # zero-pad; real_length marks how much is genuine signal
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]  # unchanged from before padding

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


# ---------------------------------------------------
# Load ASD and TD groups
# ---------------------------------------------------
asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Labels shape   :", y.shape)
print("Unique sites   :", sorted(set(sites)))
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))

if len(X):
    print("Example subject:", subject_ids[0], "| shape:", X[0].shape, "| real_length:", real_lengths[0])

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> SVM-FC baseline -> MI ROI selection ->
# ROI-token Transformer (with MI-selected ROIs)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import spearmanr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)


# =====================================================
# 5. ROI-TOKEN TRANSFORMER
# =====================================================
class fMRITransformerROIToken(nn.Module):
    def __init__(self, n_rois, seq_len=178, d_model=64, n_heads=4,
                 n_layers=2, dim_ff=128, dropout=0.3, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(seq_len, d_model)
        self.roi_embedding = nn.Parameter(torch.randn(1, n_rois, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_rois, seq_len)
        h = self.input_proj(x)
        h = h + self.roi_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


# =====================================================
# 5b. ROI-TOKEN TRANSFORMER -- STANDALONE EVAL (ALL ROIs, no selection)
# =====================================================
def make_padding_mask(real_lengths, target_time, device):
    if real_lengths is None:
        return None
    B = len(real_lengths)
    mask = torch.zeros(B, target_time, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < target_time:
            mask[i, l:] = True
    return mask.to(device)


def train_one_fold_roi_token(X_train, y_train, X_val, y_val,
                              n_rois, seq_len, epochs=60, lr=1e-4):
    """
    Trains the ROI-token transformer for one CV fold.
    X_train / X_val here are (N, T, n_rois) -- transposed to
    (N, n_rois, T) internally so each ROI's full timecourse
    becomes one token's feature vector.
    """
    model = fMRITransformerROIToken(n_rois=n_rois, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # transpose (N, T, n_rois) -> (N, n_rois, T): ROI becomes the token axis
    Xtr = torch.tensor(X_train.transpose(0, 2, 1)).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_roi_token_transformer(X, y, skf):
    """
    Standalone ROI-as-token transformer evaluation, using ALL ROIs
    (no MI-based feature selection) -- a clean baseline for the
    architecture itself before any selection is layered on top.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=X_train.shape[2], seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(n_rois={X_train.shape[2]})")

    print("\n===== ROI-Token Transformer (ALL ROIs, no selection): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer (ALL 116 ROIs, no selection)")
print("=" * 60)
transformer_roitoken_results = evaluate_roi_token_transformer(X, y, skf)


# =====================================================
# 6. MUTUAL-INFORMATION ROI SELECTION (fold-safe)
# =====================================================
def select_top_rois_by_mutual_info(X_train, y_train, k=None):
    """
    Ranks ROIs by mutual information with the label, using ONLY
    the training subjects passed in (must be called per-fold on
    that fold's training split -- never on the full dataset).

    X_train: (N_train, T, n_rois)
    Returns: (top_roi_idx sorted ascending, full mi_scores array)
    """
    N, T, R = X_train.shape
    # summary feature per ROI per subject: mean activity over time
    roi_features = X_train.mean(axis=1)  # (N, R)

    mi_scores = mutual_info_classif(roi_features, y_train, random_state=42)

    if k is None:
        k = R  # return all, ranked

    top_roi_idx = np.argsort(mi_scores)[::-1][:k]
    return sorted(top_roi_idx.tolist()), mi_scores


# =====================================================
# 7. STABILITY CHECK -- run before trusting the selection
# =====================================================
def check_mi_stability(X, y, skf):
    mi_scores_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full = X[tr_idx]
        y_train = y[tr_idx]
        _, mi_scores = select_top_rois_by_mutual_info(X_train_full, y_train, k=None)
        mi_scores_per_fold.append(mi_scores)

    mi_scores_per_fold = np.array(mi_scores_per_fold)  # (n_folds, n_rois)

    print("\n===== MI Stability Check: Spearman rank correlation across folds =====")
    rhos = []
    n_folds = mi_scores_per_fold.shape[0]
    for i in range(n_folds):
        for j in range(i + 1, n_folds):
            rho, _ = spearmanr(mi_scores_per_fold[i], mi_scores_per_fold[j])
            rhos.append(rho)
            print(f"Fold {i+1} vs Fold {j+1}: rho = {rho:.3f}")

    print(f"\nMean pairwise rho: {np.mean(rhos):.3f} ± {np.std(rhos):.3f}")
    if np.mean(rhos) > 0.6:
        print("-> ROI importance ranking looks STABLE across folds.")
    elif np.mean(rhos) > 0.3:
        print("-> ROI importance ranking is MODERATELY stable -- proceed cautiously.")
    else:
        print("-> ROI importance ranking is UNSTABLE -- selection may not generalize well.")

    return mi_scores_per_fold


print("\n" + "=" * 60)
print("Checking MI-based ROI selection stability across folds")
print("=" * 60)
mi_stability_scores = check_mi_stability(X, y, skf)


# =====================================================
# 8. TRANSFORMER TRAINING LOOP + MI SELECTION (per fold, ROI-token)
# =====================================================
# NOTE: make_padding_mask and train_one_fold_roi_token are already
# defined in section 5b above and reused here as-is.

def evaluate_roi_token_transformer_with_mi_selection(X, y, real_lengths, skf, k=40):
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_rois_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full, X_val_full = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        # --- MI selection using ONLY this fold's training subjects ---
        top_roi_idx, _ = select_top_rois_by_mutual_info(X_train_full, y_train, k=k)
        selected_rois_per_fold.append(top_roi_idx)

        X_train = X_train_full[:, :, top_roi_idx]
        X_val = X_val_full[:, :, top_roi_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=len(top_roi_idx), seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(k={len(top_roi_idx)} ROIs)")

    print(f"\n===== ROI-Token Transformer + MI Selection (k={k}): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    # overlap check: how consistent is the actual selected ROI SET across folds
    print("\n--- ROI selection overlap across folds (Jaccard-style) ---")
    for i in range(len(selected_rois_per_fold)):
        for j in range(i + 1, len(selected_rois_per_fold)):
            set_i, set_j = set(selected_rois_per_fold[i]), set(selected_rois_per_fold[j])
            overlap = len(set_i & set_j) / len(set_i | set_j)
            print(f"Fold {i+1} vs Fold {j+1}: {len(set_i & set_j)}/{k} ROIs shared "
                  f"(Jaccard={overlap:.3f})")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs, "selected_rois_per_fold": selected_rois_per_fold}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer with MI-based ROI selection (k=40)")
print("=" * 60)
transformer_mi_results = evaluate_roi_token_transformer_with_mi_selection(
    X, y, real_lengths, skf, k=40
)


# =====================================================
# 9. FINAL SUMMARY COMPARISON
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs ROI-Token Transformer (all ROIs) vs ROI-Token Transformer (MI-selected)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'Transformer (all)':<22}{'Transformer (MI)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]);          svm_std = np.std(svm_results[metric])
    tf_all_mean = np.mean(transformer_roitoken_results[metric]); tf_all_std = np.std(transformer_roitoken_results[metric])
    tf_mi_mean = np.mean(transformer_mi_results[metric]);  tf_mi_std = np.std(transformer_mi_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      "
          f"{tf_all_mean:.3f} ± {tf_all_std:.3f}      "
          f"{tf_mi_mean:.3f} ± {tf_mi_std:.3f}")


# =====================================================
# 10. FC-PROFILE-AS-TOKEN TRANSFORMER
# =====================================================
# Instead of feeding each ROI's RAW timecourse as its token (fragile to
# scanner/site differences, as every experiment so far has shown), each
# ROI's token is now its FULL CONNECTIVITY PROFILE -- i.e. its row in the
# subject's FC matrix (correlation with every other ROI). This directly
# inherits the SVM-FC baseline's main strength (scale/site-invariant,
# correlation-based features) while still letting self-attention learn
# NONLINEAR combinations of connectivity patterns, which a linear SVM
# structurally cannot do.
#
# Input to the model changes from (N, n_rois, seq_len) -> (N, n_rois, n_rois)
# =====================================================

def compute_fc_matrix_per_subject(X, verbose=True):
    """
    X: (N, T, n_rois) raw (already per-ROI standardized) timeseries.
    Returns: (N, n_rois, n_rois) -- each subject's FULL FC matrix
             (not just the upper triangle -- every ROI's ROW becomes
             that ROI's token/feature vector for the transformer).
    """
    N, T, R = X.shape
    fc_matrices = np.zeros((N, R, R), dtype=np.float32)
    nan_subject_count = 0

    for i in range(N):
        fc = np.corrcoef(X[i].T)
        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)
        fc_matrices[i] = fc

    if verbose:
        print(f"Subjects with NaN in FC matrix (fixed via nan_to_num): {nan_subject_count} / {N}")

    return fc_matrices


class fMRITransformerFCToken(nn.Module):
    """
    Same overall architecture as fMRITransformerROIToken, except each
    ROI-token's input feature is its length-n_rois connectivity profile
    (a row of the FC matrix) instead of its raw seq_len-length timecourse.
    """
    def __init__(self, n_rois, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        # input dim is n_rois here (connectivity profile length),
        # NOT seq_len -- this is the key structural change
        self.input_proj = nn.Linear(n_rois, d_model)
        self.roi_embedding = nn.Parameter(torch.randn(1, n_rois, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_rois, n_rois) -- x[:, i, :] is ROI i's connectivity profile
        h = self.input_proj(x)
        h = h + self.roi_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_fc_token(X_train_fc, y_train, X_val_fc, y_val,
                             n_rois, epochs=60, lr=1e-4):
    """
    X_train_fc / X_val_fc: (N, n_rois, n_rois) FC matrices (already computed
    per-subject, per-fold-safe -- see evaluate_fc_token_transformer below).
    """
    model = fMRITransformerFCToken(n_rois=n_rois).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_fc).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_fc).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_fc_token_transformer(X, y, skf):
    """
    Trains/evaluates the FC-profile-as-token transformer, computing the
    FC matrix separately per fold split (train and val subjects use their
    OWN raw timeseries to build their own FC matrix -- no leakage, since
    each subject's FC matrix only depends on that subject's own signal,
    not on other subjects' labels or data).
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        # each subject's FC matrix depends only on their own timeseries --
        # safe to compute independently for train and val subjects
        X_train_fc = compute_fc_matrix_per_subject(X_train_raw, verbose=False)
        X_val_fc = compute_fc_matrix_per_subject(X_val_raw, verbose=False)

        model, auc = train_one_fold_fc_token(
            X_train_fc, y_train, X_val_fc, y_val, n_rois=X_train_fc.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_fc).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== FC-Profile-Token Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training FC-Profile-as-Token Transformer")
print("=" * 60)
transformer_fctoken_results = evaluate_fc_token_transformer(X, y, skf)


# =====================================================
# 12. DYNAMIC FC-WINDOW TRANSFORMER
# =====================================================
# Static FC (section 10) collapses the ENTIRE scan into one correlation
# matrix per subject -- robust to scanner/site noise, but throws away
# any information about how connectivity changes over the course of the
# scan (e.g. transient synchronization, state changes).
#
# Here, each subject's scan is split into overlapping time windows; FC is
# computed WITHIN each window (still a robust, correlation-based feature),
# and the SEQUENCE of window-FC vectors becomes the transformer's input.
# The token axis is now genuinely TEMPORAL again (windows in order), so
# positional encoding is reintroduced -- order matters here, unlike the
# ROI-token versions where ROI order was arbitrary.
#
# Net effect: keeps the SVM's robust correlation-based feature type, while
# recovering some of the temporal dynamics that a single static FC matrix
# cannot represent, and that the raw-signal transformer failed to exploit
# reliably (sections 5b/8) due to scale/site fragility.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed here because,
    unlike ROI tokens, WINDOW tokens have a genuine temporal order."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_dynamic_fc_windows(X, window_size=20, stride=10, verbose=True):
    """
    X: (N, T, n_rois) raw (already per-ROI standardized) timeseries.
    Returns: (N, n_windows, n_roi_pairs) -- for each subject, a SEQUENCE
             of flattened upper-triangle FC vectors, one per sliding window.

    n_windows is determined by T, window_size, stride and is IDENTICAL
    for every subject (since every subject shares the same T after your
    loader's resample/pad step) -- this keeps the sequence length fixed,
    which the transformer requires.
    """
    N, T, R = X.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    starts = list(range(0, T - window_size + 1, stride))
    n_windows = len(starts)

    if n_windows == 0:
        raise ValueError(
            f"window_size={window_size} is larger than seq_len={T} -- "
            f"reduce window_size or stride."
        )

    windows_fc = np.zeros((N, n_windows, n_pairs), dtype=np.float32)
    nan_window_count = 0

    for i in range(N):
        for w, start in enumerate(starts):
            seg = X[i, start:start + window_size, :]
            fc = np.corrcoef(seg.T)
            if np.isnan(fc).any():
                nan_window_count += 1
                fc = np.nan_to_num(fc, nan=0.0)
            windows_fc[i, w] = fc[triu_idx]

    if verbose:
        print(f"n_windows per subject : {n_windows} (window_size={window_size}, stride={stride})")
        print(f"n_roi_pairs per window: {n_pairs}")
        print(f"Windows with NaN FC (fixed via nan_to_num): {nan_window_count} / {N * n_windows}")

    return windows_fc


class fMRITransformerDynamicFC(nn.Module):
    """
    Token axis = time WINDOWS (in order), not ROIs. Each token's feature
    is that window's flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_windows, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(n_windows, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_windows, n_pairs)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_dynamic_fc(X_train_win, y_train, X_val_win, y_val,
                               n_pairs, n_windows, epochs=60, lr=1e-4):
    model = fMRITransformerDynamicFC(n_pairs=n_pairs, n_windows=n_windows).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_win).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_win).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_dynamic_fc_transformer(X, y, skf, window_size=30, stride=15):
    """
    Dynamic-FC-window features are computed ONCE up front (per subject,
    from that subject's own raw timeseries only -- no cross-subject
    statistics involved, so this is leakage-safe to precompute outside
    the fold loop, same reasoning as the static FC-token transformer).
    """
    print(f"\nComputing dynamic FC windows (window_size={window_size}, stride={stride})...")
    X_windows = compute_dynamic_fc_windows(X, window_size=window_size, stride=stride)
    n_windows = X_windows.shape[1]
    n_pairs = X_windows.shape[2]

    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_win, X_val_win = X_windows[tr_idx], X_windows[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_dynamic_fc(
            X_train_win, y_train, X_val_win, y_val,
            n_pairs=n_pairs, n_windows=n_windows
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_win).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Dynamic-FC-Window Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Dynamic-FC-Window Transformer")
print("=" * 60)
# window_size/stride are chosen relative to your seq_len (TARGET_TIME).
# With TARGET_TIME=78: window_size=20, stride=10 gives (78-20)/10+1 = 6 windows.
# With TARGET_TIME=178: same settings give (178-20)/10+1 = 16 windows.
# Adjust these if your TARGET_TIME changes -- too few windows (e.g. <4)
# defeats the purpose (not enough sequence to attend over); too many
# windows with small stride increases compute and overlap redundancy.
transformer_dynamicfc_results = evaluate_dynamic_fc_transformer(
    X, y, skf, window_size=20, stride=10
)


# =====================================================
# 13. UPDATED FINAL SUMMARY -- ALL FIVE MODELS
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs TF(raw,all) vs TF(raw,MI) vs TF(FC-token) vs TF(dynamic-FC)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<18}{'TF(raw,all)':<18}{'TF(raw,MI)':<18}{'TF(FC-tok)':<18}{'TF(dynFC)':<18}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]);          svm_std = np.std(svm_results[metric])
    tf_all_mean = np.mean(transformer_roitoken_results[metric]); tf_all_std = np.std(transformer_roitoken_results[metric])
    tf_mi_mean = np.mean(transformer_mi_results[metric]);  tf_mi_std = np.std(transformer_mi_results[metric])
    tf_fc_mean = np.mean(transformer_fctoken_results[metric]); tf_fc_std = np.std(transformer_fctoken_results[metric])
    tf_dyn_mean = np.mean(transformer_dynamicfc_results[metric]); tf_dyn_std = np.std(transformer_dynamicfc_results[metric])
    print(f"{metric:<15}{svm_mean:.3f}±{svm_std:.3f}    "
          f"{tf_all_mean:.3f}±{tf_all_std:.3f}    "
          f"{tf_mi_mean:.3f}±{tf_mi_std:.3f}    "
          f"{tf_fc_mean:.3f}±{tf_fc_std:.3f}    "
          f"{tf_dyn_mean:.3f}±{tf_dyn_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> SVM-FC baseline -> Dynamic FC-Window
# Transformer (+ grid search over window_size / stride)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)


# =====================================================
# 5. DYNAMIC FC-WINDOW TRANSFORMER
# =====================================================
# A static FC matrix collapses the ENTIRE scan into one correlation
# matrix per subject -- robust to scanner/site noise, but throws away
# any information about how connectivity changes over the course of the
# scan (e.g. transient synchronization, state changes).
#
# Here, each subject's scan is split into overlapping time windows; FC is
# computed WITHIN each window (still a robust, correlation-based feature),
# and the SEQUENCE of window-FC vectors becomes the transformer's input.
# The token axis is genuinely TEMPORAL (windows in order), so positional
# encoding is used -- order matters here.
#
# Net effect: keeps the SVM's robust correlation-based feature type,
# while recovering the temporal dynamics a single static FC matrix
# cannot represent.
# =====================================================

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding -- needed because WINDOW
    tokens have a genuine temporal order (unlike, e.g., ROI tokens)."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def compute_dynamic_fc_windows(X, window_size=20, stride=10, verbose=True):
    """
    X: (N, T, n_rois) raw (already per-ROI standardized) timeseries.
    Returns: (N, n_windows, n_roi_pairs) -- for each subject, a SEQUENCE
             of flattened upper-triangle FC vectors, one per sliding window.

    n_windows is determined by T, window_size, stride and is IDENTICAL
    for every subject (since every subject shares the same T after your
    loader's resample/pad step) -- this keeps the sequence length fixed,
    which the transformer requires.
    """
    N, T, R = X.shape
    triu_idx = np.triu_indices(R, k=1)
    n_pairs = len(triu_idx[0])

    starts = list(range(0, T - window_size + 1, stride))
    n_windows = len(starts)

    if n_windows == 0:
        raise ValueError(
            f"window_size={window_size} is larger than seq_len={T} -- "
            f"reduce window_size or stride."
        )

    windows_fc = np.zeros((N, n_windows, n_pairs), dtype=np.float32)
    nan_window_count = 0

    for i in range(N):
        for w, start in enumerate(starts):
            seg = X[i, start:start + window_size, :]
            fc = np.corrcoef(seg.T)
            if np.isnan(fc).any():
                nan_window_count += 1
                fc = np.nan_to_num(fc, nan=0.0)
            windows_fc[i, w] = fc[triu_idx]

    if verbose:
        print(f"n_windows per subject : {n_windows} (window_size={window_size}, stride={stride})")
        print(f"n_roi_pairs per window: {n_pairs}")
        print(f"Windows with NaN FC (fixed via nan_to_num): {nan_window_count} / {N * n_windows}")

    return windows_fc


class fMRITransformerDynamicFC(nn.Module):
    """
    Token axis = time WINDOWS (in order). Each token's feature is that
    window's flattened upper-triangle FC vector.
    """
    def __init__(self, n_pairs, n_windows, d_model=32, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_pairs, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max(n_windows, 16))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_windows, n_pairs)
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_dynamic_fc(X_train_win, y_train, X_val_win, y_val,
                               n_pairs, n_windows, epochs=60, lr=1e-4):
    model = fMRITransformerDynamicFC(n_pairs=n_pairs, n_windows=n_windows).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_win).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_win).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_dynamic_fc_transformer(X, y, skf, window_size=20, stride=10):
    """
    Dynamic-FC-window features are computed ONCE up front (per subject,
    from that subject's own raw timeseries only -- no cross-subject
    statistics involved, so this is leakage-safe to precompute outside
    the fold loop).
    """
    print(f"\nComputing dynamic FC windows (window_size={window_size}, stride={stride})...")
    X_windows = compute_dynamic_fc_windows(X, window_size=window_size, stride=stride)
    n_windows = X_windows.shape[1]
    n_pairs = X_windows.shape[2]

    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_win, X_val_win = X_windows[tr_idx], X_windows[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_dynamic_fc(
            X_train_win, y_train, X_val_win, y_val,
            n_pairs=n_pairs, n_windows=n_windows
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_win).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== Dynamic-FC-Window Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training Dynamic-FC-Window Transformer (default window_size=20, stride=10)")
print("=" * 60)
# window_size/stride are chosen relative to your seq_len (TARGET_TIME).
# With TARGET_TIME=178: window_size=20, stride=10 gives (178-20)/10+1 = 16 windows.
# Adjust these if your TARGET_TIME changes -- too few windows (e.g. <4)
# defeats the purpose (not enough sequence to attend over); too many
# windows with small stride increases compute and overlap redundancy.
transformer_dynamicfc_results = evaluate_dynamic_fc_transformer(
    X, y, skf, window_size=20, stride=10
)


# =====================================================
# 6. FINAL SUMMARY -- SVM-FC vs DYNAMIC-FC TRANSFORMER (default params)
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs Dynamic-FC-Window Transformer (default params)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'TF(dynamic-FC)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]); svm_std = np.std(svm_results[metric])
    tf_mean = np.mean(transformer_dynamicfc_results[metric]); tf_std = np.std(transformer_dynamicfc_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      {tf_mean:.3f} ± {tf_std:.3f}")


# =====================================================
# 7. GRID SEARCH -- BEST (window_size, stride) FOR DYNAMIC-FC TRANSFORMER
# =====================================================
# Reuses evaluate_dynamic_fc_transformer as-is for each candidate
# (window_size, stride) pair, on the SAME skf folds each time so results
# are directly comparable. Selection is based on mean AUC across folds.
#
# IMPORTANT CAVEAT: because this search reuses validation-fold AUC to
# pick the winning (window_size, stride), the reported AUC for the BEST
# combination is optimistically biased (the same folds used to pick the
# winner are also used to report its score). Treat the winning
# combination's exact number as an upper estimate, not a final
# generalization guarantee -- if this needs to go in a report, ideally
# re-validate the chosen (window_size, stride) on a fresh held-out split
# not used in the search.
# =====================================================

def grid_search_dynamic_fc(X, y, skf, window_sizes, strides, min_windows=4):
    """
    Tries every (window_size, stride) combination from the given lists.
    Skips combinations that would produce fewer than `min_windows`
    windows (too short a sequence for attention to do anything useful)
    or where window_size > seq_len (invalid).

    Returns a list of result dicts, one per valid combination tried,
    each containing the params and the full metric results, sorted by
    mean AUC descending.
    """
    seq_len = X.shape[1]
    grid_results = []

    for window_size in window_sizes:
        for stride in strides:
            if window_size > seq_len:
                print(f"Skipping window_size={window_size} (> seq_len={seq_len})")
                continue

            n_windows = len(range(0, seq_len - window_size + 1, stride))
            if n_windows < min_windows:
                print(f"Skipping window_size={window_size}, stride={stride} "
                      f"-> only {n_windows} windows (< min_windows={min_windows})")
                continue

            print("\n" + "-" * 60)
            print(f"Trying window_size={window_size}, stride={stride} "
                  f"(-> {n_windows} windows)")
            print("-" * 60)

            results = evaluate_dynamic_fc_transformer(
                X, y, skf, window_size=window_size, stride=stride
            )

            grid_results.append({
                "window_size": window_size,
                "stride": stride,
                "n_windows": n_windows,
                "auc_mean": np.mean(results["auc"]),
                "auc_std": np.std(results["auc"]),
                "sensitivity_mean": np.mean(results["sensitivity"]),
                "specificity_mean": np.mean(results["specificity"]),
                "f1_mean": np.mean(results["f1"]),
                "accuracy_mean": np.mean(results["accuracy"]),
                "full_results": results,
            })

    grid_results.sort(key=lambda r: r["auc_mean"], reverse=True)
    return grid_results


def print_grid_search_summary(grid_results):
    print("\n" + "=" * 60)
    print("GRID SEARCH SUMMARY (sorted by mean AUC, best first)")
    print("=" * 60)
    header = f"{'window':<8}{'stride':<8}{'n_win':<7}{'AUC':<16}{'Sens':<8}{'Spec':<8}{'F1':<8}{'Acc':<8}"
    print(header)
    for r in grid_results:
        print(f"{r['window_size']:<8}{r['stride']:<8}{r['n_windows']:<7}"
              f"{r['auc_mean']:.3f}±{r['auc_std']:.3f}    "
              f"{r['sensitivity_mean']:.3f}   "
              f"{r['specificity_mean']:.3f}   "
              f"{r['f1_mean']:.3f}   "
              f"{r['accuracy_mean']:.3f}")

    if grid_results:
        best = grid_results[0]
        print(f"\nBest combination: window_size={best['window_size']}, "
              f"stride={best['stride']}  (AUC={best['auc_mean']:.3f} ± {best['auc_std']:.3f})")
        print("NOTE: this AUC is selected FROM this same grid search's fold "
              "results -- re-validate on a fresh split before treating it as "
              "a final generalization estimate (see caveat above section 7).")


# --- Candidate grids ---
# Adjust these based on your actual seq_len (TARGET_TIME). For seq_len=178,
# window_sizes up to ~40 and strides down to ~5 are reasonable.
# min_windows=4 filters out combinations that would leave too short a
# sequence to be useful.
WINDOW_SIZES = [10, 15, 20, 30, 40]
STRIDES = [5, 10, 15, 20]

print("\n" + "=" * 60)
print("GRID SEARCH: Dynamic-FC-Window Transformer (window_size x stride)")
print("=" * 60)
dynamic_fc_grid_results = grid_search_dynamic_fc(
    X, y, skf, window_sizes=WINDOW_SIZES, strides=STRIDES, min_windows=4
)
print_grid_search_summary(dynamic_fc_grid_results)

In [ ]:
# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 178
SITE_FILTER = "NYU"   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))

print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across both models)
# =====================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> SVM-FC baseline -> MI ROI selection ->
# ROI-token Transformer (with MI-selected ROIs)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import spearmanr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)


# =====================================================
# 5. ROI-TOKEN TRANSFORMER
# =====================================================
class fMRITransformerROIToken(nn.Module):
    def __init__(self, n_rois, seq_len=78, d_model=64, n_heads=2,
                 n_layers=1, dim_ff=128, dropout=0.5, n_classes=2):
        #fMRITransformerROIToken(n_rois=116, d_model=16, n_heads=2, n_layers=1, dim_ff=32, dropout=0.5)
        super().__init__()
        self.input_proj = nn.Linear(seq_len, d_model)
        self.roi_embedding = nn.Parameter(torch.randn(1, n_rois, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_rois, seq_len)
        h = self.input_proj(x)
        h = h + self.roi_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


# =====================================================
# 5b. ROI-TOKEN TRANSFORMER -- STANDALONE EVAL (ALL ROIs, no selection)
# =====================================================
def make_padding_mask(real_lengths, target_time, device):
    if real_lengths is None:
        return None
    B = len(real_lengths)
    mask = torch.zeros(B, target_time, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < target_time:
            mask[i, l:] = True
    return mask.to(device)


def train_one_fold_roi_token(X_train, y_train, X_val, y_val,
                              n_rois, seq_len, epochs=60, lr=1e-4):
    """
    Trains the ROI-token transformer for one CV fold.
    X_train / X_val here are (N, T, n_rois) -- transposed to
    (N, n_rois, T) internally so each ROI's full timecourse
    becomes one token's feature vector.
    """
    model = fMRITransformerROIToken(n_rois=n_rois, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # transpose (N, T, n_rois) -> (N, n_rois, T): ROI becomes the token axis
    Xtr = torch.tensor(X_train.transpose(0, 2, 1)).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_roi_token_transformer(X, y, skf):
    """
    Standalone ROI-as-token transformer evaluation, using ALL ROIs
    (no MI-based feature selection) -- a clean baseline for the
    architecture itself before any selection is layered on top.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=X_train.shape[2], seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(n_rois={X_train.shape[2]})")

    print("\n===== ROI-Token Transformer (ALL ROIs, no selection): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer (ALL 116 ROIs, no selection)")
print("=" * 60)
transformer_roitoken_results = evaluate_roi_token_transformer(X, y, skf)


# =====================================================
# 6. MUTUAL-INFORMATION ROI SELECTION (fold-safe)
# =====================================================
def select_top_rois_by_mutual_info(X_train, y_train, k=None):
    """
    Ranks ROIs by mutual information with the label, using ONLY
    the training subjects passed in (must be called per-fold on
    that fold's training split -- never on the full dataset).

    X_train: (N_train, T, n_rois)
    Returns: (top_roi_idx sorted ascending, full mi_scores array)
    """
    N, T, R = X_train.shape
    # summary feature per ROI per subject: mean activity over time
    roi_features = X_train.mean(axis=1)  # (N, R)

    mi_scores = mutual_info_classif(roi_features, y_train, random_state=42)

    if k is None:
        k = R  # return all, ranked

    top_roi_idx = np.argsort(mi_scores)[::-1][:k]
    return sorted(top_roi_idx.tolist()), mi_scores


# =====================================================
# 7. STABILITY CHECK -- run before trusting the selection
# =====================================================
def check_mi_stability(X, y, skf):
    mi_scores_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full = X[tr_idx]
        y_train = y[tr_idx]
        _, mi_scores = select_top_rois_by_mutual_info(X_train_full, y_train, k=None)
        mi_scores_per_fold.append(mi_scores)

    mi_scores_per_fold = np.array(mi_scores_per_fold)  # (n_folds, n_rois)

    print("\n===== MI Stability Check: Spearman rank correlation across folds =====")
    rhos = []
    n_folds = mi_scores_per_fold.shape[0]
    for i in range(n_folds):
        for j in range(i + 1, n_folds):
            rho, _ = spearmanr(mi_scores_per_fold[i], mi_scores_per_fold[j])
            rhos.append(rho)
            print(f"Fold {i+1} vs Fold {j+1}: rho = {rho:.3f}")

    print(f"\nMean pairwise rho: {np.mean(rhos):.3f} ± {np.std(rhos):.3f}")
    if np.mean(rhos) > 0.6:
        print("-> ROI importance ranking looks STABLE across folds.")
    elif np.mean(rhos) > 0.3:
        print("-> ROI importance ranking is MODERATELY stable -- proceed cautiously.")
    else:
        print("-> ROI importance ranking is UNSTABLE -- selection may not generalize well.")

    return mi_scores_per_fold


print("\n" + "=" * 60)
print("Checking MI-based ROI selection stability across folds")
print("=" * 60)
mi_stability_scores = check_mi_stability(X, y, skf)


# =====================================================
# 8. TRANSFORMER TRAINING LOOP (per fold, ROI-token)
# =====================================================
def make_padding_mask(real_lengths, target_time, device):
    if real_lengths is None:
        return None
    B = len(real_lengths)
    mask = torch.zeros(B, target_time, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < target_time:
            mask[i, l:] = True
    return mask.to(device)


def train_one_fold_roi_token(X_train, y_train, X_val, y_val,
                              n_rois, seq_len, epochs=60, lr=1e-4):
    model = fMRITransformerROIToken(n_rois=n_rois, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # transpose to (B, n_rois, seq_len) for ROI-token input
    Xtr = torch.tensor(X_train.transpose(0, 2, 1)).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_roi_token_transformer_with_mi_selection(X, y, real_lengths, skf, k=40):
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_rois_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full, X_val_full = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        # --- MI selection using ONLY this fold's training subjects ---
        top_roi_idx, _ = select_top_rois_by_mutual_info(X_train_full, y_train, k=k)
        selected_rois_per_fold.append(top_roi_idx)

        X_train = X_train_full[:, :, top_roi_idx]
        X_val = X_val_full[:, :, top_roi_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=len(top_roi_idx), seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(k={len(top_roi_idx)} ROIs)")

    print(f"\n===== ROI-Token Transformer + MI Selection (k={k}): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    # overlap check: how consistent is the actual selected ROI SET across folds
    print("\n--- ROI selection overlap across folds (Jaccard-style) ---")
    for i in range(len(selected_rois_per_fold)):
        for j in range(i + 1, len(selected_rois_per_fold)):
            set_i, set_j = set(selected_rois_per_fold[i]), set(selected_rois_per_fold[j])
            overlap = len(set_i & set_j) / len(set_i | set_j)
            print(f"Fold {i+1} vs Fold {j+1}: {len(set_i & set_j)}/{k} ROIs shared "
                  f"(Jaccard={overlap:.3f})")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs, "selected_rois_per_fold": selected_rois_per_fold}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer with MI-based ROI selection (k=40)")
print("=" * 60)
transformer_mi_results = evaluate_roi_token_transformer_with_mi_selection(
    X, y, real_lengths, skf, k=40
)


# =====================================================
# 9. FINAL SUMMARY COMPARISON
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC Baseline vs ROI-Token Transformer (MI-selected)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'Transformer (MI)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric])
    svm_std = np.std(svm_results[metric])
    tf_mean = np.mean(transformer_mi_results[metric])
    tf_std = np.std(transformer_mi_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      {tf_mean:.3f} ± {tf_std:.3f}")

In [ ]:
# =====================================================
# fMRI ABIDE ROI-based ASD vs TD Classification Pipeline
# Data loader -> SVM-FC baseline -> MI ROI selection ->
# ROI-token Transformer (with MI-selected ROIs)
# =====================================================

# =====================================================
# 1. IMPORTS
# =====================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import spearmanr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =====================================================
# 2. DATA LOADER (corrected version)
# =====================================================
ASD_DATA_PATH = "/content/drive/MyDrive/FC/rois_cc200/ASD/ABIDE_pcp/cpac/nofilt_noglobal/"
TD_DATA_PATH  = "/content/drive/MyDrive/FC/rois_cc200/TD/ABIDE_pcp/cpac/nofilt_noglobal/"

N_ROIS = 200
TARGET_TIME = 78
SITE_FILTER = None   # set to None to load all sites

NETWORKS = {
    "WHOLE_BRAIN": list(range(200)),
    "DMN":     [22,23,24,25,30,31,34,35,64,65,66,67,84,85,86,87],
    "SN":      [28,29,30,31,36,37,40,41],
    "ECN":     [6,7,12,13,60,61,62,63],
    "DAN":     [2,3,58,59,66,67],
    "SMN":     [0,1,56,57,68,69],
    "VISUAL":  [42,43,44,45,46,47,48,49,50,51,52,53,54,55],
    "LIMBIC":  [36,37,38,39,40,41],
    "SOCIAL":  [12,13,40,41,54,55,64,65,72,73,84,85,86,87],
}

NETWORKS_COMBINED = {
    "ASD_CORE": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] + NETWORKS["ECN"]
    )),
    "SOCIAL_DMN_SN": sorted(set(
        NETWORKS["DMN"] + NETWORKS["SN"] + NETWORKS["SOCIAL"] +
        NETWORKS["LIMBIC"] + NETWORKS["VISUAL"] + NETWORKS["SMN"] +
        NETWORKS["DAN"] + NETWORKS["ECN"]
    )),
}

SELECTED_NETWORK = "WHOLE_BRAIN"

if SELECTED_NETWORK in NETWORKS:
    SELECTED_ROIS = NETWORKS[SELECTED_NETWORK]
elif SELECTED_NETWORK in NETWORKS_COMBINED:
    SELECTED_ROIS = NETWORKS_COMBINED[SELECTED_NETWORK]
else:
    raise ValueError(f"Unknown network: {SELECTED_NETWORK}")

if len(SELECTED_ROIS) == 0:
    raise ValueError("SELECTED_ROIS is empty — check NETWORKS definition")

print("=" * 60)
print("Selected Network :", SELECTED_NETWORK)
print("Number of ROIs   :", len(SELECTED_ROIS))
print("=" * 60)


def parse_site_from_filename(name):
    return name.split("_")[0]


def load_subjects(folder, label):
    results = {
        "X": [], "y": [], "subject_ids": [],
        "sites": [], "real_lengths": []
    }

    files = sorted(glob.glob(os.path.join(folder, "*.1D")))
    if len(files) == 0:
        print(f"WARNING: no .1D files found in {folder}")
        return results

    for file in files:
        name = os.path.basename(file)
        site = parse_site_from_filename(name)

        if SITE_FILTER and SITE_FILTER.upper() not in site.upper():
            continue

        try:
            ts = np.loadtxt(file)

            if ts.ndim != 2:
                print("Skipping (bad ndim):", name, ts.shape)
                continue

            if ts.shape[0] == N_ROIS and ts.shape[1] != N_ROIS:
                ts = ts.T

            if ts.shape[1] != N_ROIS:
                print("Skipping (ROI mismatch):", name, ts.shape)
                continue

            ts = StandardScaler().fit_transform(ts)
            ts = ts[:, SELECTED_ROIS]

            real_length = ts.shape[0]

            if ts.shape[0] >= TARGET_TIME:
                idx = np.linspace(0, ts.shape[0] - 1, TARGET_TIME, dtype=int)
                ts = ts[idx]
                real_length = TARGET_TIME
            else:
                pad = np.zeros((TARGET_TIME - ts.shape[0], len(SELECTED_ROIS)), dtype=np.float32)
                ts = np.vstack((ts, pad))
                real_length = ts.shape[0] - pad.shape[0]

            results["X"].append(ts.astype(np.float32))
            results["y"].append(label)
            results["subject_ids"].append(name)
            results["sites"].append(site)
            results["real_lengths"].append(real_length)

        except Exception as e:
            print(f"ERROR loading {name}: {e}")

    return results


asd_data = load_subjects(ASD_DATA_PATH, 1)
td_data  = load_subjects(TD_DATA_PATH, 0)

X = np.asarray(asd_data["X"] + td_data["X"], dtype=np.float32)
y = np.asarray(asd_data["y"] + td_data["y"], dtype=np.int32)
subject_ids  = asd_data["subject_ids"] + td_data["subject_ids"]
sites        = asd_data["sites"] + td_data["sites"]
real_lengths = np.asarray(asd_data["real_lengths"] + td_data["real_lengths"], dtype=np.int32)

print("\n================ DATASET SUMMARY ================")
print("Network        :", SELECTED_NETWORK)
print("ASD subjects   :", len(asd_data["X"]))
print("TD subjects    :", len(td_data["X"]))
print("Total subjects :", len(X))
print("Shape          :", X.shape)
print("Padded subjects:", int((real_lengths < TARGET_TIME).sum()), "/", len(X))
print("Unique sites   :", sorted(set(sites)))
print("\nNaNs in X:", np.isnan(X).any(), "| Infs in X:", np.isinf(X).any())


# =====================================================
# 3. STRATIFIED K-FOLD SETUP (shared across all models)
# =====================================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# =====================================================
# 4. SVM-FC BASELINE
# =====================================================
def compute_fc_features(X, verbose=True):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))

    nan_subject_count = 0
    zero_var_roi_counts = []

    for i in range(N):
        ts = X[i]
        stds = ts.std(axis=0)
        zero_var_rois = np.where(stds < 1e-8)[0]
        zero_var_roi_counts.append(len(zero_var_rois))

        fc = np.corrcoef(ts.T)

        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)

        fc_feats[i] = fc[triu_idx]

    if verbose:
        print(f"Subjects with NaN in FC matrix : {nan_subject_count} / {N}")
        print(f"Avg zero-variance ROIs/subject : {np.mean(zero_var_roi_counts):.2f}")
        print(f"Max zero-variance ROIs (any 1) : {np.max(zero_var_roi_counts)}")

    return fc_feats


def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Computing FC features + running SVM baseline")
print("=" * 60)
fc_X = compute_fc_features(X)
assert not np.isnan(fc_X).any(), "Still NaNs after sanitizing — investigate further"
svm_results = evaluate_fc_svm(fc_X, y, skf)


# =====================================================
# 5. ROI-TOKEN TRANSFORMER
# =====================================================
class fMRITransformerROIToken(nn.Module):
    def __init__(self, n_rois, seq_len=178, d_model=64, n_heads=4,
                 n_layers=2, dim_ff=128, dropout=0.3, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(seq_len, d_model)
        self.roi_embedding = nn.Parameter(torch.randn(1, n_rois, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_rois, seq_len)
        h = self.input_proj(x)
        h = h + self.roi_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


# =====================================================
# 5b. ROI-TOKEN TRANSFORMER -- STANDALONE EVAL (ALL ROIs, no selection)
# =====================================================
def make_padding_mask(real_lengths, target_time, device):
    if real_lengths is None:
        return None
    B = len(real_lengths)
    mask = torch.zeros(B, target_time, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < target_time:
            mask[i, l:] = True
    return mask.to(device)


def train_one_fold_roi_token(X_train, y_train, X_val, y_val,
                              n_rois, seq_len, epochs=60, lr=1e-4):
    """
    Trains the ROI-token transformer for one CV fold.
    X_train / X_val here are (N, T, n_rois) -- transposed to
    (N, n_rois, T) internally so each ROI's full timecourse
    becomes one token's feature vector.
    """
    model = fMRITransformerROIToken(n_rois=n_rois, seq_len=seq_len).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # transpose (N, T, n_rois) -> (N, n_rois, T): ROI becomes the token axis
    Xtr = torch.tensor(X_train.transpose(0, 2, 1)).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_roi_token_transformer(X, y, skf):
    """
    Standalone ROI-as-token transformer evaluation, using ALL ROIs
    (no MI-based feature selection) -- a clean baseline for the
    architecture itself before any selection is layered on top.
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=X_train.shape[2], seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(n_rois={X_train.shape[2]})")

    print("\n===== ROI-Token Transformer (ALL ROIs, no selection): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer (ALL 116 ROIs, no selection)")
print("=" * 60)
transformer_roitoken_results = evaluate_roi_token_transformer(X, y, skf)


# =====================================================
# 6. MUTUAL-INFORMATION ROI SELECTION (fold-safe)
# =====================================================
def select_top_rois_by_mutual_info(X_train, y_train, k=None):
    """
    Ranks ROIs by mutual information with the label, using ONLY
    the training subjects passed in (must be called per-fold on
    that fold's training split -- never on the full dataset).

    X_train: (N_train, T, n_rois)
    Returns: (top_roi_idx sorted ascending, full mi_scores array)
    """
    N, T, R = X_train.shape
    # summary feature per ROI per subject: mean activity over time
    roi_features = X_train.mean(axis=1)  # (N, R)

    mi_scores = mutual_info_classif(roi_features, y_train, random_state=42)

    if k is None:
        k = R  # return all, ranked

    top_roi_idx = np.argsort(mi_scores)[::-1][:k]
    return sorted(top_roi_idx.tolist()), mi_scores


# =====================================================
# 7. STABILITY CHECK -- run before trusting the selection
# =====================================================
def check_mi_stability(X, y, skf):
    mi_scores_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full = X[tr_idx]
        y_train = y[tr_idx]
        _, mi_scores = select_top_rois_by_mutual_info(X_train_full, y_train, k=None)
        mi_scores_per_fold.append(mi_scores)

    mi_scores_per_fold = np.array(mi_scores_per_fold)  # (n_folds, n_rois)

    print("\n===== MI Stability Check: Spearman rank correlation across folds =====")
    rhos = []
    n_folds = mi_scores_per_fold.shape[0]
    for i in range(n_folds):
        for j in range(i + 1, n_folds):
            rho, _ = spearmanr(mi_scores_per_fold[i], mi_scores_per_fold[j])
            rhos.append(rho)
            print(f"Fold {i+1} vs Fold {j+1}: rho = {rho:.3f}")

    print(f"\nMean pairwise rho: {np.mean(rhos):.3f} ± {np.std(rhos):.3f}")
    if np.mean(rhos) > 0.6:
        print("-> ROI importance ranking looks STABLE across folds.")
    elif np.mean(rhos) > 0.3:
        print("-> ROI importance ranking is MODERATELY stable -- proceed cautiously.")
    else:
        print("-> ROI importance ranking is UNSTABLE -- selection may not generalize well.")

    return mi_scores_per_fold


print("\n" + "=" * 60)
print("Checking MI-based ROI selection stability across folds")
print("=" * 60)
mi_stability_scores = check_mi_stability(X, y, skf)


# =====================================================
# 8. TRANSFORMER TRAINING LOOP + MI SELECTION (per fold, ROI-token)
# =====================================================
# NOTE: make_padding_mask and train_one_fold_roi_token are already
# defined in section 5b above and reused here as-is.

def evaluate_roi_token_transformer_with_mi_selection(X, y, real_lengths, skf, k=40):
    aucs, senss, specs, f1s, accs = [], [], [], [], []
    selected_rois_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_full, X_val_full = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        # --- MI selection using ONLY this fold's training subjects ---
        top_roi_idx, _ = select_top_rois_by_mutual_info(X_train_full, y_train, k=k)
        selected_rois_per_fold.append(top_roi_idx)

        X_train = X_train_full[:, :, top_roi_idx]
        X_val = X_val_full[:, :, top_roi_idx]

        model, auc = train_one_fold_roi_token(
            X_train, y_train, X_val, y_val,
            n_rois=len(top_roi_idx), seq_len=X_train.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val.transpose(0, 2, 1)).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}  "
              f"(k={len(top_roi_idx)} ROIs)")

    print(f"\n===== ROI-Token Transformer + MI Selection (k={k}): Mean ± Std =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    # overlap check: how consistent is the actual selected ROI SET across folds
    print("\n--- ROI selection overlap across folds (Jaccard-style) ---")
    for i in range(len(selected_rois_per_fold)):
        for j in range(i + 1, len(selected_rois_per_fold)):
            set_i, set_j = set(selected_rois_per_fold[i]), set(selected_rois_per_fold[j])
            overlap = len(set_i & set_j) / len(set_i | set_j)
            print(f"Fold {i+1} vs Fold {j+1}: {len(set_i & set_j)}/{k} ROIs shared "
                  f"(Jaccard={overlap:.3f})")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs, "selected_rois_per_fold": selected_rois_per_fold}


print("\n" + "=" * 60)
print("Training ROI-Token Transformer with MI-based ROI selection (k=40)")
print("=" * 60)
transformer_mi_results = evaluate_roi_token_transformer_with_mi_selection(
    X, y, real_lengths, skf, k=40
)


# =====================================================
# 9. FINAL SUMMARY COMPARISON
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs ROI-Token Transformer (all ROIs) vs ROI-Token Transformer (MI-selected)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'Transformer (all)':<22}{'Transformer (MI)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]);          svm_std = np.std(svm_results[metric])
    tf_all_mean = np.mean(transformer_roitoken_results[metric]); tf_all_std = np.std(transformer_roitoken_results[metric])
    tf_mi_mean = np.mean(transformer_mi_results[metric]);  tf_mi_std = np.std(transformer_mi_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      "
          f"{tf_all_mean:.3f} ± {tf_all_std:.3f}      "
          f"{tf_mi_mean:.3f} ± {tf_mi_std:.3f}")


# =====================================================
# 10. FC-PROFILE-AS-TOKEN TRANSFORMER
# =====================================================
# Instead of feeding each ROI's RAW timecourse as its token (fragile to
# scanner/site differences, as every experiment so far has shown), each
# ROI's token is now its FULL CONNECTIVITY PROFILE -- i.e. its row in the
# subject's FC matrix (correlation with every other ROI). This directly
# inherits the SVM-FC baseline's main strength (scale/site-invariant,
# correlation-based features) while still letting self-attention learn
# NONLINEAR combinations of connectivity patterns, which a linear SVM
# structurally cannot do.
#
# Input to the model changes from (N, n_rois, seq_len) -> (N, n_rois, n_rois)
# =====================================================

def compute_fc_matrix_per_subject(X, verbose=True):
    """
    X: (N, T, n_rois) raw (already per-ROI standardized) timeseries.
    Returns: (N, n_rois, n_rois) -- each subject's FULL FC matrix
             (not just the upper triangle -- every ROI's ROW becomes
             that ROI's token/feature vector for the transformer).
    """
    N, T, R = X.shape
    fc_matrices = np.zeros((N, R, R), dtype=np.float32)
    nan_subject_count = 0

    for i in range(N):
        fc = np.corrcoef(X[i].T)
        if np.isnan(fc).any():
            nan_subject_count += 1
            fc = np.nan_to_num(fc, nan=0.0)
        fc_matrices[i] = fc

    if verbose:
        print(f"Subjects with NaN in FC matrix (fixed via nan_to_num): {nan_subject_count} / {N}")

    return fc_matrices


class fMRITransformerFCToken(nn.Module):
    """
    Same overall architecture as fMRITransformerROIToken, except each
    ROI-token's input feature is its length-n_rois connectivity profile
    (a row of the FC matrix) instead of its raw seq_len-length timecourse.
    """
    def __init__(self, n_rois, d_model=64, n_heads=2, n_layers=1,
                 dim_ff=64, dropout=0.5, n_classes=2):
        super().__init__()
        # input dim is n_rois here (connectivity profile length),
        # NOT seq_len -- this is the key structural change
        self.input_proj = nn.Linear(n_rois, d_model)
        self.roi_embedding = nn.Parameter(torch.randn(1, n_rois, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, return_attn=False):
        # x: (B, n_rois, n_rois) -- x[:, i, :] is ROI i's connectivity profile
        h = self.input_proj(x)
        h = h + self.roi_embedding
        h = self.encoder(h)

        attn_scores = self.attn_pool(h).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        logits = self.classifier(pooled)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        return logits


def train_one_fold_fc_token(X_train_fc, y_train, X_val_fc, y_val,
                             n_rois, epochs=60, lr=1e-4):
    """
    X_train_fc / X_val_fc: (N, n_rois, n_rois) FC matrices (already computed
    per-subject, per-fold-safe -- see evaluate_fc_token_transformer below).
    """
    model = fMRITransformerFCToken(n_rois=n_rois).to(device)

    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train_fc).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val_fc).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


def evaluate_fc_token_transformer(X, y, skf):
    """
    Trains/evaluates the FC-profile-as-token transformer, computing the
    FC matrix separately per fold split (train and val subjects use their
    OWN raw timeseries to build their own FC matrix -- no leakage, since
    each subject's FC matrix only depends on that subject's own signal,
    not on other subjects' labels or data).
    """
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_train_raw, X_val_raw = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        # each subject's FC matrix depends only on their own timeseries --
        # safe to compute independently for train and val subjects
        X_train_fc = compute_fc_matrix_per_subject(X_train_raw, verbose=False)
        X_val_fc = compute_fc_matrix_per_subject(X_val_raw, verbose=False)

        model, auc = train_one_fold_fc_token(
            X_train_fc, y_train, X_val_fc, y_val, n_rois=X_train_fc.shape[1]
        )

        model.eval()
        with torch.no_grad():
            Xva_t = torch.tensor(X_val_fc).to(device)
            val_logits = model(Xva_t)
            val_preds = val_logits.argmax(dim=1).cpu().numpy()

        tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_score(y_val, val_preds, zero_division=0)
        acc = accuracy_score(y_val, val_preds)

        aucs.append(auc); senss.append(sensitivity); specs.append(specificity)
        f1s.append(f1); accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== FC-Profile-Token Transformer: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


print("\n" + "=" * 60)
print("Training FC-Profile-as-Token Transformer")
print("=" * 60)
transformer_fctoken_results = evaluate_fc_token_transformer(X, y, skf)


# =====================================================
# 11. UPDATED FINAL SUMMARY -- ALL FOUR MODELS
# =====================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON: SVM-FC vs Transformer(raw,all) vs Transformer(raw,MI) vs Transformer(FC-token)")
print("=" * 60)
print(f"{'Metric':<15}{'SVM-FC':<20}{'TF(raw,all)':<20}{'TF(raw,MI)':<20}{'TF(FC-token)':<20}")
for metric in ["auc", "sensitivity", "specificity", "f1", "accuracy"]:
    svm_mean = np.mean(svm_results[metric]);          svm_std = np.std(svm_results[metric])
    tf_all_mean = np.mean(transformer_roitoken_results[metric]); tf_all_std = np.std(transformer_roitoken_results[metric])
    tf_mi_mean = np.mean(transformer_mi_results[metric]);  tf_mi_std = np.std(transformer_mi_results[metric])
    tf_fc_mean = np.mean(transformer_fctoken_results[metric]); tf_fc_std = np.std(transformer_fctoken_results[metric])
    print(f"{metric:<15}{svm_mean:.3f} ± {svm_std:.3f}      "
          f"{tf_all_mean:.3f} ± {tf_all_std:.3f}      "
          f"{tf_mi_mean:.3f} ± {tf_mi_std:.3f}      "
          f"{tf_fc_mean:.3f} ± {tf_fc_std:.3f}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------
# Positional encoding (standard sinusoidal)
# ---------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ---------------------------------------------------
# ROI-time-series Transformer classifier
# ---------------------------------------------------
class fMRITransformer(nn.Module):
    def __init__(self, n_rois=200, d_model=64, n_heads=4, n_layers=2,
                 dim_ff=128, dropout=0.3, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_rois, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.attn_pool = nn.Linear(d_model, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes)
        )

    def forward(self, x, src_key_padding_mask=None):
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h, src_key_padding_mask=src_key_padding_mask)

        attn_scores = self.attn_pool(h).squeeze(-1)
        if src_key_padding_mask is not None:
            attn_scores = attn_scores.masked_fill(src_key_padding_mask, float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = (h * attn_weights).sum(dim=1)

        return self.classifier(pooled)

In [ ]:
def make_padding_mask(X_batch, target_time=178, real_lengths=None):
    # if you track real (pre-pad) lengths per subject, mask out padded steps
    if real_lengths is None:
        return None
    B, T, _ = X_batch.shape
    mask = torch.zeros(B, T, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < T:
            mask[i, l:] = True
    return mask


def train_one_fold(X_train, y_train, X_val, y_val, epochs=60, lr=1e-4):
    model = fMRITransformer(n_rois=X_train.shape[-1]).to(device)

    # class weighting for imbalance (ASD vs TD counts often uneven)
    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc


# ---------------------------------------------------
# Run 5-fold stratified CV
# ---------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs, fold_accs = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X[tr_idx], X[va_idx]
    y_train, y_val = y[tr_idx], y[va_idx]

    model, auc = train_one_fold(X_train, y_train, X_val, y_val)

    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val).to(device))
        val_preds = val_logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_val, val_preds)
    fold_aucs.append(auc)
    fold_accs.append(acc)
    print(f"Fold {fold+1}: AUC={auc:.3f}  ACC={acc:.3f}")

print(f"\nMean AUC: {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
print(f"Mean ACC: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score

def make_padding_mask(real_lengths, target_time):
    """
    Returns a (B, T) bool mask where True = padded position (to be ignored).
    """
    if real_lengths is None:
        return None
    B = len(real_lengths)
    mask = torch.zeros(B, target_time, dtype=torch.bool)
    for i, l in enumerate(real_lengths):
        if l < target_time:
            mask[i, l:] = True
    return mask


def train_one_fold(X_train, y_train, X_val, y_val,
                    real_lengths_train=None, real_lengths_val=None,
                    epochs=60, lr=1e-4, target_time=178):

    model = fMRITransformer(n_rois=X_train.shape[-1]).to(device)

    # class weighting for imbalance (ASD vs TD counts often uneven)
    classes, counts = np.unique(y_train, return_counts=True)
    weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    Xtr = torch.tensor(X_train).to(device)
    ytr = torch.tensor(y_train, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    # build padding masks once (constant across epochs for a given fold)
    train_mask = make_padding_mask(real_lengths_train, target_time)
    val_mask = make_padding_mask(real_lengths_val, target_time)
    if train_mask is not None:
        train_mask = train_mask.to(device)
    if val_mask is not None:
        val_mask = val_mask.to(device)

    best_auc, best_state, patience, bad_epochs = 0, None, 10, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(Xtr, src_key_padding_mask=train_mask)
        loss = criterion(logits, ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva, src_key_padding_mask=val_mask)
            val_probs = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y_val, val_probs)

        if val_auc > best_auc:
            best_auc, best_state, bad_epochs = val_auc, model.state_dict(), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc, val_mask


# ---------------------------------------------------
# Run 5-fold stratified CV
# ---------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs, fold_accs, fold_sens, fold_spec, fold_f1 = [], [], [], [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X[tr_idx], X[va_idx]
    y_train, y_val = y[tr_idx], y[va_idx]
    rl_train, rl_val = real_lengths[tr_idx], real_lengths[va_idx]

    model, auc, val_mask = train_one_fold(
        X_train, y_train, X_val, y_val,
        real_lengths_train=rl_train, real_lengths_val=rl_val
    )

    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val).to(device), src_key_padding_mask=val_mask)
        val_preds = val_logits.argmax(dim=1).cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = f1_score(y_val, val_preds, zero_division=0)
    acc = accuracy_score(y_val, val_preds)

    fold_aucs.append(auc)
    fold_accs.append(acc)
    fold_sens.append(sensitivity)
    fold_spec.append(specificity)
    fold_f1.append(f1)

    print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
          f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

print("\n===== Transformer: Mean ± Std across folds =====")
print(f"AUC         : {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
print(f"Sensitivity : {np.mean(fold_sens):.3f} ± {np.std(fold_sens):.3f}")
print(f"Specificity : {np.mean(fold_spec):.3f} ± {np.std(fold_spec):.3f}")
print(f"F1 Score    : {np.mean(fold_f1):.3f} ± {np.std(fold_f1):.3f}")
print(f"Accuracy    : {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

def evaluate_fc_svm(fc_X, y, skf):
    aucs, senss, specs, f1s, accs = [], [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(fc_X, y)):
        X_train, X_val = fc_X[tr_idx], fc_X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        svm = SVC(kernel="linear", class_weight="balanced", probability=True)
        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_val)
        y_prob = svm.predict_proba(X_val)[:, 1]

        # confusion matrix: rows=true, cols=pred -> [[TN, FP], [FN, TP]]
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan   # recall for class 1 (ASD)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan  # recall for class 0 (TD)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        auc = roc_auc_score(y_val, y_prob)
        acc = (tp + tn) / (tp + tn + fp + fn)

        aucs.append(auc)
        senss.append(sensitivity)
        specs.append(specificity)
        f1s.append(f1)
        accs.append(acc)

        print(f"Fold {fold+1}: AUC={auc:.3f}  Sens={sensitivity:.3f}  "
              f"Spec={specificity:.3f}  F1={f1:.3f}  Acc={acc:.3f}")

    print("\n===== SVM-FC Baseline: Mean ± Std across folds =====")
    print(f"AUC         : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}")
    print(f"Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}")
    print(f"F1 Score    : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")

    return {"auc": aucs, "sensitivity": senss, "specificity": specs,
            "f1": f1s, "accuracy": accs}


results = evaluate_fc_svm(fc_X, y, skf)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Functional connectivity baseline (much lower-dim, standard in the field)
def compute_fc_features(X):
    N = X.shape[0]
    n_rois = X.shape[2]
    triu_idx = np.triu_indices(n_rois, k=1)
    fc_feats = np.zeros((N, len(triu_idx[0])))
    for i in range(N):
        fc = np.corrcoef(X[i].T)
        fc_feats[i] = fc[triu_idx]
    return fc_feats

fc_X = compute_fc_features(X)
svm = SVC(kernel="linear", class_weight="balanced")
scores = cross_val_score(svm, fc_X, y, cv=skf, scoring="roc_auc")
print(f"SVM-FC baseline AUC: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
# =====================================================
# Cell 2 : Dataset Information
# =====================================================

print("="*60)
print("Subjects      :", X.shape[0])
print("Time Points   :", X.shape[1])
print("ROIs          :", X.shape[2])
print("ASD           :", np.sum(y==1))
print("TD            :", np.sum(y==0))
print("="*60)

NUM_SUBJECTS  = X.shape[0]
TIME_POINTS   = X.shape[1]
NUM_ROIS      = X.shape[2]
NUM_CLASSES   = 2

In [ ]:
# =====================================================
# Cell 3 : Pearson Functional Connectivity
# =====================================================

import numpy as np

def compute_fc(timeseries):

    """
    Input
        (178,116)

    Output
        (116,116)
    """

    fc = np.corrcoef(timeseries.T)

    fc = np.nan_to_num(fc)

    return fc.astype(np.float32)


FC = np.asarray([compute_fc(ts) for ts in X])

print("FC Shape :", FC.shape)

In [ ]:
# =====================================================
# Cell 4 : Upper Triangle Features
# =====================================================

triu = np.triu_indices(NUM_ROIS,1)

FC_FEATURES = FC[:,triu[0],triu[1]]

print(FC_FEATURES.shape)

In [ ]:
# =====================================================
# Cell 5 : ROI Tokens
# =====================================================

ROI_TOKENS = np.transpose(X,(0,2,1))

print(ROI_TOKENS.shape)

In [ ]:
# =====================================================
# Cell 6 : Transformer Connectivity Layer
# =====================================================

import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization

class ConnectivityTransformer(Layer):

    def __init__(self,
                 embed_dim=128,
                 heads=4):

        super().__init__()

        self.projection = Dense(embed_dim)

        self.attention = MultiHeadAttention(
            num_heads=heads,
            key_dim=embed_dim//heads,
            output_shape=embed_dim
        )

        self.norm = LayerNormalization()

    def call(self,x):

        """
        x

        (batch,116,178)
        """

        z = self.projection(x)

        out,scores = self.attention(
            z,
            z,
            return_attention_scores=True
        )

        out = self.norm(out+z)

        return out,scores

In [ ]:
# =====================================================
# Cell 7 : Connectivity Model
# =====================================================

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

inputs = Input(shape=(NUM_ROIS, TIME_POINTS))

embeddings, connectivity = ConnectivityTransformer()(inputs)

model = Model(
    inputs=inputs,
    outputs=connectivity,
    name="TransformerConnectivity"
)

model.summary()

In [ ]:
# =====================================================
# Cell 8 : Build Transformer Classifier
# =====================================================

import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model


def build_classifier():

    inputs = Input(shape=(NUM_ROIS, TIME_POINTS))

    embeddings, attention = ConnectivityTransformer()(inputs)

    x = GlobalAveragePooling1D()(embeddings)

    x = Dense(256, activation="relu")(x)
    x = Dropout(0.40)(x)

    x = Dense(64, activation="relu")(x)
    x = Dropout(0.30)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(inputs, outputs)

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    return model

In [ ]:
# =====================================================
# Cell 9
# =====================================================

ROI_TOKENS = np.transpose(X,(0,2,1))

print(ROI_TOKENS.shape)

In [ ]:
# =====================================================
# Cell 10
# =====================================================

from sklearn.model_selection import StratifiedKFold

NUM_FOLDS = 10

skf = StratifiedKFold(

    n_splits=NUM_FOLDS,

    shuffle=True,

    random_state=42

)

In [ ]:
# =====================================================
# Cell 11
# =====================================================

from tensorflow.keras.callbacks import EarlyStopping

histories=[]

scores=[]

best_model=None

best_auc=0

fold=1

for train_idx,test_idx in skf.split(ROI_TOKENS,y):

    print("="*60)

    print("Fold",fold)

    print("="*60)

    classifier = build_classifier()

    callback = EarlyStopping(

        monitor="val_auc",

        mode="max",

        patience=10,

        restore_best_weights=True

    )

    history = classifier.fit(

        ROI_TOKENS[train_idx],

        y[train_idx],

        validation_split=0.10,

        epochs=100,

        batch_size=16,

        callbacks=[callback],

        verbose=1

    )

    score = classifier.evaluate(

        ROI_TOKENS[test_idx],

        y[test_idx],

        verbose=0

    )

    print(score)

    scores.append(score)

    histories.append(history)

    if score[2] > best_auc:

        best_auc = score[2]

        best_model = classifier

    fold += 1

In [ ]:
# =====================================================
# Cell 12
# =====================================================

scores=np.asarray(scores)

print()

print("Average Accuracy :",scores[:,1].mean())

print("Average AUC :",scores[:,2].mean())

print("Average Precision :",scores[:,3].mean())

print("Average Recall :",scores[:,4].mean())

In [ ]:
# =====================================================
# Cell 13
# =====================================================

from tensorflow.keras.models import Model

attention_model = Model(

    inputs=best_model.input,

    outputs=best_model.layers[1].output[1]

)

ATTENTION = attention_model.predict(

    ROI_TOKENS,

    batch_size=16,

    verbose=1

)

print(ATTENTION.shape)

In [ ]:
# =====================================================
# Cell 14 : Average Multi-head Attention
# =====================================================

ATTENTION = ATTENTION.mean(axis=1)

print("Attention Shape :", ATTENTION.shape)

In [ ]:
print(type(ATTENTION))
print(ATTENTION.shape)

In [ ]:
# =====================================================
# Cell 15 : Symmetric Attention Connectivity
# =====================================================

ATTENTION_SYM = 0.5 * (

    ATTENTION +

    np.transpose(ATTENTION, (0,2,1))

)

print("Symmetric Attention Shape :", ATTENTION_SYM.shape)

In [ ]:
# =====================================================
# Cell 16 : Upper Triangle Features
# =====================================================

triu = np.triu_indices(NUM_ROIS,1)

ATTENTION_FEATURES = ATTENTION_SYM[:,

                                   triu[0],

                                   triu[1]]

print("Attention Features :",ATTENTION_FEATURES.shape)

In [ ]:
# =====================================================
# Cell 17 : Visual Comparison
# =====================================================

import matplotlib.pyplot as plt

subject = 0

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(FC[subject], cmap="jet")
plt.title("Pearson FC")
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(ATTENTION_SYM[subject], cmap="jet")
plt.title("Transformer Connectivity")
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# Cell 18 : Connectivity Similarity
# =====================================================

from scipy.stats import pearsonr

corr=[]

for i in range(len(FC)):

    r,_ = pearsonr(

        FC[i][triu],

        ATTENTION_SYM[i][triu]

    )

    corr.append(r)

corr=np.asarray(corr)

print("Average Correlation :",corr.mean())

In [ ]:
# =====================================================
# Cell 19 : SVM Comparison
# =====================================================

import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

NUM_FOLDS = 5

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)


def evaluate_svm(features, labels):

    accuracy = []
    precision = []
    recall = []
    f1 = []
    auc = []

    for train_idx, test_idx in skf.split(features, labels):

        X_train = features[train_idx]
        X_test  = features[test_idx]

        y_train = labels[train_idx]
        y_test  = labels[test_idx]

        clf = Pipeline([

            ("scaler", StandardScaler()),

            ("svm",
             SVC(
                 kernel="rbf",
                 C=1.0,
                 gamma="scale",
                 probability=True,
                 random_state=42
             )
            )

        ])

        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)

        y_prob = clf.predict_proba(X_test)[:,1]

        accuracy.append(
            accuracy_score(y_test, y_pred)
        )

        precision.append(
            precision_score(y_test, y_pred)
        )

        recall.append(
            recall_score(y_test, y_pred)
        )

        f1.append(
            f1_score(y_test, y_pred)
        )

        auc.append(
            roc_auc_score(y_test, y_prob)
        )

    return {

        "Accuracy" : np.mean(accuracy),

        "Precision": np.mean(precision),

        "Recall"   : np.mean(recall),

        "F1 Score" : np.mean(f1),

        "AUC"      : np.mean(auc)

    }


print("="*70)
print("Pearson Functional Connectivity")
print("="*70)

pearson_results = evaluate_svm(FC_FEATURES, y)

for k,v in pearson_results.items():
    print(f"{k:12s}: {v:.4f}")

print()

print("="*70)
print("Transformer Connectivity")
print("="*70)

transformer_results = evaluate_svm(ATTENTION_FEATURES, y)

for k,v in transformer_results.items():
    print(f"{k:12s}: {v:.4f}")

In [ ]:
print(np.bincount(y))

print(ATTENTION_FEATURES.shape)

print(np.std(ATTENTION_FEATURES))

In [ ]:
# =====================================================
# Cell 20 : Comparison Table
# =====================================================

import pandas as pd

comparison = pd.DataFrame({

    "Metric": list(pearson_results.keys()),

    "Pearson FC": list(pearson_results.values()),

    "Transformer FC": list(transformer_results.values())

})

comparison["Improvement"] = (

    comparison["Transformer FC"]

    -

    comparison["Pearson FC"]

)

comparison

In [ ]:
# =====================================================
# Cell 2 : 10-Fold Stratified Cross Validation
# =====================================================

import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split
)

# -----------------------------------------------------
# Parameters
# -----------------------------------------------------

NUM_FOLDS = 10
VALIDATION_SIZE = 0.10
RANDOM_STATE = 42

# -----------------------------------------------------
# Stratified K-Fold
# -----------------------------------------------------

skf = StratifiedKFold(

    n_splits=NUM_FOLDS,

    shuffle=True,

    random_state=RANDOM_STATE

)

print("=" * 70)
print("10-Fold Stratified Cross Validation")
print("=" * 70)

fold = 1

# -----------------------------------------------------
# Cross Validation Loop
# -----------------------------------------------------

for train_index, test_index in skf.split(X, y):

    print("\n")
    print("=" * 70)
    print(f"Fold {fold}/{NUM_FOLDS}")
    print("=" * 70)

    # -------------------------------------------------
    # Training and Test Split
    # -------------------------------------------------

    X_train_full = X[train_index]
    y_train_full = y[train_index]

    X_test = X[test_index]
    y_test = y[test_index]

    # -------------------------------------------------
    # Validation Split
    # -------------------------------------------------

    X_train, X_val, y_train, y_val = train_test_split(

        X_train_full,
        y_train_full,

        test_size=VALIDATION_SIZE,

        stratify=y_train_full,

        random_state=RANDOM_STATE

    )

    # -------------------------------------------------
    # Dataset Summary
    # -------------------------------------------------

    print("\nDataset Shapes")
    print("-" * 40)

    print("Training   :", X_train.shape)
    print("Validation :", X_val.shape)
    print("Testing    :", X_test.shape)

    # -------------------------------------------------
    # Class Distribution
    # -------------------------------------------------

    print("\nClass Distribution")
    print("-" * 40)

    print("Training   :", np.bincount(y_train))
    print("Validation :", np.bincount(y_val))
    print("Testing    :", np.bincount(y_test))

    # -------------------------------------------------
    # Next Step
    # -------------------------------------------------

    print("\nNext Step")
    print(">> F-score ROI Selection on TRAINING SET ONLY")

    # =================================================
    # Cell 3 will start here
    # =================================================

    fold += 1

In [ ]:
# =====================================================
# Cell 3 : ROI Selection using ANOVA F-score (Improved)
# =====================================================

import numpy as np
from sklearn.feature_selection import f_classif

# -----------------------------------------------------
# Parameters
# -----------------------------------------------------

ROI_PERCENTAGE = 0.40

# -----------------------------------------------------
# ROI Selection Function
# -----------------------------------------------------

def fscore_roi_selection(
        X_train,
        y_train,
        X_val,
        X_test,
        percentage=ROI_PERCENTAGE
):

    """
    Parameters
    ----------
    X_train : (subjects, time_points, rois)

    Returns
    -------
    X_train_selected
    X_val_selected
    X_test_selected
    selected_roi_idx
    roi_scores
    """

    n_subjects, n_time, n_rois = X_train.shape

    roi_scores = np.zeros(n_rois, dtype=np.float32)

    # ---------------------------------------------
    # Compute one F-score per ROI
    # ---------------------------------------------

    for roi in range(n_rois):

        # Shape
        # (subjects, time_points)

        roi_data = X_train[:, :, roi]

        # Remove constant time points
        valid_columns = np.std(roi_data, axis=0) > 1e-8

        if np.sum(valid_columns) == 0:

            roi_scores[roi] = 0
            continue

        roi_data = roi_data[:, valid_columns]

        # Compute F-score for all remaining time points
        F, _ = f_classif(roi_data, y_train)

        # Remove NaN and Inf
        F = np.nan_to_num(
            F,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        # Mean F-score for this ROI
        roi_scores[roi] = np.mean(F)

    # ---------------------------------------------
    # Select Top ROIs
    # ---------------------------------------------

    n_selected = max(1, int(np.ceil(n_rois * percentage)))

    selected_roi_idx = np.argsort(roi_scores)[::-1][:n_selected]

    # ---------------------------------------------
    # Apply ROI Selection
    # ---------------------------------------------

    X_train_selected = X_train[:, :, selected_roi_idx]
    X_val_selected   = X_val[:, :, selected_roi_idx]
    X_test_selected  = X_test[:, :, selected_roi_idx]

    # ---------------------------------------------
    # Print Results
    # ---------------------------------------------

    print("\n")
    print("=" * 60)
    print("ROI FEATURE SELECTION")
    print("=" * 60)

    print("Original ROIs :", n_rois)
    print("Selected ROIs :", n_selected)

    print("\nTop Selected ROIs")

    for rank, roi in enumerate(selected_roi_idx):

        print(
            f"{rank+1:02d}. "
            f"ROI {roi:3d} "
            f"F-score = {roi_scores[roi]:.4f}"
        )

    print("=" * 60)

    return (

        X_train_selected,

        X_val_selected,

        X_test_selected,

        selected_roi_idx,

        roi_scores

    )

# =====================================================
# Apply ROI Selection
# =====================================================

X_train, X_val, X_test, selected_roi_idx, roi_scores = \
    fscore_roi_selection(

        X_train,
        y_train,

        X_val,
        X_test,

        percentage=ROI_PERCENTAGE

    )

print("\nSelected Dataset Shapes")

print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

In [ ]:
!pip install pycwt

In [ ]:
# =====================================================
# Cell 4 : Morlet Continuous Wavelet Transform
# =====================================================

import numpy as np
import pycwt as wavelet
from tqdm import tqdm

# -----------------------------------------------------
# Wavelet Parameters
# -----------------------------------------------------

DT = 1.0              # Sampling interval (1 TR)
DJ = 1 / 12           # Scale spacing
S0 = 2 * DT           # Smallest scale
J = 64                # Number of scales
MOTHER = wavelet.Morlet(6)

print("="*60)
print("Wavelet Parameters")
print("="*60)
print(f"Mother Wavelet : Morlet (ω0 = 6)")
print(f"Sampling Interval (DT): {DT}")
print(f"Scale Resolution (DJ): {DJ}")
print(f"Smallest Scale (S0): {S0}")
print(f"Number of Scales (J): {J}")
print("="*60)

# -----------------------------------------------------
# Continuous Wavelet Transform for One Subject
# -----------------------------------------------------

def compute_wavelet_transform(subject):

    """
    Parameters
    ----------
    subject : ndarray
        Shape = (time_points, n_selected_rois)

    Returns
    -------
    wavelet_coeffs : list
        List of complex wavelet coefficient matrices.

    scales : ndarray

    frequencies : ndarray
    """

    n_time, n_rois = subject.shape

    wavelet_coeffs = []

    scales = None
    frequencies = None

    for roi in range(n_rois):

        signal = subject[:, roi]

        W, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(

            signal,

            DT,

            dj=DJ,

            s0=S0,

            J=J,

            wavelet=MOTHER

        )

        wavelet_coeffs.append(W)

    return wavelet_coeffs, scales, freqs


# -----------------------------------------------------
# Compute Wavelet Transform for Entire Dataset
# -----------------------------------------------------

def build_wavelet_dataset(X):

    """
    Parameters
    ----------
    X : ndarray

        Shape

        (subjects,time,rois)

    Returns
    -------
    dataset_wavelets

        List

        dataset_wavelets[subject][roi]

        Shape

        (scales,time)
    """

    dataset_wavelets = []

    scales = None
    freqs = None

    for subject in tqdm(X, desc="Wavelet Transform"):

        coeffs, scales, freqs = compute_wavelet_transform(subject)

        dataset_wavelets.append(coeffs)

    return dataset_wavelets, scales, freqs


# -----------------------------------------------------
# Build Wavelet Datasets
# -----------------------------------------------------

print("\nComputing Training Wavelets...")

wavelet_train, scales, freqs = build_wavelet_dataset(X_train)

print("\nComputing Validation Wavelets...")

wavelet_val, _, _ = build_wavelet_dataset(X_val)

print("\nComputing Test Wavelets...")

wavelet_test, _, _ = build_wavelet_dataset(X_test)

# -----------------------------------------------------
# Summary
# -----------------------------------------------------

print("\n")
print("="*60)
print("Wavelet Transform Summary")
print("="*60)

print("Training Subjects   :", len(wavelet_train))
print("Validation Subjects :", len(wavelet_val))
print("Test Subjects       :", len(wavelet_test))

print()

print("Selected ROIs :", len(wavelet_train[0]))

print()

print("Coefficient Shape")

print(wavelet_train[0][0].shape)

print()

print("Number of Scales :", len(scales))

print()

print("Frequency Range")

print(freqs.min(), "to", freqs.max())

print("="*60)

In [ ]:
# =====================================================
# Cell 5 : Smoothed Wavelet Coherence
# (Bernas et al. implementation)
# =====================================================

import numpy as np
from scipy.ndimage import gaussian_filter

# -----------------------------------------------------
# Smoothing Parameters
# -----------------------------------------------------

TIME_SIGMA = 2.0
SCALE_SIGMA = 1.0

print("="*60)
print("Wavelet Coherence Parameters")
print("="*60)
print("Time smoothing :", TIME_SIGMA)
print("Scale smoothing:", SCALE_SIGMA)
print("="*60)

# -----------------------------------------------------
# Gaussian Smoothing
# -----------------------------------------------------

def smooth_matrix(mat):
    """
    Smooth over both
    scale and time.
    """

    return gaussian_filter(
        mat,
        sigma=(SCALE_SIGMA, TIME_SIGMA),
        mode="nearest"
    )


# -----------------------------------------------------
# Wavelet Coherence
# -----------------------------------------------------

def compute_wavelet_coherence(Wx, Wy):
    """
    Parameters
    ----------
    Wx : complex wavelet coefficients
         shape (scale,time)

    Wy : complex wavelet coefficients
         shape (scale,time)

    Returns
    -------
    coherence
        shape (scale,time)

    phase
        shape (scale,time)
    """

    # ----------------------------------------------
    # Cross wavelet
    # ----------------------------------------------

    Wxy = Wx * np.conjugate(Wy)

    # ----------------------------------------------
    # Smoothed power
    # ----------------------------------------------

    Sxx = smooth_matrix(np.abs(Wx) ** 2)

    Syy = smooth_matrix(np.abs(Wy) ** 2)

    Sxy = smooth_matrix(Wxy)

    # ----------------------------------------------
    # Wavelet coherence
    # ----------------------------------------------

    coherence = np.abs(Sxy) ** 2

    coherence /= (Sxx * Syy + 1e-12)

    coherence = np.clip(coherence, 0, 1)

    # ----------------------------------------------
    # Phase difference
    # ----------------------------------------------

    phase = np.angle(Sxy)

    return coherence, phase


# -----------------------------------------------------
# Compute WTC for One Subject
# -----------------------------------------------------

def compute_subject_coherence(subject_wavelets):
    """
    Parameters
    ----------
    subject_wavelets

        list of wavelet coefficients

    Returns
    -------
    coherence_list

    phase_list
    """

    n_rois = len(subject_wavelets)

    coherence_list = []

    phase_list = []

    roi_pairs = []

    for i in range(n_rois):

        for j in range(i + 1, n_rois):

            coh, phase = compute_wavelet_coherence(

                subject_wavelets[i],
                subject_wavelets[j]

            )

            coherence_list.append(coh)

            phase_list.append(phase)

            roi_pairs.append((i, j))

    return coherence_list, phase_list, roi_pairs


# -----------------------------------------------------
# Dataset Builder
# -----------------------------------------------------

def build_coherence_dataset(dataset_wavelets):

    coherence_dataset = []

    phase_dataset = []

    roi_pairs = None

    for subject in tqdm(dataset_wavelets,
                        desc="Wavelet Coherence"):

        coh, phase, roi_pairs = compute_subject_coherence(subject)

        coherence_dataset.append(coh)

        phase_dataset.append(phase)

    return coherence_dataset, phase_dataset, roi_pairs


# -----------------------------------------------------
# Build Training Dataset
# -----------------------------------------------------

print("\nTraining...")

coherence_train, phase_train, roi_pairs = build_coherence_dataset(
    wavelet_train
)

print("\nValidation...")

coherence_val, phase_val, _ = build_coherence_dataset(
    wavelet_val
)

print("\nTesting...")

coherence_test, phase_test, _ = build_coherence_dataset(
    wavelet_test
)

# -----------------------------------------------------
# Summary
# -----------------------------------------------------

print("\n")
print("="*60)
print("Wavelet Coherence Summary")
print("="*60)

print("Training subjects   :", len(coherence_train))
print("Validation subjects :", len(coherence_val))
print("Testing subjects    :", len(coherence_test))

print()

print("ROI pairs :", len(roi_pairs))

print()

print("Example coherence shape")

print(coherence_train[0][0].shape)

print()

print("Example phase shape")

print(phase_train[0][0].shape)

print("="*60)

In [ ]:
# =====================================================
# Cell 6 : Time of In-Phase Coherence (TIPC)
# =====================================================

import numpy as np
from tqdm import tqdm

# -----------------------------------------------------
# TIPC Parameters
# -----------------------------------------------------

# Phase threshold (45° = π/4)
PHASE_THRESHOLD = np.pi / 4

print("="*60)
print("TIPC Parameters")
print("="*60)
print(f"Phase Threshold : ±{PHASE_THRESHOLD:.3f} rad")
print(f"                 ±45 degrees")
print("="*60)

# -----------------------------------------------------
# Compute TIPC for One ROI Pair
# -----------------------------------------------------

def compute_tipc(phase_matrix):
    """
    Parameters
    ----------
    phase_matrix : ndarray
        Shape = (n_scales, n_time)

    Returns
    -------
    tipc : float
        Fraction of time-scale points that are in-phase.
    """

    in_phase = np.abs(phase_matrix) <= PHASE_THRESHOLD

    return np.mean(in_phase)


# -----------------------------------------------------
# Compute TIPC for One Subject
# -----------------------------------------------------

def compute_subject_tipc(subject_phase):

    """
    Parameters
    ----------
    subject_phase

        list of phase matrices

    Returns
    -------
    tipc_vector

        shape = (ROI_pairs,)
    """

    features = np.zeros(len(subject_phase), dtype=np.float32)

    for i, phase in enumerate(subject_phase):

        features[i] = compute_tipc(phase)

    return features


# -----------------------------------------------------
# Dataset Builder
# -----------------------------------------------------

def build_tipc_dataset(phase_dataset):

    X = []

    for subject in tqdm(phase_dataset,
                        desc="Computing TIPC"):

        X.append(compute_subject_tipc(subject))

    return np.asarray(X, dtype=np.float32)


# -----------------------------------------------------
# Build Feature Matrices
# -----------------------------------------------------

print("\nTraining...")

X_train_tipc = build_tipc_dataset(phase_train)

print("\nValidation...")

X_val_tipc = build_tipc_dataset(phase_val)

print("\nTesting...")

X_test_tipc = build_tipc_dataset(phase_test)

# -----------------------------------------------------
# Summary
# -----------------------------------------------------

print("\n")
print("="*60)
print("TIPC Feature Summary")
print("="*60)

print("Training :", X_train_tipc.shape)
print("Validation :", X_val_tipc.shape)
print("Testing :", X_test_tipc.shape)

print()

print("ROI pairs :", X_train_tipc.shape[1])

print()

print("Example Feature Vector")

print(np.round(X_train_tipc[0][:20],3))

print("="*60)

In [ ]:
# =====================================================
# Cell 7 : Standardize TIPC Features
# =====================================================

from sklearn.preprocessing import StandardScaler

print("="*60)
print("Standardizing TIPC Features")
print("="*60)

# -----------------------------------------------------
# Fit scaler ONLY on training data
# -----------------------------------------------------

scaler = StandardScaler()

X_train_tipc = scaler.fit_transform(X_train_tipc)

X_val_tipc = scaler.transform(X_val_tipc)

X_test_tipc = scaler.transform(X_test_tipc)

# -----------------------------------------------------
# Summary
# -----------------------------------------------------

print("\nFeature Shapes")

print("Training   :", X_train_tipc.shape)
print("Validation :", X_val_tipc.shape)
print("Testing    :", X_test_tipc.shape)

print("\nTraining Statistics")

print("Mean :", np.mean(X_train_tipc))
print("Std  :", np.std(X_train_tipc))

print("\nFirst 10 standardized features of Subject 1")

print(np.round(X_train_tipc[0][:10], 3))

print("="*60)

In [ ]:
# =====================================================
# Cell 8 : Build SVM Classifier
# =====================================================

from sklearn.svm import SVC

# -----------------------------------------------------
# Build SVM
# -----------------------------------------------------

model = SVC(

    kernel="rbf",          # RBF kernel

    C=1.0,

    gamma="scale",

    probability=True,      # Needed for ROC-AUC

    class_weight="balanced",

    random_state=42

)

print("="*60)
print("Support Vector Machine")
print("="*60)
print(model)

In [ ]:
# =====================================================
# Cell 9 : Train SVM Classifier
# =====================================================

import time

print("="*60)
print("Training Support Vector Machine")
print("="*60)

# -----------------------------------------------------
# Start Timer
# -----------------------------------------------------

start_time = time.time()

# -----------------------------------------------------
# Train SVM
# -----------------------------------------------------

model.fit(

    X_train_tipc,

    y_train

)

# -----------------------------------------------------
# Training Time
# -----------------------------------------------------

training_time = time.time() - start_time

print("\nTraining Completed Successfully.")

print(f"\nTraining Time : {training_time:.2f} seconds")

print("="*60)

# -----------------------------------------------------
# Validation Accuracy
# -----------------------------------------------------

val_accuracy = model.score(

    X_val_tipc,

    y_val

)

print(f"Validation Accuracy : {val_accuracy:.4f}")

print("="*60)

In [ ]:
# =====================================================
# Cell 10 : Final Evaluation
# =====================================================

import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# -----------------------------------------------------
# Prediction
# -----------------------------------------------------

y_prob = model.predict(X_test_tipc).ravel()

y_pred = (y_prob >= 0.5).astype(int)

# -----------------------------------------------------
# Confusion Matrix
# -----------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

# -----------------------------------------------------
# Metrics
# -----------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

specificity = tn / (tn + fp)

sensitivity = recall

# -----------------------------------------------------
# Results
# -----------------------------------------------------

print("\n")
print("="*70)
print("FINAL TEST RESULTS")
print("="*70)

print(f"Accuracy     : {accuracy:.4f}")

print(f"AUC          : {auc:.4f}")

print(f"Precision    : {precision:.4f}")

print(f"Recall       : {recall:.4f}")

print(f"F1-score     : {f1:.4f}")

print(f"Sensitivity  : {sensitivity:.4f}")

print(f"Specificity  : {specificity:.4f}")

print()

print("Confusion Matrix")

print(confusion_matrix(
    y_test,
    y_pred
))

print()

print("Classification Report")

print(classification_report(
    y_test,
    y_pred,
    digits=4
))

print("="*70)

In [ ]:
# =====================================================
# Cell 4 : Prepare Dataset
# =====================================================

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("="*60)
print("Preparing Dataset")
print("="*60)

# -----------------------------------------------------
# Input data
# -----------------------------------------------------

print("Input Shape :", X.shape)
print("Label Shape :", y.shape)

# -----------------------------------------------------
# Train-Test Split
# -----------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
    shuffle=True
)

# -----------------------------------------------------
# Class Weights
# -----------------------------------------------------

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

# -----------------------------------------------------
# Summary
# -----------------------------------------------------

print("\nTraining Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nTraining Labels")

print("TD  :", np.sum(y_train == 0))
print("ASD :", np.sum(y_train == 1))

print("\nTesting Labels")

print("TD  :", np.sum(y_test == 0))
print("ASD :", np.sum(y_test == 1))

print("\nClass Weights")

print(class_weights)

print("="*60)

In [ ]:
# =====================================================
# Improved Temporal Transformer for ROI Time Series
# =====================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    Add
)

from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

# -----------------------------------------------------
# Hyperparameters
# -----------------------------------------------------

TIME_STEPS = X.shape[1]
N_FEATURES = X.shape[2]

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 256
NUM_LAYERS = 2          # reduced from 4
DROPOUT = 0.30

# -----------------------------------------------------
# Positional Embedding
# -----------------------------------------------------

class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(self, sequence_length, embed_dim):
        super().__init__()

        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

        self.sequence_length = sequence_length

    def call(self, inputs):

        positions = tf.range(
            start=0,
            limit=self.sequence_length,
            delta=1
        )

        pos_embed = self.position_embedding(positions)

        return inputs + pos_embed


# -----------------------------------------------------
# Attention Pooling
# -----------------------------------------------------

class AttentionPooling(tf.keras.layers.Layer):

    def __init__(self):
        super().__init__()

        self.score = Dense(1)

    def call(self, x):

        weights = self.score(x)

        weights = tf.nn.softmax(weights, axis=1)

        pooled = tf.reduce_sum(weights * x, axis=1)

        return pooled


# -----------------------------------------------------
# Pre-LayerNorm Transformer Block
# -----------------------------------------------------

def transformer_encoder(inputs):

    # ===========================
    # Multi-head Attention
    # ===========================

    shortcut = inputs

    x = LayerNormalization(epsilon=1e-6)(inputs)

    attention = MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=EMBED_DIM // NUM_HEADS,
        dropout=DROPOUT
    )(x, x)

    attention = Dropout(DROPOUT)(attention)

    x = Add()([shortcut, attention])

    # ===========================
    # Feed Forward Network
    # ===========================

    shortcut = x

    x = LayerNormalization(epsilon=1e-6)(x)

    x = Dense(
        FF_DIM,
        activation=tf.nn.gelu,
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(DROPOUT)(x)

    x = Dense(
        EMBED_DIM,
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Add()([shortcut, x])

    return x


# -----------------------------------------------------
# Build Model
# -----------------------------------------------------

def build_temporal_transformer():

    inputs = Input(shape=(TIME_STEPS, 29))

    # ===========================
    # Input Normalization
    # ===========================

    x = LayerNormalization()(inputs)

    # ===========================
    # Linear Projection
    # ===========================

    x = Dense(
        EMBED_DIM,
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.10)(x)

    # ===========================
    # Positional Embedding
    # ===========================

    x = PositionalEmbedding(
        TIME_STEPS,
        EMBED_DIM
    )(x)

    # ===========================
    # Transformer Encoder Stack
    # ===========================

    for _ in range(NUM_LAYERS):
        x = transformer_encoder(x)

    # ===========================
    # Attention Pooling
    # ===========================

    x = AttentionPooling()(x)

    # ===========================
    # Classification Head
    # ===========================

    x = Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.40)(x)

    x = Dense(
        64,
        activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.30)(x)

    outputs = Dense(
        1,
        activation='sigmoid'
    )(x)

    model = Model(inputs, outputs)

    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4
    )

    loss = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.05
    )

    model.compile(

        optimizer=optimizer,

        loss=loss,

        metrics=[

            'accuracy',

            tf.keras.metrics.AUC(name='auc'),

            tf.keras.metrics.Precision(name='precision'),

            tf.keras.metrics.Recall(name='recall')

        ]

    )

    return model


# -----------------------------------------------------
# Create Model
# -----------------------------------------------------

model = build_temporal_transformer()

model.summary()

In [ ]:
#ROI as tokens
def build_temporal_transformer():

    inputs = Input(shape=(TIME_STEPS, 29))

    # -----------------------------------------
    # ROI Tokens
    # (176,116) → (116,176)
    # -----------------------------------------

    x = tf.keras.layers.Permute((2,1))(inputs)

    x = LayerNormalization()(x)

    # -----------------------------------------
    # Project each ROI time series
    # 176 → 64
    # -----------------------------------------

    x = Dense(
        EMBED_DIM,
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.10)(x)

    # -----------------------------------------
    # ROI positional embedding
    # -----------------------------------------

    x = PositionalEmbedding(
        29,
        EMBED_DIM
    )(x)

    # Transformer stack

    for _ in range(NUM_LAYERS):
        x = transformer_encoder(x)

    # Attention Pooling

    x = AttentionPooling()(x)

    # Classification Head

    x = Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.40)(x)

    x = Dense(
        64,
        activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)

    x = Dropout(0.30)(x)

    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)

    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    return model

In [ ]:
# =====================================================
# Cell 6 : Training Callbacks
# =====================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        verbose=1
    ),

    ModelCheckpoint(
        "best_dmn_transformer.keras",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

]

In [ ]:
# =====================================================
# Cell 7 : Improved Cross Validation + F-score ROI Selection
# =====================================================

import os
import numpy as np
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import f_classif

NUM_FOLDS = 10
ROI_PERCENTAGE = 0.25        # Keep top 25% ROIs

# =====================================================
# ROI Feature Selection Function
# =====================================================

def select_top_rois_fscore(
        X_train,
        y_train,
        X_val,
        X_test,
        percentage=0.25
):

    """
    X shape:
    (subjects, time_steps, num_rois)
    """

    n_subjects, n_time, n_rois = X_train.shape

    roi_scores = np.zeros(n_rois)

    # ---------------------------------------------
    # Compute F-score for each ROI across all
    # time points
    # ---------------------------------------------

    for roi in range(n_rois):

        # Shape = (subjects, time_steps)
        roi_data = X_train[:, :, roi]

        time_scores = []

        for t in range(n_time):

            feature = roi_data[:, t].reshape(-1, 1)

            score, _ = f_classif(feature, y_train)

            time_scores.append(score[0])

        roi_scores[roi] = np.nanmean(time_scores)

    roi_scores = np.nan_to_num(roi_scores)

    # ---------------------------------------------
    # Select Top ROIs
    # ---------------------------------------------

    n_selected = int(n_rois * percentage)

    selected_idx = np.argsort(roi_scores)[::-1][:n_selected]

    print("\n")
    print("="*60)
    print("ROI Feature Selection")
    print("="*60)
    print(f"Original ROIs : {n_rois}")
    print(f"Selected ROIs : {n_selected}")

    # ---------------------------------------------
    # Keep complete time series
    # ---------------------------------------------

    X_train = X_train[:, :, selected_idx]
    X_val   = X_val[:, :, selected_idx]
    X_test  = X_test[:, :, selected_idx]

    return X_train, X_val, X_test, selected_idx, roi_scores


# =====================================================
# Cross Validation
# =====================================================

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

history_list = []

loss_scores = []
acc_scores = []
auc_scores = []
precision_scores = []
recall_scores = []

selected_rois_each_fold = []

fold = 1

for train_idx, test_idx in skf.split(X, y):

    print("\n")
    print("="*70)
    print(f"Fold {fold}/{NUM_FOLDS}")
    print("="*70)

    # -------------------------------------------------
    # Train/Test Split
    # -------------------------------------------------

    X_train_full = X[train_idx]
    y_train_full = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    # -------------------------------------------------
    # Validation Split
    # -------------------------------------------------

    X_train, X_val, y_train, y_val = train_test_split(

        X_train_full,
        y_train_full,

        test_size=0.10,

        stratify=y_train_full,

        random_state=42

    )

    print("\nClass Distribution")
    print("Train :", np.bincount(y_train))
    print("Val   :", np.bincount(y_val))
    print("Test  :", np.bincount(y_test))

    # =====================================================
    # F-score ROI Selection
    # =====================================================

    X_train, X_val, X_test, selected_idx, roi_scores = \
        select_top_rois_fscore(
            X_train,
            y_train,
            X_val,
            X_test,
            percentage=ROI_PERCENTAGE
        )

    selected_rois_each_fold.append(selected_idx)

    print("Input Shape After Selection :", X_train.shape)

    # -------------------------------------------------
    # Class Weights
    # -------------------------------------------------

    classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights = dict(zip(classes, weights))

    print("\nClass Weights")
    print(class_weights)

    # -------------------------------------------------
    # Clear Session
    # -------------------------------------------------

    tf.keras.backend.clear_session()

    # -------------------------------------------------
    # Build Model
    # -------------------------------------------------

    model = build_temporal_transformer()

    # -------------------------------------------------
    # Checkpoint
    # -------------------------------------------------

    checkpoint_name = f"best_fold_{fold}.keras"

    checkpoint = tf.keras.callbacks.ModelCheckpoint(

        checkpoint_name,

        monitor="val_auc",

        mode="max",

        save_best_only=True,

        verbose=1

    )

    history = model.fit(

        X_train,
        y_train,

        validation_data=(X_val, y_val),

        epochs=100,

        batch_size=16,

        shuffle=True,

        class_weight=class_weights,

        callbacks=[
            checkpoint,
            *callbacks
        ],

        verbose=1

    )

    history_list.append(history)

    # -------------------------------------------------
    # Load Best Model
    # -------------------------------------------------

    if os.path.exists(checkpoint_name):
        model.load_weights(checkpoint_name)

    # -------------------------------------------------
    # Evaluate
    # -------------------------------------------------

    results = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    loss = results[0]
    acc = results[1]
    auc = results[2]
    precision = results[3]
    recall = results[4]

    loss_scores.append(loss)
    acc_scores.append(acc)
    auc_scores.append(auc)
    precision_scores.append(precision)
    recall_scores.append(recall)

    print("\nFold Results")
    print("-"*40)
    print(f"Loss      : {loss:.4f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"AUC       : {auc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")

    fold += 1


# =====================================================
# Final Results
# =====================================================

print("\n")
print("="*70)
print("Cross Validation Results")
print("="*70)

print(f"Average Loss      : {np.mean(loss_scores):.4f} ± {np.std(loss_scores):.4f}")
print(f"Average Accuracy  : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print(f"Average AUC       : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Average Precision : {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
print(f"Average Recall    : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")

print("="*70)

In [ ]:
# =====================================================
# Cell 8 : Final Results
# =====================================================

print("\n")
print("="*70)
print("FINAL PERFORMANCE")
print("="*70)

print(f"Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")

print(f"Precision: {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")

print(f"Recall   : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")

print(f"AUC      : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

print("="*70)